# ARC-AGI-3 — v2 evaluation on the public games

Generated by `scripts/build_kaggle.py --agent bench`. Attach: the competition dataset, the vLLM wheelhouse, the model snapshot, and the public games (`environment_files/`). Results land in /kaggle/working/results.

In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

In [ ]:
import os
os.environ['ARC3_FORCE_LLM'] = '1'
import glob, os, subprocess, sys, time, urllib.request

T0 = time.time()
LLM_PORT = 8000
LLM_BUDGET_HOURS = 6.5   # stop calling the model after this; play on as v1
os.environ['ARC3_LLM_DEADLINE'] = str(T0 + LLM_BUDGET_HOURS * 3600)
os.environ['ARC3_CALLS_DIR'] = '/tmp/llm_calls'   # /tmp: not a submission candidate

# Sampling, per the Qwen3.6 model card's NON-thinking recommendation. Thinking
# is on by default in Qwen3.x and would spend the token cap on reasoning, so it
# is switched off per request. For a thinking run: set enable_thinking True,
# ARC3_LLM_TEMPERATURE 0.6, top_p 0.95, presence_penalty 0, and raise
# ARC3_LLM_MAX_TOKENS to ~8000 (roughly 5-10x the tokens and time per call).
os.environ.setdefault('ARC3_LLM_EXTRA', '{"chat_template_kwargs": {"enable_thinking": false}, '
                      '"top_p": 0.8, "top_k": 20, "presence_penalty": 1.5}')
os.environ.setdefault('ARC3_LLM_TEMPERATURE', '0.7')
os.environ.setdefault('ARC3_LLM_MAX_TOKENS', '900')

def _model_dir():
    # With several models attached (e.g. Qwen3.6 and Qwen3.8 for a comparison),
    # set ARC3_MODEL_MATCH to a substring of the wanted directory path.
    want = os.environ.get('ARC3_MODEL_MATCH', '').lower()
    for cfg in sorted(glob.glob('/kaggle/input/**/config.json', recursive=True)):
        d = os.path.dirname(cfg)
        if want and want not in d.lower():
            continue
        if glob.glob(os.path.join(d, '*.safetensors')) and os.path.exists(os.path.join(d, 'tokenizer_config.json')):
            return d
    return None

def _start_llm():
    wheel_dirs = sorted({os.path.dirname(p) for p in glob.glob('/kaggle/input/**/vllm-*.whl', recursive=True)})
    model = _model_dir()
    print('vllm wheel dirs:', wheel_dirs, '| model:', model)
    if not wheel_dirs or not model:
        print('LLM unavailable: playing as v1'); return
    if not any(t in model.lower() for t in ('instruct', 'chat', 'qwen3', 'gemma-4', '-it')):
        # Measured 2026-09-27: attaching the 'qwen2.5/transformers/7b' variation
        # (a full-precision base model) instead of '7b-instruct-awq' failed on a
        # T4 at engine start. A base model would not follow the prompt anyway.
        print('WARNING: this does not look like an instruction-tuned model:', model)
    links = sum((['--find-links', d] for d in wheel_dirs), [])
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', *links, 'vllm'],
                       capture_output=True, text=True)
    print(r.stdout[-2000:], r.stderr[-2000:])
    log = open('/tmp/vllm.log', 'w')
    args = [sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
            '--model', model, '--served-model-name', 'local',
            '--host', '127.0.0.1', '--port', str(LLM_PORT),
            '--max-model-len', '16384', '--gpu-memory-utilization', '0.90',
            '--max-num-seqs', '32']
    if 'qwen3' in model.lower():
        # Keeps any reasoning out of the answer text our parser reads.
        args += ['--reasoning-parser', 'qwen3']
    env = dict(os.environ)
    # Pre-Ampere GPUs (T4 is compute capability 7.5): no bf16 and no
    # FlashAttention 2. The settings validated on Kaggle's 2xT4 by
    # github.com/kaggle-vllm/kaggle-vllm: fp16, eager mode, Triton attention.
    try:
        q = subprocess.run(['nvidia-smi', '--query-gpu=compute_cap,name', '--format=csv,noheader'],
                           capture_output=True, text=True).stdout.strip().splitlines()
        major = int(q[0].split('.')[0]) if q else 0
    except (OSError, ValueError):
        q, major = [], 0
    print('GPU:', q)
    if 0 < major < 8:
        args += ['--dtype', 'float16', '--enforce-eager']
        env.setdefault('VLLM_ATTENTION_BACKEND', 'TRITON_ATTN')
    # Reported workaround for a vLLM startup crash on SM120 (RTX Pro 6000)
    # with Qwen3.8 (recipes.vllm.ai/Qwen/Qwen3.8-27B). Harmless elsewhere.
    env.setdefault('VLLM_USE_FLASHINFER_SAMPLER', '0')
    subprocess.Popen(args, stdout=log, stderr=subprocess.STDOUT, env=env)
    for _ in range(240):   # up to 20 minutes; a 27B model measured ~6 min to load
        try:
            with urllib.request.urlopen(f'http://127.0.0.1:{LLM_PORT}/health', timeout=5) as h:
                if h.status == 200:
                    os.environ['ARC3_LLM_URL'] = f'http://127.0.0.1:{LLM_PORT}'
                    print(f'LLM up after {time.time() - T0:.0f}s'); return
        except Exception:
            time.sleep(5)
    print('LLM did not come up: playing as v1')
    # The root cause is usually far above the final traceback, so print the
    # lines that name an error first, then the tail.
    text = open('/tmp/vllm.log', errors='replace').read()
    keys = ('error', 'out of memory', 'exception', 'not supported', 'insufficient', 'cannot')
    hits = [ln for ln in text.splitlines() if any(k in ln.lower() for k in keys)]
    print('--- lines naming an error ---'); print('\n'.join(hits[:40]))
    print('--- end of log ---'); print(text[-2000:])

if os.getenv('KAGGLE_IS_COMPETITION_RERUN') or os.getenv('ARC3_FORCE_LLM'):
    _start_llm()


In [ ]:
%%writefile /tmp/arc3_bundle.py
"""ARC-AGI-3 agent — generated by scripts/build_kaggle.py. Do not edit by hand.

Source: the `arc3` package of this repository, embedded below. Licence: see the
repository. Model-based exploration agent, D-034 in docs/decision-log.md.
"""
import importlib.abc
import importlib.util
import sys

_ARC3_SOURCES = {'arc3': (True, '"""ARC-AGI-3 agent. See docs/state-of-play.md."""\n'), 'arc3.agent': (False, '"""The first end-to-end agent: act, observe, remember, learn, act again (D-034).\n\nEverything before this module was a measurement instrument. This one plays.\n\n**What it does, in one paragraph.** It builds the game\'s state graph as it goes.\nA state is the resting frame with the bookkeeping band masked out, so a move\ncounter ticking does not make every state unique. From the current state it\ntakes an action it has not yet tried there; when there are none left worth\ntrying, it walks the known graph to the nearest state that still has some, or\npresses `RESET` when the level start is the shorter route there. Untried actions\nare ranked by two things: the explorer\'s Thompson estimate of how often that\n*kind* of action moves the playfield (D-031), and the induced dynamics model\'s\none-step prediction (D-025, D-033) — an action the model predicts leads to an\nalready-visited state is pushed down, and one it predicts does nothing is pushed\nfurther.\n\n**Why a graph.** The engine is byte-deterministic (D-009), so every observed\ntransition is an exact, permanent fact about the game. On visited states the\ngraph *is* a perfect dynamics model; the induced rules exist to generalise to\nstates the graph has not reached. The graph is the model-based thesis in its\nmost literal form, and it is the part that can plan today: navigating to a\nfrontier is shortest-path search against a model, done entirely internally.\n\n**What this does not do.**\n\n- **No goal.** It explores the state space; it does not know what winning looks\n  like. Levels are cleared when systematic exploration stumbles onto the win\n  state. The goal subsystem of the design thesis is not built, and this agent is\n  the baseline it must beat.\n- **No multi-step use of the rules.** The dynamics model ranks one step ahead.\n  Rollouts through predicted frames are not attempted, because multi-step\n  prediction accuracy has never been measured (state-of-play, objective 6).\n- **No transfer across levels** beyond the explorer\'s arm statistics and the\n  rule corpus. The state graph is discarded on each level change, since states\n  from level *n* never recur in level *n+1*.\n- **No hidden state.** If the same masked frame and action lead to different\n  outcomes, the latest observation overwrites the edge. A game whose response\n  depends on history will look to this agent like a graph that keeps changing.\n- **Bookkeeping by rows only.** The band (`explore.OnlineBand`) is a set of rows.\n  A counter drawn in a column, or inside the playfield, still splits states.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport random\nfrom collections import deque\nfrom dataclasses import dataclass, field\nfrom typing import Any, Optional\n\nimport numpy as np\n\nfrom .dynamics.induce import TransitionObservation, induce, observe_transition\nfrom .dynamics.rules import (\n    STATIC,\n    VANISH,\n    RuleSet,\n    apply_effect,\n    describe_object,\n    reveal_colour,\n)\nfrom .explore import DECAY, OnlineBand, _Arm\nfrom .perception.segment import Segmentation, background_colour, segment\nfrom .types import CLICK, RESET, Action, GameState, Observation, Transition\n\n#: Carving the dynamics model is induced and evaluated under. D-025\'s best mean.\nMODEL_CARVING = "colour_4"\n\n#: Carving click targets are drawn from. The explorer\'s measured choice (D-031).\nCLICK_CARVING = "colour_8"\n\n#: Re-induce the rule set after this many new transitions.\nINDUCE_EVERY = 10\n\n#: Cap on transitions fed to induction. D-025 found the model saturates at ~50.\nINDUCE_WINDOW = 150\n\n#: Cap on click candidates per state, keeping the largest distinct targets.\n#: bp35 proposes 196 targets for 8 true outcomes (D-021); beyond this the arms\n#: decide which colours are worth the rest.\nMAX_CLICKS = 64\n\n#: Ranking penalties from the model\'s one-step prediction.\nPENALTY_KNOWN = 0.35   # predicted to reach a state already in the graph\nPENALTY_NOOP = 0.6     # predicted to change nothing at all\n\n#: Rows from the top or bottom edge a bookkeeping band may occupy (edge_band).\nEDGE_ROWS = 4\n\n#: How many of the arm-ranked untried actions the model is consulted on.\nMODEL_TOP_K = 8\n\n#: Extra path cost per unit of missing priority when choosing a frontier.\nFRONTIER_WEIGHT = 4.0\n\n\ndef state_key(frame: np.ndarray, band: np.ndarray) -> bytes:\n    """Hash of the frame with bookkeeping rows blanked."""\n    f = frame.copy() if band.any() else frame\n    if band.any():\n        f[band] = 0\n    return hashlib.blake2b(np.ascontiguousarray(f).tobytes(), digest_size=12).digest()\n\n\n@dataclass\nclass _Node:\n    frame: np.ndarray\n    candidates: list[tuple[Any, Action]]  # (arm key, action)\n    tried: set[Action] = field(default_factory=set)\n    edges: dict[Action, bytes] = field(default_factory=dict)\n    dead: bool = False  # GAME_OVER state; only RESET leaves it\n\n    def untried(self) -> list[tuple[Any, Action]]:\n        return [(k, a) for k, a in self.candidates if a not in self.tried]\n\n\n@dataclass\nclass GraphAgent:\n    """Explores a level\'s state graph, ranked by arms and the induced model.\n\n    Feed it observations with :meth:`act`; it returns the next action. It keeps\n    no reference to any client, so the same object drives the local engine, a\n    replay, or the Kaggle harness through an adapter.\n    """\n\n    seed: int = 0\n    use_model: bool = False  # the model lowers RHAE 3.46% -> 1.18% (D-034)\n    #: Leave a state whose untried actions all belong to arms with a mean below\n    #: this, if a frontier elsewhere offers better. 0 disables.\n    local_min: float = 0.3\n    #: Restrict the bookkeeping band to the top and bottom EDGE_ROWS rows.\n    edge_band: float = 0.0\n    #: Priority bonus for repeating the arm of the last action, if it moved the\n    #: playfield. Biases exploration towards straight lines. 0 disables.\n    repeat: float = 0.0\n    _moved_key: Any = field(default=None, init=False)\n    _rng: random.Random = field(init=False)\n    band: OnlineBand = field(default_factory=OnlineBand, init=False)\n    _band_sig: bytes = field(default=b"", init=False)\n    _nodes: dict[bytes, _Node] = field(default_factory=dict, init=False)\n    _start: Optional[bytes] = field(default=None, init=False)\n    _arms: dict[Any, _Arm] = field(default_factory=dict, init=False)\n    _history: list[tuple[np.ndarray, Action, np.ndarray]] = field(default_factory=list, init=False)\n    _obs_cache: list[TransitionObservation] = field(default_factory=list, init=False)\n    _ruleset: Optional[RuleSet] = field(default=None, init=False)\n    _since_induce: int = field(default=0, init=False)\n    _last: Optional[tuple[bytes, np.ndarray, Action, Any]] = field(default=None, init=False)\n    _level: int = field(default=0, init=False)\n    _path: list[Action] = field(default_factory=list, init=False)\n    stats: dict[str, int] = field(default_factory=dict, init=False)\n\n    def __post_init__(self) -> None:\n        self._rng = random.Random(self.seed)\n\n    # -- bookkeeping ---------------------------------------------------------\n\n    def _bump(self, k: str) -> None:\n        self.stats[k] = self.stats.get(k, 0) + 1\n\n    def _arm(self, key: Any) -> _Arm:\n        return self._arms.setdefault(key, _Arm())\n\n    def _rekey_if_band_changed(self) -> None:\n        """Re-hash the graph when the bookkeeping band estimate moves.\n\n        Keys depend on the band, so a band change silently orphans every node.\n        Rebuilding from stored frames is cheap and keeps the graph coherent.\n        """\n        mask = self._mask()\n        sig = np.packbits(mask).tobytes()\n        if sig == self._band_sig:\n            return\n        self._band_sig = sig\n        old = self._nodes\n        remap = {k: state_key(n.frame, mask) for k, n in old.items()}\n        self._nodes = {}\n        for k, n in old.items():\n            nk = remap[k]\n            n.edges = {a: remap.get(t, t) for a, t in n.edges.items()}\n            if nk in self._nodes:  # two states merged by the new mask\n                m = self._nodes[nk]\n                m.tried |= n.tried\n                m.edges.update(n.edges)\n                m.dead = m.dead and n.dead\n            else:\n                self._nodes[nk] = n\n        if self._start is not None:\n            self._start = remap.get(self._start, self._start)\n        if self._last is not None:\n            k, f, a, ak = self._last\n            self._last = (remap.get(k, k), f, a, ak)\n        self._path = []\n        self._bump("rekeys")\n\n    def _mask(self) -> np.ndarray:\n        """The bookkeeping band, optionally restricted to the frame\'s edge rows.\n\n        The row-frequency estimate cannot tell a counter from a playfield\n        region that happens to change on about half of all actions. On tr87\n        it flips rows 48-60 in and out of the band every step and the graph is\n        re-keyed 489 times in 500 actions. Every counter measured so far sits\n        at an edge (sp80 row 0, ls20 rows 61-62), where the UI layer is drawn.\n        """\n        m = self.band.mask()\n        if self.edge_band:\n            m = m.copy()\n            m[EDGE_ROWS:-EDGE_ROWS, :] = False\n        return m\n\n    def _key(self, frame: np.ndarray) -> bytes:\n        return state_key(frame, self._mask())\n\n    def _node(self, key: bytes, frame: np.ndarray, available: tuple[int, ...]) -> _Node:\n        n = self._nodes.get(key)\n        if n is None:\n            n = _Node(frame=frame, candidates=self._candidates(frame, available))\n            self._nodes[key] = n\n        return n\n\n    def _candidates(self, frame: np.ndarray, available: tuple[int, ...]) -> list[tuple[Any, Action]]:\n        out: list[tuple[Any, Action]] = []\n        for a in sorted(a for a in available if a not in (RESET, CLICK)):\n            out.append((("act", a), Action.simple(a)))\n        if CLICK in available:\n            seg = segment(frame, include=(CLICK_CARVING,))[CLICK_CARVING]\n            bg = seg.background_colour\n            comps = [c for c in seg.components if bg is None or c.dominant_colour != bg]\n            comps.sort(key=lambda c: -c.size)\n            seen: set[tuple[int, int]] = set()\n            for c in comps[:MAX_CLICKS]:\n                xy = c.click_target\n                if xy in seen:\n                    continue\n                seen.add(xy)\n                out.append((("click", c.dominant_colour), Action.click(*xy)))\n        return out\n\n    # -- learning ------------------------------------------------------------\n\n    def _learn(self, obs: Observation) -> None:\n        if self._last is None:\n            return\n        pkey, pframe, action, akey = self._last\n        after = obs.last if obs.last is not None else pframe\n        if action.is_reset or obs.levels_completed != self._level:\n            # A restart or a level change is not the game\'s dynamics.\n            return\n        self.band.update(pframe, after)\n        moved = self.band.moved_playfield(pframe, after)\n        self._moved_key = akey if moved else None\n        if akey is not None:\n            self._arm(akey).update(moved)\n            for arm in self._arms.values():\n                arm.decay(DECAY)\n        self._history.append((pframe, action, after))\n        self._since_induce += 1\n        self._bump("playfield_moves" if moved else "playfield_still")\n\n    def _maybe_induce(self) -> None:\n        if not self.use_model or self._since_induce < INDUCE_EVERY:\n            return\n        self._since_induce = 0\n        recent = self._history[-INDUCE_WINDOW:]\n        # Observations are cached by position; the window slides, so rebuild the\n        # tail that is new and drop what fell out.\n        need = len(recent)\n        cached = self._obs_cache[-need:] if self._obs_cache else []\n        missing = recent[len(cached):]\n        for before, action, after in missing:\n            t = Transition(before=before, action=action, frames=(after,),\n                           levels_completed_before=0, levels_completed_after=0,\n                           state_after=GameState.NOT_FINISHED)\n            cached.append(observe_transition(t, MODEL_CARVING))\n        self._obs_cache = cached[-INDUCE_WINDOW:]\n        self._ruleset = induce(self._obs_cache)\n        self._bump("inductions")\n\n    def _predict(self, frame: np.ndarray, seg: Segmentation, action: Action) -> np.ndarray:\n        """`predict_frame`, reusing one segmentation across candidate actions."""\n        assert self._ruleset is not None\n        bg = seg.background_colour if seg.background_colour is not None else background_colour(frame)\n        canvas = frame.copy()\n        effects = [(c, self._ruleset.effect_for(action, describe_object(c, action)))\n                   for c in seg.components]\n        for c, e in effects:\n            if e.kind != STATIC:\n                canvas[c.mask] = reveal_colour(frame, c.mask, bg)\n        for c, e in effects:\n            if e.kind not in (STATIC, VANISH):\n                apply_effect(canvas, frame, c, e, bg, self._ruleset.shapes)\n        return canvas\n\n    # -- choosing ------------------------------------------------------------\n\n    def _priority(self, key: bytes, node: _Node) -> list[tuple[float, Any, Action]]:\n        """Untried actions at ``node``, each with a priority in roughly [0, 1]."""\n        untried = node.untried()\n        if not untried:\n            return []\n        seg = None\n        if self.use_model and self._ruleset is not None:\n            seg = segment(node.frame, include=(MODEL_CARVING,))[MODEL_CARVING]\n        mask = self._mask()\n        def base(k: Any) -> float:\n            p = self._arm(k).sample(self._rng)\n            if self.repeat and k is not None and k == self._moved_key:\n                p += self.repeat\n            return p\n\n        scored = sorted(((base(k), k, a) for k, a in untried), key=lambda t: -t[0])\n        out = []\n        for i, (p, akey, action) in enumerate(scored):\n            # Predicting is the expensive part; only the front of the queue can\n            # change the choice, so only it is consulted.\n            if seg is not None and i < MODEL_TOP_K:\n                pred = self._predict(node.frame, seg, action)\n                changed = bool(((pred != node.frame) & ~mask).any())\n                if not changed:\n                    p -= PENALTY_NOOP\n                elif state_key(pred, mask) in self._nodes:\n                    p -= PENALTY_KNOWN\n            out.append((p, akey, action))\n        out.sort(key=lambda t: -t[0])\n        return out\n\n    def _frontier_path(self, here: bytes) -> list[Action]:\n        """Shortest known route to a state with worthwhile untried actions.\n\n        Breadth-first over observed edges from ``here`` and, at a cost of one\n        extra action, from the level start via `RESET`. Candidates are scored\n        by path length plus a penalty for low best-arm mean, so a long walk is\n        only taken for an action class that has been paying off.\n        """\n        def bfs(src: bytes) -> dict[bytes, list[Action]]:\n            paths = {src: []}\n            q = deque([src])\n            while q:\n                u = q.popleft()\n                n = self._nodes.get(u)\n                if n is None or n.dead:\n                    continue\n                for a, v in n.edges.items():\n                    if v not in paths and v in self._nodes:\n                        paths[v] = paths[u] + [a]\n                        q.append(v)\n            return paths\n\n        options: dict[bytes, list[Action]] = bfs(here)\n        if self._start is not None and self._start != here:\n            for k, p in bfs(self._start).items():\n                rp = [Action.simple(RESET)] + p\n                if k not in options or len(rp) < len(options[k]):\n                    options[k] = rp\n\n        best: Optional[tuple[float, list[Action]]] = None\n        for k, p in options.items():\n            if not p:\n                continue\n            n = self._nodes[k]\n            if n.dead:\n                continue\n            u = n.untried()\n            if not u:\n                continue\n            q = max(self._arm(ak).mean for ak, _ in u)\n            cost = len(p) + FRONTIER_WEIGHT * (1.0 - q) - self.goal_weight * self._node_value(k)\n            if best is None or cost < best[0]:\n                best = (cost, p)\n        return best[1] if best else []\n\n    def act(self, obs: Observation) -> Action:\n        """The next action. **Returning it is free; the caller spends it.**"""\n        self._learn(obs)\n\n        if obs.levels_completed > self._level:\n            # New level: its states are all new. Keep arms and rules.\n            if self._last is not None:\n                self._on_level_won(self._last[0], self._last[1], self._last[2])\n            self._level = obs.levels_completed\n            self._nodes, self._start, self._path = {}, None, []\n            self._bump("levels")\n\n        frame = obs.last if obs.last is not None else (self._last[1] if self._last else None)\n        if frame is None:\n            self._last = None\n            return Action.simple(RESET)\n\n        self._rekey_if_band_changed()\n        key = self._key(frame)\n        node = self._node(key, frame, obs.available_actions)\n        if self._start is None or obs.full_reset or (\n            self._last is not None and self._last[2].is_reset\n        ):\n            self._start = key\n\n        # Record the edge we just took.\n        if self._last is not None:\n            pkey, _, paction, _ = self._last\n            prev = self._nodes.get(pkey)\n            if prev is not None and not paction.is_reset:\n                prev.edges[paction] = key\n                prev.tried.add(paction)\n\n        if obs.state == GameState.GAME_OVER:\n            node.dead = True\n            self._path = []\n            return self._emit(key, frame, Action.simple(RESET), None)\n        if obs.state == GameState.WIN:\n            return self._emit(key, frame, Action.simple(RESET), None)\n\n        self._maybe_induce()\n        self._on_state(key, frame, obs)\n\n        # Follow a planned route while it still matches the graph.\n        if self._path:\n            a = self._path.pop(0)\n            if a.is_reset or a in node.edges or a in {c for _, c in node.candidates}:\n                self._bump("nav_steps")\n                return self._emit(key, frame, a, None)\n            self._path = []\n\n        ranked = self._priority(key, node)\n        local_best = max((self._arm(k).mean for _, k, _ in ranked), default=0.0)\n        if (ranked and ranked[0][0] > -0.2 and local_best >= self.local_min\n                and not self._prefer_frontier(key)):\n            _, akey, a = ranked[0]\n            self._bump("explore_steps")\n            return self._emit(key, frame, a, akey)\n\n        path = self._frontier_path(key)\n        if path:\n            self._path = path[1:]\n            self._bump("nav_steps")\n            return self._emit(key, frame, path[0], None)\n\n        if ranked:\n            _, akey, a = ranked[0]\n            self._bump("explore_steps")\n            return self._emit(key, frame, a, akey)\n\n        # Graph exhausted: every reachable state fully tried. Retry at random\n        # rather than stall, since hidden state may make a retry informative.\n        self._bump("exhausted")\n        akey, a = self._rng.choice(node.candidates) if node.candidates else (None, Action.simple(RESET))\n        return self._emit(key, frame, a, akey)\n\n    def took(self, action: Action) -> None:\n        """Another controller acted instead of this agent (v3 uses the graph as a\n        tool). Rewrite the last emitted action so the next observation is\n        recorded as the edge that was really taken."""\n        if self._last is None:\n            return\n        key, frame, planned, _ = self._last\n        if action != planned:\n            self._path = []\n        self._last = None\n        self._emit(key, frame, action, None)\n\n    # -- hooks for a goal layer (D-035). All inert here, so v1 is unchanged. --\n\n    #: Frontier cost per unit of state value. Irrelevant while values are 0.\n    goal_weight = 0.0\n\n    def _node_value(self, key: bytes) -> float:\n        """How close state ``key`` is to winning, in [0, 1]. v1 has no idea."""\n        return 0.0\n\n    def _prefer_frontier(self, here: bytes) -> bool:\n        """Leave this state for a better one even though it has untried actions?"""\n        return False\n\n    def _on_state(self, key: bytes, frame: np.ndarray, obs: Observation) -> None:\n        """A non-terminal state was reached and the level is still in play."""\n\n    def _on_level_won(self, key: bytes, frame: np.ndarray, action: Action) -> None:\n        """``action`` from state ``key`` completed the level."""\n\n    def _emit(self, key: bytes, frame: np.ndarray, action: Action, akey: Any) -> Action:\n        if akey is None and not action.is_reset:\n            node = self._nodes.get(key)\n            if node is not None:\n                akey = next((k for k, a in node.candidates if a == action), None)\n        self._last = (key, frame, action, akey)\n        return action\n\n    def describe(self) -> str:\n        return (f"{len(self._nodes)} states, level {self._level}, "\n                + ", ".join(f"{k}={v}" for k, v in sorted(self.stats.items())))\n'), 'arc3.agent_v2': (False, '"""v2: GraphAgent steered by LLM-proposed goal hypotheses (D-035).\n\n`GraphAgent` (D-034) knows how to act, remember and navigate, and has no idea\nwhere to go. This subclass supplies the missing part: a population of goal\nhypotheses (`hypothesis.Population`) proposed as code by an LLM, which gives\nevery known state a value, and the value steers the frontier choice — a\nbest-first search over the exact state graph, guided by the hypotheses.\n\nWhere the pieces meet:\n\n- **Proposing.** The LLM is asked for hypotheses (`propose.goal_messages`) at\n  three moments: ``WARMUP`` actions into a level, so there are observed effects\n  to report; every ``STUCK_EVERY`` actions without a win; and never more than\n  ``MAX_LEVEL_CALLS`` per level or ``MAX_GAME_CALLS`` per game.\n- **Testing.** Every visited state is shown to the population\n  (`observe_state`: states a hypothesis calls "reached" that do not end the\n  level count against it); a win scores every hypothesis on the state the\n  winning action was taken from (`observe_win`).\n- **Deciding.** `_node_value` is the population\'s weighted progress. It lowers\n  the frontier cost of promising states, and `_prefer_frontier` leaves the\n  current state for a known state whose value is clearly higher, even if\n  untried actions remain here.\n- **Transfer.** Hypotheses that gained weight on a win are carried to the next\n  level at the prior weight, since levels of one game tend to share a goal. The\n  rest are dropped with the level.\n\nWith no LLM, or before the first proposal, every value is 0 and this is exactly\nv1.\n\n**What this does not do.**\n\n- **The LLM never picks actions** and never sees anything but the scene text.\n- **No dynamics hypotheses.** Effects are reported to the LLM as evidence, but\n  it is not asked to predict them; the state graph is still the only dynamics\n  model in use.\n- **Values are for visited states only.** A hypothesis cannot value a state the\n  agent has not reached, because nothing predicts unseen frames reliably yet\n  (the induced model hurt, D-034). The search is best-first over what exists.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom typing import Any, Optional\n\nimport numpy as np\n\nfrom .agent import GraphAgent\nfrom .hypothesis import PRIOR_WEIGHT, Hypothesis, Population\nfrom .llm import LLM\nfrom .propose import PROMPT_VERSION, Evidence, describe_effect, goal_messages\nfrom .scene import Scene, make_scene\nfrom .types import CLICK, RESET, Action, Observation\n\nWARMUP = 10\nSTUCK_EVERY = 150\nMAX_LEVEL_CALLS = 4\nMAX_GAME_CALLS = 16\n\n\nMAX_TOKENS = 900\n\n#: Weight ceiling for a hypothesis carried into the next level.\nCARRY_CAP = 4.0\n\n#: States followed under one hypothesis without improving its best value\n#: before it is judged stalled, and the soft penalty for stalling.\nFOCUS_STALL = 40\nSTALL_PENALTY = 0.8\n\n\n@dataclass\nclass HypothesisAgent(GraphAgent):\n    llm: Optional[LLM] = None\n    #: Sampling for proposals. 0.0 suits small local models; Qwen3.6\'s card\n    #: recommends 0.7 in non-thinking mode (set from the Kaggle notebook).\n    temperature: float = 0.0\n    #: Output cap per call. Thinking mode needs far more than the 900 that\n    #: suffices for three short functions.\n    max_tokens: float = MAX_TOKENS\n    #: Value units worth one step of walking (the frontier trades them off).\n    goal_weight: float = 100.0\n    #: v2 needs the band confined to edge rows. Unconfined, a player moving\n    #: sideways within the same rows gets those rows masked as a "counter"\n    #: (measured on the synthetic room in tests/test_agent_v2.py, and the same\n    #: failure as tr87\'s 489 re-keys, D-034) — and then the hypotheses cannot\n    #: see the player at all. Neutral on v1\'s score on ten games.\n    edge_band: float = 1.0\n    pop: Population = field(default_factory=Population, init=False)\n    _scenes: dict = field(default_factory=dict, init=False)\n    _scene_sig: bytes = field(default=b"", init=False)\n    _effects: list = field(default_factory=list, init=False)\n    _prev: Optional[tuple[bytes, np.ndarray, Action]] = field(default=None, init=False)\n    _level_actions: int = field(default=0, init=False)\n    _level_calls: int = field(default=0, init=False)\n    _since_call: int = field(default=0, init=False)\n    _previous_win: Optional[str] = field(default=None, init=False)\n    _available: tuple = field(default=(), init=False)\n    calls: list = field(default_factory=list, init=False)  # (level, actions, n hyps, n compiled)\n    #: One population snapshot per won level, taken after the win is scored and\n    #: before the population is rebuilt — the evidence a later reader needs.\n    levels_log: list = field(default_factory=list, init=False)\n    #: Follow one sampled hypothesis at a time (True) or the weighted average of\n    #: all of them (False, the first design; kept for comparison).\n    focus_mode: bool = True\n    _focus: Optional[Hypothesis] = field(default=None, init=False)\n    _focus_best: float = field(default=-1.0, init=False)\n    _focus_since: int = field(default=0, init=False)\n\n    # -- scenes ------------------------------------------------------------\n\n    def _scene(self, key: bytes) -> Scene:\n        # Scenes hide the band, so a band change makes every cached scene stale.\n        if self._scene_sig != self._band_sig:\n            self._scenes, self._scene_sig = {}, self._band_sig\n        s = self._scenes.get(key)\n        if s is None:\n            node = self._nodes.get(key)\n            frame = node.frame if node is not None else self._prev[1]\n            s = make_scene(frame, self._mask())\n            self._scenes[key] = s\n        return s\n\n    # -- hooks -------------------------------------------------------------\n\n    def _node_value(self, key: bytes) -> float:\n        if self.focus_mode:\n            if self._focus is None:\n                return 0.0\n            return self._focus.value(key, lambda: self._scene(key))\n        v = self.pop.value(key, lambda: self._scene(key))\n        return 0.0 if v is None else v\n\n    # -- focus: follow one hypothesis at a time ----------------------------------\n\n    def _choose_focus(self) -> None:\n        """Pick a live hypothesis to follow, with probability proportional to weight.\n\n        Averaging rivals was the first design and failed on the synthetic room:\n        "go to the far corner" and "go to the near corner" summed to a constant,\n        so nothing steered, and the wrong one (max 0.97) was never called\n        "reached" and so never lost weight. Following one at a time lets each\n        be tested; a stalled one pays a soft penalty and another is drawn.\n        """\n        live = self.pop.live()\n        if not live:\n            self._focus = None\n            return\n        total = sum(h.weight for h in live)\n        r = self._rng.random() * total\n        for h in live:\n            r -= h.weight\n            if r <= 0:\n                break\n        self._focus, self._focus_best, self._focus_since = h, -1.0, 0\n        self._bump("focus_switches")\n\n    def _track_focus(self, key: bytes) -> None:\n        live = self.pop.live()\n        if not live:\n            self._focus = None\n            return\n        if self._focus is None or self._focus not in live:\n            self._choose_focus()\n        v = self._focus.value(key, lambda: self._scene(key))\n        if v > self._focus_best + 1e-9:\n            self._focus_best, self._focus_since = v, 0\n            return\n        self._focus_since += 1\n        if self._focus_since >= FOCUS_STALL:\n            # Followed for FOCUS_STALL states without reaching a better one:\n            # soft evidence against it (Guardrail 1: scaled, never removed).\n            self._focus._scale(STALL_PENALTY)\n            self._focus.stalls += 1\n            self._choose_focus()\n\n    def _prefer_frontier(self, here: bytes) -> bool:\n        """Best-first: leave if a known state beats this one by more than the walk.\n\n        A fixed value margin was tried first (0.15) and failed on the synthetic\n        room: one step backwards from a 0.969 state cost 0.06, under the margin,\n        so the agent never returned and drifted away from a goal one move off.\n        Trading value against path length (``goal_weight`` value units per\n        step) is what best-first search needs.\n        """\n        if not self.pop.live():\n            return False\n        here_v = self._node_value(here)\n        dist = self._distances(here)\n        for k, n in self._nodes.items():\n            if k == here or n.dead or not n.untried() or k not in dist:\n                continue\n            if self.goal_weight * (self._node_value(k) - here_v) > dist[k]:\n                self._bump("goal_detours")\n                return True\n        return False\n\n    def _distances(self, src: bytes) -> dict[bytes, int]:\n        """Steps from ``src`` over known edges, with RESET as a one-step jump to the start."""\n        from collections import deque\n        dist = {src: 0}\n        q = deque([src])\n        if self._start is not None and self._start not in dist:\n            dist[self._start] = 1\n            q.append(self._start)\n        while q:\n            u = q.popleft()\n            n = self._nodes.get(u)\n            if n is None or n.dead:\n                continue\n            for v in n.edges.values():\n                if v not in dist and v in self._nodes:\n                    dist[v] = dist[u] + 1\n                    q.append(v)\n        return dist\n\n    def _on_state(self, key: bytes, frame: np.ndarray, obs: Observation) -> None:\n        self._available = obs.available_actions\n        self._level_actions += 1\n        self._since_call += 1\n        if self.llm is None:\n            # Scenes and effect lines are prompt material; without a model they\n            # are pure cost, and the no-LLM fallback should run at v1\'s speed.\n            return\n        if self._prev is not None and self._prev[2] is not None and not self._prev[2].is_reset:\n            pkey = self._prev[0]\n            if pkey in self._nodes:\n                line = describe_effect(self._scene(pkey), self._scene(key), str(self._prev[2]))\n                if not self._effects or self._effects[-1] != line:\n                    self._effects.append(line)\n                    self._effects = self._effects[-40:]\n        self.pop.observe_state(key, lambda: self._scene(key))\n        if self.focus_mode:\n            self._track_focus(key)\n        self._maybe_propose(key)\n\n    def _snapshot(self, won: bool) -> dict:\n        """The level\'s population as data, for diagnosis (scripts/diagnose_v2.py)."""\n        return {\n            "level": self._level, "won": won, "actions": self._level_actions,\n            "calls": sum(1 for c in self.calls if c[0] == self._level),\n            "hypotheses": [{\n                "id": h.id, "goal": h.goal, "origin": h.origin, "weight": round(h.weight, 3),\n                "compiled": h.fn is not None, "compile_error": h.compile_error,\n                "informative": h.informative, "errors": h.n_errors,\n                "false_goals": h.false_goals, "stalls": h.stalls,\n                "wins_confirmed": h.wins_confirmed,\n                "wins_missed": h.wins_missed, "states_scored": len(h._seen),\n                "distinct_values": len(set(h._seen.values())),\n            } for h in self.pop.hypotheses],\n        }\n\n    def level_report(self) -> list[dict]:\n        """One snapshot per finished level plus the level in progress."""\n        return [*self.levels_log, self._snapshot(won=False)]\n\n    def _on_level_won(self, key: bytes, frame: np.ndarray, action: Action) -> None:\n        scene = make_scene(frame, self._mask())\n        self.pop.observe_win(key, lambda: scene)\n        self.levels_log.append(self._snapshot(won=True))\n        self._previous_win = (f"{action} was taken from a state with "\n                              + scene.describe(with_grid=False).replace("\\n", "; "))\n        keep = [h for h in self.pop.hypotheses if h.fn is not None and h.weight > PRIOR_WEIGHT]\n        self.pop = Population()\n        for h in keep:\n            # Carried at the weight it earned (capped), not the prior: entering\n            # level 2 at weight 1 beside a fresh wrong guess at weight 1 left\n            # the two pulling in opposite directions (synthetic room, measured).\n            self.pop.hypotheses.append(Hypothesis(\n                id=self.pop._next, goal=h.goal, source=h.source, fn=h.fn,\n                origin=f"carried from level {self._level + 1}",\n                weight=min(h.weight, CARRY_CAP), wins_confirmed=h.wins_confirmed))\n            self.pop._next += 1\n        for _ in keep:\n            self._bump("carried_hypotheses")\n        self._scenes, self._effects = {}, []\n        self._focus = None\n        self._level_actions = self._level_calls = self._since_call = 0\n\n    # -- proposing -----------------------------------------------------------\n\n    def _maybe_propose(self, key: bytes) -> None:\n        if self.llm is None or len(self.calls) >= MAX_GAME_CALLS:\n            return\n        if self._level_calls >= MAX_LEVEL_CALLS:\n            return\n        first = self._level_calls == 0 and self._level_actions >= WARMUP\n        stuck = self._level_calls > 0 and self._since_call >= STUCK_EVERY\n        if not (first or stuck):\n            return\n        actions = [f"ACTION{a}" for a in sorted(self._available) if a not in (RESET, CLICK)]\n        if CLICK in self._available:\n            actions.append("CLICK(x, y) on any cell")\n        ranked = sorted(self.pop.hypotheses, key=lambda h: -h.weight)\n        ev = Evidence(scene=self._scene(key), actions=actions, effects=list(self._effects),\n                      level=self._level, actions_used=self._level_actions,\n                      previous_win=self._previous_win, hypotheses=ranked)\n        try:\n            text = self.llm.complete(goal_messages(ev), max_tokens=int(self.max_tokens),\n                                     temperature=self.temperature, seed=self.seed + len(self.calls))\n        except Exception:\n            self._bump("llm_errors")\n            text = ""\n        added = self.pop.add_from_text(\n            text, origin=f"{PROMPT_VERSION} level {self._level + 1} call {self._level_calls + 1}")\n        self.calls.append((self._level, self._level_actions, len(added),\n                           sum(1 for h in added if h.fn is not None)))\n        self._level_calls += 1\n        self._since_call = 0\n        self._bump("llm_calls")\n\n    # -- bookkeeping -----------------------------------------------------------\n\n    def act(self, obs: Observation) -> Action:\n        a = super().act(obs)\n        self._prev = (self._last[0], self._last[1], self._last[2]) if self._last else None\n        return a\n\n    def describe(self) -> str:\n        n = len(self.pop.hypotheses)\n        live = len(self.pop.live())\n        return f"{super().describe()}, hyps={n} live={live}"\n'), 'arc3.client': (True, '"""Environment access. The only package that may import the engine."""\n\nfrom .budget import ActionBudget, GameResult, LevelResult, level_score\nfrom .env import Client, EnvClient, available_environments\nfrom .replay import ReplayClient, ReplayMismatch, read_episode, write_episode\n\n__all__ = [\n    "ActionBudget",\n    "Client",\n    "EnvClient",\n    "GameResult",\n    "LevelResult",\n    "ReplayClient",\n    "ReplayMismatch",\n    "available_environments",\n    "level_score",\n    "read_episode",\n    "write_episode",\n]\n'), 'arc3.client.budget': (False, '"""Action accounting and offline RHAE scoring.\n\nThe human baselines are in every game\'s ``metadata.json`` (D-009), so we can\ncompute the real metric locally, per level, from day one. Do it on every run -\n``docs/benchmark.md`` warns that local development permits infinite retries\nwhile RHAE punishes them quadratically, and the only defence is to make the\nscore visible from the start rather than at the end.\n\n**Two aggregation details are not pinned down by the published methodology**, and\nare exposed here as explicit parameters rather than guessed silently:\n\n- The stated "per-level score is capped at 1.15x the human baseline" could mean\n  the *score* is capped at 1.15, or the *efficiency ratio* is, giving a score cap\n  of 1.15**2 = 1.3225. We default to capping the score, the more literal reading.\n- The stated example "completing 4 of 5 levels cannot exceed 66.7%" does not\n  follow from a plain mean over attempted levels (which gives 80%) or from\n  squaring it (64%), so the game-level aggregation rule is not recoverable from\n  the text. We use the mean over *all* levels in the game, scoring 0 for levels\n  never completed, and flag the result as approximate.\n\nAnything reported from this module should say which convention it used.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Optional, Sequence\n\n#: Per-level score cap, as the more literal reading of the published rule.\nDEFAULT_SCORE_CAP = 1.15\n\n\ndef level_score(\n    baseline_actions: int,\n    ai_actions: int,\n    *,\n    cap: float = DEFAULT_SCORE_CAP,\n    cap_ratio: bool = False,\n) -> float:\n    """RHAE for one completed level: ``(baseline / ai) ** 2``, capped.\n\n    ``cap_ratio=True`` caps the efficiency ratio at ``cap`` instead of the score,\n    which is the alternative reading of the published rule (see module docstring).\n    """\n    if ai_actions <= 0:\n        raise ValueError("ai_actions must be positive for a completed level")\n    if baseline_actions <= 0:\n        raise ValueError("baseline_actions must be positive")\n    ratio = baseline_actions / ai_actions\n    if cap_ratio:\n        return min(ratio, cap) ** 2\n    return min(ratio**2, cap)\n\n\n@dataclass(frozen=True, slots=True)\nclass LevelResult:\n    """One level\'s outcome. ``actions`` counts only actions spent on this level."""\n\n    index: int\n    actions: int\n    completed: bool\n    baseline: Optional[int] = None\n\n    def score(self, *, cap: float = DEFAULT_SCORE_CAP, cap_ratio: bool = False) -> Optional[float]:\n        """RHAE for this level, or None if the baseline is unavailable.\n\n        ``None`` is a real answer, not a failure: under D-012 the metadata\n        provider may withhold baselines, and the caller must cope.\n        """\n        if not self.completed:\n            return 0.0\n        if self.baseline is None:\n            return None\n        return level_score(self.baseline, self.actions, cap=cap, cap_ratio=cap_ratio)\n\n\n@dataclass(frozen=True, slots=True)\nclass GameResult:\n    game_id: str\n    levels: tuple[LevelResult, ...]\n    total_levels: int\n    actions_used: int\n\n    @property\n    def levels_completed(self) -> int:\n        return sum(1 for lv in self.levels if lv.completed)\n\n    def score(\n        self, *, cap: float = DEFAULT_SCORE_CAP, cap_ratio: bool = False\n    ) -> Optional[float]:\n        """Mean level score over *all* levels in the game, or None.\n\n        Levels never reached score 0, which is what makes an incomplete game\n        cost more than the levels it did finish. Returns ``None`` if any\n        completed level lacks a baseline, since a partial score would silently\n        mix conventions.\n        """\n        if self.total_levels <= 0:\n            return None\n        total = 0.0\n        for lv in self.levels:\n            s = lv.score(cap=cap, cap_ratio=cap_ratio)\n            if s is None:\n                return None\n            total += s\n        return total / self.total_levels\n\n\nclass ActionBudget:\n    """Tracks actions per level and reports RHAE as a run proceeds.\n\n    Feed it every observation. It segments the run into levels on\n    ``levels_completed`` increments and on full resets, so per-level action\n    counts come out without the caller bookkeeping anything.\n    """\n\n    def __init__(self, game_id: str, total_levels: int, baselines: Optional[Sequence[int]] = None):\n        self.game_id = game_id\n        self.total_levels = total_levels\n        self.baselines: Optional[tuple[int, ...]] = (\n            tuple(int(b) for b in baselines) if baselines is not None else None\n        )\n        self.actions_used = 0\n        self._level_index = 0\n        self._actions_this_level = 0\n        self._completed: list[LevelResult] = []\n\n    @property\n    def level_index(self) -> int:\n        return self._level_index\n\n    @property\n    def actions_this_level(self) -> int:\n        return self._actions_this_level\n\n    def baseline_here(self) -> Optional[int]:\n        """Human baseline for the current level, or None if withheld."""\n        if self.baselines is None or not 0 <= self._level_index < len(self.baselines):\n            return None\n        return self.baselines[self._level_index]\n\n    def remaining_at_par(self) -> Optional[int]:\n        """Actions left before this level\'s score starts dropping below 1.0.\n\n        ``None`` when baselines are withheld - the planner then has no cost\n        model and must fall back on something else (D-012).\n        """\n        b = self.baseline_here()\n        return None if b is None else b - self._actions_this_level\n\n    def observe(self, obs) -> None:\n        """Record one step. Errors and no-ops count: the environment charges them."""\n        if obs.action is None:\n            return\n        self.actions_used += 1\n        self._actions_this_level += 1\n        if obs.full_reset:\n            # Back to level 0 with a clean slate; prior level results stand.\n            self._level_index = 0\n            self._actions_this_level = 0\n            return\n        if obs.levels_completed > self._level_index:\n            # One or more levels cleared by this action.\n            while self._level_index < obs.levels_completed:\n                base = (\n                    self.baselines[self._level_index]\n                    if self.baselines is not None and self._level_index < len(self.baselines)\n                    else None\n                )\n                self._completed.append(\n                    LevelResult(\n                        index=self._level_index,\n                        actions=self._actions_this_level,\n                        completed=True,\n                        baseline=base,\n                    )\n                )\n                self._level_index += 1\n                self._actions_this_level = 0\n\n    def result(self) -> GameResult:\n        """The run so far, with unreached levels recorded as incomplete."""\n        levels = list(self._completed)\n        done = {lv.index for lv in levels}\n        for i in range(self.total_levels):\n            if i not in done:\n                base = (\n                    self.baselines[i]\n                    if self.baselines is not None and i < len(self.baselines)\n                    else None\n                )\n                actions = self._actions_this_level if i == self._level_index else 0\n                levels.append(LevelResult(index=i, actions=actions, completed=False, baseline=base))\n        levels.sort(key=lambda lv: lv.index)\n        return GameResult(\n            game_id=self.game_id,\n            levels=tuple(levels),\n            total_levels=self.total_levels,\n            actions_used=self.actions_used,\n        )\n'), 'arc3.client.env': (False, '"""The environment client - the only place in the codebase that spends actions.\n\nTwo rules this module exists to enforce:\n\n1. **Stepping is visible.** :meth:`EnvClient.step` is the sole way to advance an\n   environment, it always returns an :class:`Observation` carrying the running\n   ``actions_used``, and nothing else in ``arc3`` may import ``arc_agi``.\n   Actions are the scarce resource (CLAUDE.md); internal compute is free.\n2. **Errors are not no-ops.** ``LocalEnvironmentWrapper.step`` catches every\n   exception and returns ``None``. Silently treating that as "nothing happened"\n   would corrupt dynamics induction with fabricated null transitions, so a\n   ``None`` return becomes an :class:`Observation` with ``error`` set, and it is\n   *not* counted as a no-op.\n"""\n\nfrom __future__ import annotations\n\nimport logging\nfrom typing import Any, Optional, Protocol, runtime_checkable\n\nfrom ..types import Action, Episode, GameMetadata, GameState, Observation, as_sequence\n\nlog = logging.getLogger(__name__)\n\n#: Pinned in pyproject; recorded into every episode so replays are traceable to\n#: the engine that produced them (D-008).\nENGINE_VERSION = "arc-agi==0.9.9"\n\n\n@runtime_checkable\nclass Client(Protocol):\n    """What perception and (later) the agent may assume about an environment.\n\n    :class:`EnvClient` and :class:`~arc3.client.replay.ReplayClient` both satisfy\n    this, so every test can run offline against recorded fixtures and no test can\n    accidentally step a live environment.\n    """\n\n    @property\n    def game_id(self) -> str: ...\n\n    @property\n    def metadata(self) -> GameMetadata: ...\n\n    @property\n    def actions_used(self) -> int: ...\n\n    @property\n    def current(self) -> Observation: ...\n\n    def step(self, action: Action) -> Observation: ...\n\n\ndef make_arcade(\n    *, offline: bool = True, environments_dir: str = "environment_files"\n) -> Any:\n    """One engine handle, for callers that build many clients in a process.\n\n    :class:`EnvClient` constructs its own ``Arcade`` when it is not given one,\n    which is right for a single agent and wrong for an evaluation harness: each\n    instance loads and caches the environment files, so a script that builds one\n    client per game per driver holds dozens of copies. `eval_exploration.py`\n    building 75 clients exhausted memory and died inside the connected-component\n    labeller, which had nothing to do with it — the traceback simply lands\n    wherever the next allocation falls.\n\n    Scripts that loop over games should build one of these and pass it to every\n    client.\n    """\n    import arc_agi\n    from arc_agi import OperationMode\n\n    quiet = logging.getLogger("arc3.engine")\n    quiet.setLevel(logging.ERROR)\n    return arc_agi.Arcade(\n        operation_mode=OperationMode.OFFLINE if offline else OperationMode.NORMAL,\n        environments_dir=environments_dir,\n        logger=quiet,\n    )\n\n\nclass EnvClient:\n    """A typed wrapper over ``arc_agi``\'s environment wrapper.\n\n    Constructing the client performs the engine\'s implicit reset, which the\n    engine does **not** charge as an action, so ``actions_used`` starts at 0 and\n    the initial observation is available as :attr:`current` before any step.\n    """\n\n    def __init__(\n        self,\n        game_id: str,\n        *,\n        arcade: Any = None,\n        seed: int = 0,\n        offline: bool = True,\n        environments_dir: str = "environment_files",\n        record: bool = True,\n        withhold_metadata: bool = False,\n    ) -> None:\n        import arc_agi  # imported lazily: perception must not depend on the engine\n        from arc_agi import OperationMode\n\n        self._seed = seed\n        if arcade is None:\n            quiet = logging.getLogger("arc3.engine")\n            quiet.setLevel(logging.ERROR)\n            arcade = arc_agi.Arcade(\n                operation_mode=OperationMode.OFFLINE if offline else OperationMode.NORMAL,\n                environments_dir=environments_dir,\n                logger=quiet,\n            )\n        self._arcade = arcade\n\n        env = arcade.make(game_id, seed=seed)\n        if env is None:\n            raise RuntimeError(\n                f"could not make environment {game_id!r}; "\n                f"is {environments_dir}/ populated? see scripts/fetch_environments.py"\n            )\n        self._env = env\n\n        info = env.info\n        meta = GameMetadata(\n            game_id=info.game_id,\n            win_levels=int(getattr(env.observation_space, "win_levels", 0) or 0),\n            title=info.title,\n            tags=tuple(info.tags) if info.tags else None,\n            baseline_actions=(\n                tuple(int(b) for b in info.baseline_actions) if info.baseline_actions else None\n            ),\n        )\n        self._metadata = meta.withheld() if withhold_metadata else meta\n\n        self._actions_used = 0\n        self._current = self._wrap(env.observation_space, action=None)\n        self.episode: Optional[Episode] = None\n        if record:\n            self.episode = Episode(\n                game_id=self._metadata.game_id,\n                seed=seed,\n                metadata=self._metadata,\n                engine_version=ENGINE_VERSION,\n            )\n            self.episode.observations.append(self._current)\n\n    # -- Client protocol ---------------------------------------------------\n\n    @property\n    def game_id(self) -> str:\n        return self._metadata.game_id\n\n    @property\n    def metadata(self) -> GameMetadata:\n        return self._metadata\n\n    @property\n    def actions_used(self) -> int:\n        """Environment actions spent so far. This is the score denominator."""\n        return self._actions_used\n\n    @property\n    def current(self) -> Observation:\n        return self._current\n\n    def step(self, action: Action) -> Observation:\n        """Spend one action. **This is the expensive call.**\n\n        Counts the action whatever comes back, including errors and no-ops,\n        because the environment counts it too.\n        """\n        from arcengine import GameAction\n\n        self._actions_used += 1\n        try:\n            raw = self._env.step(GameAction.from_id(action.id), data=action.data)\n        except Exception as exc:  # pragma: no cover - wrapper already catches\n            log.exception("engine raised on %s", action)\n            obs = self._error_observation(action, f"{type(exc).__name__}: {exc}")\n        else:\n            if raw is None:\n                # The wrapper swallowed an exception. Distinct from a no-op.\n                obs = self._error_observation(action, "engine wrapper returned None")\n            else:\n                obs = self._wrap(raw, action=action)\n        self._current = obs\n        if self.episode is not None:\n            self.episode.observations.append(obs)\n        return obs\n\n    # -- helpers -----------------------------------------------------------\n\n    def _wrap(self, raw: Any, action: Optional[Action]) -> Observation:\n        return Observation(\n            frames=as_sequence(raw.frame),\n            state=GameState(raw.state.value if hasattr(raw.state, "value") else str(raw.state)),\n            levels_completed=int(raw.levels_completed),\n            win_levels=int(raw.win_levels),\n            available_actions=tuple(int(a) for a in raw.available_actions),\n            full_reset=bool(getattr(raw, "full_reset", False)),\n            action=action,\n            actions_used=self._actions_used,\n        )\n\n    def _error_observation(self, action: Action, message: str) -> Observation:\n        prev = self._current\n        return Observation(\n            frames=(),\n            state=prev.state,\n            levels_completed=prev.levels_completed,\n            win_levels=prev.win_levels,\n            available_actions=prev.available_actions,\n            action=action,\n            actions_used=self._actions_used,\n            error=message,\n        )\n\n    # -- engine handle, for the oracle only --------------------------------\n\n    @property\n    def _engine_game(self) -> Any:\n        """The live ``ARCBaseGame``. **Test-only** - see D-010.\n\n        Reaching through this from anything on the agent\'s code path is\n        cheating: it exposes exact sprite identities that will not exist on the\n        private set. ``tests/test_oracle_boundary.py`` enforces that only\n        ``arc3.oracle`` touches it.\n        """\n        return self._env._game\n\n\ndef available_environments(\n    environments_dir: str = "environment_files",\n) -> list[str]:\n    """Game ids present in the local cache, sorted."""\n    import arc_agi\n    from arc_agi import OperationMode\n\n    quiet = logging.getLogger("arc3.engine")\n    quiet.setLevel(logging.ERROR)\n    arcade = arc_agi.Arcade(\n        operation_mode=OperationMode.OFFLINE, environments_dir=environments_dir, logger=quiet\n    )\n    return sorted(e.game_id for e in arcade.get_environments())\n'), 'arc3.client.replay': (False, '"""Recording and replay.\n\nEvery interaction is written to a self-describing JSONL file so that experiments\nare reproducible offline without spending an action (D-008). The engine\'s own\n``save_recording`` writes something similar, but omits the seed, the metadata and\nthe engine version - the three things you need to reproduce a run - so we keep\nour own format and read it back into the same :class:`~arc3.types.Episode` type\nthe live client produces.\n\n:class:`ReplayClient` satisfies the same :class:`~arc3.client.env.Client`\nprotocol as the live client, so perception tests run against real recorded\nframes with no engine installed and no possibility of stepping a live\nenvironment by accident.\n"""\n\nfrom __future__ import annotations\n\nimport gzip\nimport json\nfrom pathlib import Path\nfrom typing import IO, Any, Iterable, Optional\n\nfrom ..types import Action, Episode, GameMetadata, Observation\n\nFORMAT_VERSION = 1\n\n\ndef _open(path: Path, mode: str) -> IO[str]:\n    """Open ``path``, transparently gzipping when the name ends ``.gz``.\n\n    Frames are extremely repetitive, so gzip shrinks a recorded episode by\n    roughly 40x. That is the difference between fixtures being committable and\n    not, and the format is otherwise unchanged.\n    """\n    if path.suffix == ".gz":\n        return gzip.open(path, mode + "t", encoding="utf-8")  # type: ignore[return-value]\n    return path.open(mode, encoding="utf-8")\n\n\ndef write_episode(episode: Episode, path: str | Path, *, include_frames: bool = True) -> Path:\n    """Write ``episode`` to a JSONL file. First line is the header."""\n    p = Path(path)\n    p.parent.mkdir(parents=True, exist_ok=True)\n    header = {\n        "format_version": FORMAT_VERSION,\n        "kind": "header",\n        "game_id": episode.game_id,\n        "seed": episode.seed,\n        "engine_version": episode.engine_version,\n        "metadata": episode.metadata.to_json(),\n        "notes": episode.notes,\n        "n_observations": len(episode.observations),\n        "actions_used": episode.actions_used,\n    }\n    with _open(p, "w") as f:\n        f.write(json.dumps(header) + "\\n")\n        for obs in episode.observations:\n            f.write(json.dumps(obs.to_json(include_frames=include_frames)) + "\\n")\n    return p\n\n\ndef read_episode(path: str | Path) -> Episode:\n    """Read an episode written by :func:`write_episode`."""\n    p = Path(path)\n    with _open(p, "r") as f:\n        lines = [ln for ln in (line.strip() for line in f) if ln]\n    if not lines:\n        raise ValueError(f"empty episode file: {p}")\n    header = json.loads(lines[0])\n    if header.get("kind") != "header":\n        raise ValueError(f"{p} does not start with an episode header")\n    if header.get("format_version") != FORMAT_VERSION:\n        raise ValueError(\n            f"{p} is format version {header.get(\'format_version\')}, "\n            f"this build reads {FORMAT_VERSION}"\n        )\n    episode = Episode(\n        game_id=header["game_id"],\n        seed=int(header["seed"]),\n        metadata=GameMetadata.from_json(header["metadata"]),\n        engine_version=header.get("engine_version", ""),\n        notes=header.get("notes", {}),\n    )\n    episode.observations = [Observation.from_json(json.loads(ln)) for ln in lines[1:]]\n    return episode\n\n\ndef iter_episodes(directory: str | Path) -> Iterable[Episode]:\n    """Every ``*.jsonl`` or ``*.jsonl.gz`` episode under ``directory``."""\n    paths = sorted(\n        list(Path(directory).glob("*.jsonl")) + list(Path(directory).glob("*.jsonl.gz"))\n    )\n    for p in paths:\n        yield read_episode(p)\n\n\nclass ReplayMismatch(RuntimeError):\n    """Raised when a replay is stepped with an action the recording did not take."""\n\n\nclass ReplayClient:\n    """Replays a recorded :class:`~arc3.types.Episode` as if it were live.\n\n    Stepping with an action other than the one recorded raises\n    :class:`ReplayMismatch` rather than silently returning the wrong frames -\n    a replay that quietly diverges from its recording is worse than no replay.\n    """\n\n    def __init__(self, episode: Episode, *, withhold_metadata: bool = False) -> None:\n        if not episode.observations:\n            raise ValueError("cannot replay an episode with no observations")\n        self._episode = episode\n        self._metadata = episode.metadata.withheld() if withhold_metadata else episode.metadata\n        self._index = 0\n        self._actions_used = 0\n\n    @classmethod\n    def from_file(cls, path: str | Path, **kwargs: Any) -> "ReplayClient":\n        return cls(read_episode(path), **kwargs)\n\n    @property\n    def game_id(self) -> str:\n        return self._episode.game_id\n\n    @property\n    def metadata(self) -> GameMetadata:\n        return self._metadata\n\n    @property\n    def actions_used(self) -> int:\n        return self._actions_used\n\n    @property\n    def current(self) -> Observation:\n        return self._episode.observations[self._index]\n\n    @property\n    def exhausted(self) -> bool:\n        return self._index >= len(self._episode.observations) - 1\n\n    @property\n    def next_action(self) -> Optional[Action]:\n        """The action the recording takes next, or None at the end."""\n        if self.exhausted:\n            return None\n        return self._episode.observations[self._index + 1].action\n\n    def step(self, action: Action) -> Observation:\n        if self.exhausted:\n            raise ReplayMismatch(\n                f"replay of {self.game_id} is exhausted after "\n                f"{self._actions_used} actions; asked for {action}"\n            )\n        nxt = self._episode.observations[self._index + 1]\n        if nxt.action != action:\n            raise ReplayMismatch(\n                f"replay of {self.game_id} step {self._actions_used + 1}: "\n                f"recorded {nxt.action}, asked for {action}"\n            )\n        self._index += 1\n        self._actions_used += 1\n        return nxt\n\n    def replay_all(self) -> Episode:\n        """Run the recording to its end and return the episode."""\n        while not self.exhausted:\n            action = self.next_action\n            assert action is not None\n            self.step(action)\n        return self._episode\n'), 'arc3.drive': (False, '"""Driving policies — how a rollout decides what to do next.\n\nThis is not an agent. It has no goal, no model and no memory of consequences.\nIts only job is to generate a *corpus of transitions* for dynamics induction to\nlearn from, and the thing that matters about it is the **distribution** of\ntransitions it produces, not whether it wins anything.\n\nWhy this module exists at all (D-019, D-020, D-026): every perception number in\nthe repository was measured under uniformly random play, whose transitions turn\nout to be small and low-diversity — 1–58 changed pixels of 4,096, heavily\ndominated by recolouring.\n\nThe history is worth keeping straight, because the obvious fix was tried and\nfailed before the real one was found. Round-robin over the action mask was\nproposed as the improvement and **measured worse** than random on event\ndiversity (D-020), so random was retained. D-021 later mapped the click space\nexhaustively and showed that the thing actually being wasted was not the action\nchoice but the *click placement*: object-derived targets reach 100% of a game\'s\ndistinct click outcomes on 13 of 15 live games, while as few as 2.3% of raw\npixels do anything at all. `RandomObjectDriver` is the combination those two\nresults point at, and it did not exist until D-026.\n\n**The action space is not uniform across games**, and the driver has to respect\nthat. Measured over all 25 public demo games at level 1:\n\n- ``RESET`` (action 0) **never appears** in ``available_actions``. Cycling over\n  the published mask therefore cannot accidentally restart a level, and nothing\n  here needs to special-case it.\n- **6 games publish only ``[6]``** — the click is the only input they accept, so\n  the action axis is degenerate and *where you click is the entire policy*.\n- **6 games never publish 6 at all** and are keyboard-only, so click targeting is\n  irrelevant to them.\n- 19 games publish the click at some point; the mask was stable within a game\n  over 30 steps.\n\nThat split is why click targeting is a first-class axis here rather than a\ndetail. For a quarter of the corpus it is the only source of variation.\n"""\n\nfrom __future__ import annotations\n\nimport random\nfrom dataclasses import dataclass, field\nfrom typing import Optional, Protocol, Sequence\n\nimport numpy as np\n\nfrom .perception.segment import background_colour, segment\nfrom .types import CLICK, GRID_H, GRID_W, Action, Observation\n\n#: Canonical order the round-robin cursor walks. RESET is deliberately absent:\n#: it is never offered by the engine, and stepping it would silently restart the\n#: level and corrupt the transition corpus with a discontinuity.\nACTION_CYCLE: tuple[int, ...] = (1, 2, 3, 4, 5, 6, 7)\n\n\nclass Driver(Protocol):\n    """Chooses the next action. Stateful, deterministic given its seed."""\n\n    @property\n    def name(self) -> str: ...\n\n    def next_action(self, obs: Observation) -> Optional[Action]:\n        """The next action, or ``None`` when the rollout should stop."""\n        ...\n\n\ndef _legal(obs: Observation) -> list[int]:\n    """Available action ids, RESET excluded, in ascending order."""\n    return sorted(a for a in obs.available_actions if a != 0)\n\n\n@dataclass\nclass RandomDriver:\n    """Uniform random over legal actions, with uniform random click pixels.\n\n    The policy every session-1 measurement used. Kept so that "round-robin is\n    better" is a comparison against something reproducible rather than a claim.\n    """\n\n    seed: int = 0\n    _rng: random.Random = field(init=False)\n\n    def __post_init__(self) -> None:\n        self._rng = random.Random(self.seed)\n\n    @property\n    def name(self) -> str:\n        return "random"\n\n    def next_action(self, obs: Observation) -> Optional[Action]:\n        acts = _legal(obs)\n        if not acts or obs.state.is_terminal:\n            return None\n        a = self._rng.choice(acts)\n        if a == CLICK:\n            return Action.click(self._rng.randrange(GRID_W), self._rng.randrange(GRID_H))\n        return Action.simple(a)\n\n\ndef lattice_targets(step: int = 8) -> list[tuple[int, int]]:\n    """A fixed coarse grid of click points covering the frame.\n\n    The null hypothesis for click targeting: no perception involved, just sweep\n    the frame. If object-targeting cannot beat this, the pruning is not earning\n    anything on the click-only games and we should know that.\n    """\n    half = step // 2\n    return [(x, y) for y in range(half, GRID_H, step) for x in range(half, GRID_W, step)]\n\n\ndef object_targets(frame: np.ndarray) -> list[tuple[int, int]]:\n    """One click point per candidate object, unioned across all segmentations.\n\n    Deliberately the union and deliberately without a ``Geometry``: the driver\n    only needs a plausible pixel *inside* something, not the best possible\n    carving, and making rollout generation depend on geometry — which finds\n    structure on only 4 of 25 games — would couple the corpus to the weakest\n    component in the pipeline.\n\n    **Includes the background**, because ``colour_4``, ``colour_8``,\n    ``colour_mass`` and ``bbox`` all partition the entire frame, so the\n    background region is a component like any other and contributes its own\n    click target — usually near the middle of the frame. That is correct for\n    D-005\'s purpose, where the cost of dropping a live target is worse than the\n    cost of carrying a dead one, and it is why D-021\'s proposal counts are as\n    large as they are. A driver that wants to *spend* actions should prefer\n    :func:`foreground_object_targets`.\n    """\n    return segment(frame).union_click_targets()\n\n\ndef foreground_object_targets(frame: np.ndarray) -> list[tuple[int, int]]:\n    """Object click points excluding the frame\'s background colour.\n\n    The version a driver should use. `object_targets` includes the background\n    region\'s own centroid, and on most frames that is one of the least\n    informative pixels available — it was caught by a test asserting that this\n    driver\'s clicks land inside objects, which failed on (32,32).\n\n    Not a hard exclusion of background clicks from the action space, which would\n    be a Guardrail 1 rejection of a whole region on no evidence: it is a\n    preference applied when generating a corpus, and the caller falls back to\n    the full set when nothing survives. Whether background clicks ever matter is\n    answerable from the click map (D-021) and has not been checked.\n    """\n    bg = background_colour(frame)\n    out = [(x, y) for (x, y) in object_targets(frame) if int(frame[y, x]) != bg]\n    return out\n\n\n@dataclass\nclass RoundRobinDriver:\n    """Cycle the legal actions in order; cycle click targets independently.\n\n    Two cursors, advanced separately, because the two axes carry different\n    amounts of information in different games. On a keyboard-only game the\n    target cursor never moves; on a click-only game the action cursor never\n    moves and the target cursor is doing all the work.\n\n    Round-robin rather than sampling because it **guarantees coverage**: every\n    legal action gets tried a comparable number of times, which is what a\n    dynamics corpus needs. Uniform sampling leaves coverage to luck and, over the\n    short rollouts we can afford, reliably under-samples something.\n\n    This is a deliberate placeholder. It has no notion of which actions did\n    anything, so it will keep spending actions on inputs already known to be\n    inert. Replacing it with something that follows information is future work\n    (D-020).\n    """\n\n    target_policy: str = "objects"  # "objects" | "lattice"\n    seed: int = 0\n    lattice_step: int = 8\n    _action_cursor: int = field(default=0, init=False)\n    _target_cursor: int = field(default=0, init=False)\n\n    @property\n    def name(self) -> str:\n        return f"round_robin[{self.target_policy}]"\n\n    def next_action(self, obs: Observation) -> Optional[Action]:\n        acts = _legal(obs)\n        if not acts or obs.state.is_terminal:\n            return None\n\n        # Walk the canonical order until we land on something legal, so coverage\n        # stays even even if the published mask changes between steps.\n        chosen = None\n        for _ in range(len(ACTION_CYCLE)):\n            candidate = ACTION_CYCLE[self._action_cursor % len(ACTION_CYCLE)]\n            self._action_cursor += 1\n            if candidate in acts:\n                chosen = candidate\n                break\n        if chosen is None:  # pragma: no cover - _legal() non-empty makes this unreachable\n            chosen = acts[0]\n\n        if chosen != CLICK:\n            return Action.simple(chosen)\n\n        frame = obs.last\n        targets: Sequence[tuple[int, int]]\n        if self.target_policy == "lattice" or frame is None:\n            targets = lattice_targets(self.lattice_step)\n        else:\n            targets = object_targets(frame) or lattice_targets(self.lattice_step)\n\n        x, y = targets[self._target_cursor % len(targets)]\n        self._target_cursor += 1\n        return Action.click(int(x), int(y))\n\n\n@dataclass\nclass RandomObjectDriver:\n    """Uniform random over legal actions, with clicks aimed at segmented objects.\n\n    The missing cell in the driver design space, and the one the evidence\n    actually points at. Two axes were varied independently across D-020 and\n    D-021 and never combined:\n\n    - **Action selection.** D-020 measured uniform random against a cyclic\n      round-robin over all 25 games and random won on the metric that matters\n      for a learning corpus, event diversity (40.1 distinct events per game\n      against 36.9). The likely mechanism, offered there as a hypothesis, is\n      that a deterministic engine plus a strictly periodic policy falls into\n      limit cycles that a random walk breaks out of.\n    - **Click placement.** D-021 then mapped every one of the 4,096 pixels on\n      all 19 click-publishing games and found object-derived targets reach\n      **100% of a game\'s distinct click outcomes on 13 of 15** live games,\n      against as little as 33% for a blind lattice sweep. It also found the live\n      fraction can be tiny: 7.0% of pixels on ft09, 2.3% on lp85.\n\n    So random *actions* and object-targeted *clicks* are each the measured\n    winner on their own axis, and `RandomDriver` — the driver every corpus in\n    this repository was recorded with — combines random actions with **random\n    pixels**. On a game where 2.3% of pixels do anything, that throws away\n    roughly forty-nine clicks in fifty.\n\n    That matters beyond the no-op rate. The rule language\'s only relational\n    feature is ``clicked`` (D-024), which is true when the action landed inside\n    a given object. Learning anything from it requires hitting the *same*\n    object repeatedly, and scattering clicks uniformly over 4,096 pixels almost\n    never does. Under `RandomDriver` the corpus is close to structurally\n    incapable of teaching a click rule, which is a confound in D-025\'s finding\n    that every click-only game scored zero.\n\n    One change from `RandomDriver`, deliberately: the click target is drawn from\n    the object proposal set instead of the pixel grid. Nothing else moves, so a\n    comparison between the two isolates that single variable.\n\n    **What this does not do.** It does not prefer targets it has not tried, does\n    not avoid objects already known to be inert, and does not merge targets that\n    produced identical outcomes — all of which D-021 decision 2 argues for and\n    all of which are information-seeking behaviour that D-020 deliberately\n    deferred until there is a model to seek information *for*.\n    """\n\n    seed: int = 0\n    _rng: random.Random = field(init=False)\n\n    def __post_init__(self) -> None:\n        self._rng = random.Random(self.seed)\n\n    @property\n    def name(self) -> str:\n        return "random_objects"\n\n    def next_action(self, obs: Observation) -> Optional[Action]:\n        acts = _legal(obs)\n        if not acts or obs.state.is_terminal:\n            return None\n        a = self._rng.choice(acts)\n        if a != CLICK:\n            return Action.simple(a)\n        frame = obs.last\n        targets: Sequence[tuple[int, int]] = []\n        if frame is not None:\n            # Prefer real objects; fall back to the full union (which includes\n            # the background region) before falling back to raw pixels.\n            targets = foreground_object_targets(frame) or object_targets(frame)\n        if not targets:\n            # No segmentation to aim at; fall back to the pixel grid rather than\n            # skipping the action, so the action distribution is unchanged.\n            return Action.click(self._rng.randrange(GRID_W), self._rng.randrange(GRID_H))\n        x, y = self._rng.choice(targets)\n        return Action.click(int(x), int(y))\n\n\ndef make_driver(name: str, seed: int = 0) -> Driver:\n    """Build a driver by name, for scripts that take one as an argument."""\n    if name == "random":\n        return RandomDriver(seed=seed)\n    if name in ("round_robin", "objects"):\n        return RoundRobinDriver(target_policy="objects", seed=seed)\n    if name == "lattice":\n        return RoundRobinDriver(target_policy="lattice", seed=seed)\n    if name in ("random_objects", "random_obj"):\n        return RandomObjectDriver(seed=seed)\n    if name in ("explore", "exploring"):\n        # Imported lazily: explore.py depends on dynamics, and drive.py is\n        # imported by recording scripts that should not need the learner.\n        from .explore import ExploringDriver\n\n        return ExploringDriver(seed=seed)\n    raise ValueError(\n        f"unknown driver {name!r}; have random, random_objects, explore, "\n        f"round_robin, lattice"\n    )\n'), 'arc3.dynamics': (True, '"""Dynamics induction: learning (s, a) -> s\' from observed transitions.\n\nThe subsystem D-002 stakes the project on, and the one the dynamics experiment\nis the go/no-go for. Kept separable from ontology, goal and planner (D-004),\nbecause we expect to replace at least one of them wholesale and this is the\nmost likely candidate.\n\nRead `rules.py` first: it states what a hypothesis in this system is allowed to\nsay, and its list of what the language *cannot* express is the thing to consult\nbefore believing any result from here.\n"""\n\nfrom .induce import ObjectEvent, TransitionObservation, induce, observe_transition\nfrom .predict import (\n    Score,\n    predict_frame,\n    score_replay_baseline,\n    score_static_baseline,\n    score_transitions,\n)\nfrom .rules import ATTRIBUTES, Effect, Rule, RuleSet, Selector\n\n__all__ = [\n    "ATTRIBUTES",\n    "Effect",\n    "ObjectEvent",\n    "Rule",\n    "RuleSet",\n    "Score",\n    "Selector",\n    "TransitionObservation",\n    "induce",\n    "observe_transition",\n    "predict_frame",\n    "score_replay_baseline",\n    "score_static_baseline",\n    "score_transitions",\n]\n'), 'arc3.dynamics.induce': (False, '"""Turning observed transitions into rules, and choosing between rule sets.\n\nTwo jobs, kept separate because they fail differently.\n\n**Observation** reads a transition through a fixed segmentation and reports, per\nobject present before, which of the four effects it underwent. This is pure\nperception bookkeeping and its failures are perception failures.\n\n**Induction** takes those observations and picks a rule set by **minimum\ndescription length**: total bits to write the rules down, plus total bits to\nwrite down the examples the rules get wrong. The hypothesis space searched is\nevery selector of arity 0, 1 or 2 over :data:`~arc3.dynamics.rules.ATTRIBUTES`,\nwhich is 1 + 5 + 10 = 16 candidate partitions per action — small enough to\nenumerate exhaustively, so there is no search heuristic to tune and no local\noptimum to worry about.\n\nMDL rather than accuracy because accuracy alone always prefers the finest\npartition: splitting on ``size`` gives almost every object its own rule and\nscores perfectly on the training set. The bit cost of naming all those rules is\nwhat stops it, and it stops it without a tuned regularisation constant — the\ncosts come from the sizes of the spaces actually being encoded.\n\n**What this does not do.**\n\n- **It is a point hypothesis, not a population.** Guardrail 1 and D-003 require a\n  weighted population of hypotheses with soft down-weighting; this returns the\n  single MDL-best rule set and throws the runners-up away. That is a deliberate\n  narrowing *for the experiment only*: the question on the table is whether\n  object-level rules beat chance at all, and a population cannot rescue a\n  language that cannot express the dynamics. If the experiment succeeds, the\n  population is the first thing to add, and the MDL scores computed here are\n  already the natural weights.\n- **It does not use the intermediate animation frames.** `observe_transition`\n  reads the resting endpoints only. The corpus carries whole animations and\n  `state-of-play.md` argues at length that the trajectory is far more\n  constraining than the endpoints — that remains true and unexploited. Endpoint\n  prediction is the harder target and the one D-019 defined the metric on, so it\n  is what is measured first.\n- **It cannot represent an object that both moves and recolours.** Motion wins,\n  and the recolour is silently dropped, which will cost exact-frame matches on\n  any game that does both at once.\n"""\n\nfrom __future__ import annotations\n\nimport math\nfrom collections import Counter, defaultdict\nfrom dataclasses import dataclass\nfrom itertools import combinations\nfrom typing import Any, Iterable, Optional, Sequence\n\nimport numpy as np\n\nfrom ..perception.geometry import Geometry\nfrom ..perception.segment import Segmentation, segment\nfrom ..perception.track import match_frames\nfrom ..types import Action, Transition\nfrom .rules import (\n    ATTRIBUTES,\n    BECOME,\n    RECOLOUR,\n    STATIC,\n    STATIC_EFFECT,\n    TRANSLATE,\n    VANISH,\n    Effect,\n    Rule,\n    RuleSet,\n    Selector,\n    describe_object,\n)\n\n\n@dataclass(frozen=True, slots=True)\nclass ObjectEvent:\n    """One object\'s observed fate across one transition."""\n\n    attrs: tuple[tuple[str, Any], ...]\n    effect: Effect\n\n    def as_dict(self) -> dict[str, Any]:\n        return dict(self.attrs)\n\n\n@dataclass\nclass TransitionObservation:\n    """Everything one transition contributes as evidence."""\n\n    action: Action\n    events: tuple[ObjectEvent, ...]\n    n_appeared: int\n    changed: bool\n    n_moves: int = 0\n    n_moves_deformed: int = 0\n    n_blind: int = 0\n    shapes: dict = None  # type: ignore[assignment]\n\n    def __post_init__(self) -> None:\n        if self.shapes is None:\n            self.shapes = {}\n\n    @property\n    def is_predictable_in_principle(self) -> bool:\n        """False when something appeared: no per-before-object rule can say it."""\n        return self.n_appeared == 0\n\n\ndef _effect_from_match(prev, nxt, same_colour: bool, before=None, after=None) -> tuple:\n    """Classify one matched pair into the closed effect vocabulary.\n\n    Returns ``(effect, blind)``, where ``blind`` marks a pair whose pixels\n    demonstrably changed while every property the vocabulary can see stayed the\n    same. See below.\n\n    Displacement is taken from the **bounding-box corner**, not the centroid.\n    `match_frames` rounds a centroid difference, which is correct for scoring a\n    pairing but off by a pixel when the shape changed, and a one-pixel error\n    makes an exact-frame prediction fail. The corner delta is exact whenever the\n    silhouette is preserved, which is the only case a translate should be\n    claimed for.\n\n    **Partial recolouring is invisible to this vocabulary, and pretending\n    otherwise corrupted a headline metric.** A component can keep its mask, its\n    size, its bounding box and its *dominant* colour while some of its pixels\n    change colour. `foreground_*` carvings make this common, because a component\n    there is "anything that is not background" and its internal colouring is not\n    part of its identity. The observation then reports ``static``, the model\n    predicts ``static``, the two agree, and `changed_obj_acc` scores the pair as\n    correct while the rendered frame is wrong.\n\n    Measured on s5i5 under `foreground_4`: **100% changed-object accuracy, zero\n    deformed moves, zero appearances, and a 2-pixel median residual outside the\n    bookkeeping band** — residuals which turned out to be pixels going 11->5 and\n    13->11 inside a component whose mask never moved. The model was not perfect;\n    the *observation* could not see what was wrong.\n\n    So when the frames are supplied, the pixel content is compared directly and\n    the mismatch is counted. The effect returned is still ``static``, because\n    the language genuinely cannot express a partial recolour — but the blindness\n    is now reported rather than scored as success.\n    """\n    dx = nxt.bbox.x0 - prev.bbox.x0\n    dy = nxt.bbox.y0 - prev.bbox.y0\n    same_silhouette = (\n        prev.bbox.width == nxt.bbox.width\n        and prev.bbox.height == nxt.bbox.height\n        and prev.size == nxt.size\n    )\n    if (dx or dy) and same_silhouette:\n        return Effect(TRANSLATE, dx=int(dx), dy=int(dy)), False\n    if not (dx or dy) and not same_colour:\n        return Effect(RECOLOUR, colour=int(nxt.dominant_colour)), False\n    if not (dx or dy) and same_silhouette:\n        blind = False\n        if before is not None and after is not None:\n            m = prev.mask\n            if not np.array_equal(before[m], after[m]):\n                blind = True\n        if blind:\n            # Same mask, same dominant colour, different pixels: a partial\n            # recolour. BECOME can state it, because the template carries the\n            # per-pixel colours rather than one colour.\n            return Effect(BECOME, dx=0, dy=0, shape=nxt.shape_key), True\n        return STATIC_EFFECT, False\n    # Moved and deformed, grew, shrank, or reshaped. All four are the same\n    # statement and BECOME is that statement (D-033).\n    return Effect(BECOME, dx=int(dx), dy=int(dy), shape=nxt.shape_key), False\n\n\ndef observe_transition(\n    transition: Transition,\n    name: str,\n    geometry: Optional[Geometry] = None,\n) -> TransitionObservation:\n    """Read one transition through one segmentation."""\n    before, after = transition.before, transition.after\n    seg_b: Segmentation = segment(before, geometry, include=(name,))[name]\n    seg_a: Segmentation = segment(after, geometry, include=(name,))[name]\n    matching = match_frames(seg_b, seg_a)\n\n    events: list[ObjectEvent] = []\n    shapes: dict = {}\n    n_moves = n_deformed = n_blind = 0\n    by_prev = matching.by_prev()\n    for i, comp in enumerate(seg_b.components):\n        m = by_prev.get(i)\n        if m is None:\n            effect = Effect(VANISH)\n        else:\n            nxt = seg_a.components[m.next_index]\n            effect, blind = _effect_from_match(comp, nxt, m.same_colour, before, after)\n            if effect.kind == BECOME and effect.shape not in shapes:\n                r = nxt.bbox\n                sub = nxt.mask[r.y0 : r.y1, r.x0 : r.x1]\n                shapes[effect.shape] = (sub, after[r.y0 : r.y1, r.x0 : r.x1])\n            if blind:\n                n_blind += 1\n            if (nxt.bbox.x0 - comp.bbox.x0) or (nxt.bbox.y0 - comp.bbox.y0):\n                n_moves += 1\n                if effect.kind != TRANSLATE:\n                    n_deformed += 1\n        attrs = describe_object(comp, transition.action)\n        events.append(ObjectEvent(attrs=tuple(sorted(attrs.items())), effect=effect))\n\n    return TransitionObservation(\n        action=transition.action,\n        events=tuple(events),\n        n_appeared=len(matching.appeared),\n        changed=not transition.is_noop,\n        n_moves=n_moves,\n        n_moves_deformed=n_deformed,\n        n_blind=n_blind,\n        shapes=shapes,\n    )\n\n\n# -- MDL ---------------------------------------------------------------------\n\n\ndef _bits(n: int) -> float:\n    """Bits to name one of ``n`` things. 0 when there is no choice to encode."""\n    return math.log2(n) if n > 1 else 0.0\n\n\ndef _partition_cost(\n    groups: dict[Any, list[Effect]],\n    n_effects: int,\n    n_examples: int,\n    n_candidate_partitions: int,\n    arity: int,\n) -> tuple[float, dict[Any, Effect], int, Effect]:\n    """Bits to describe one candidate partition, plus its per-group effects.\n\n    **Only groups that disagree with the default are charged for.** An earlier\n    version of this function charged bits for every group in the partition,\n    which prices the *partition* rather than the *description* and is therefore\n    not MDL at all. It had a sharp consequence, caught by printing the candidate\n    table on ls20: selecting on ``size`` made **1 error in 316** and was\n    rejected (cost 243) in favour of ``colour`` with **16 errors** (cost 190),\n    purely because ``size`` induces 37 groups. Thirty-five of those groups say\n    "static", which is the default and needs no rule written down.\n\n    The corrected accounting charges:\n\n    - naming the chosen partition, once;\n    - naming the default effect, once;\n    - per *exception group*: its attribute values plus its effect;\n    - per misfit example: which example, and its true effect.\n    """\n    chosen: dict[Any, Effect] = {}\n    errors = 0\n    tally: Counter = Counter()\n    for key, effects in groups.items():\n        best, count = Counter(effects).most_common(1)[0]\n        chosen[key] = best\n        errors += len(effects) - count\n        tally[best] += len(effects)\n\n    default = tally.most_common(1)[0][0]\n    n_rules = sum(1 for e in chosen.values() if e != default)\n\n    value_bits = arity * _bits(max(2, len(groups)))\n    model_bits = _bits(n_candidate_partitions) + _bits(n_effects)\n    model_bits += n_rules * (value_bits + _bits(n_effects))\n    error_bits = errors * (_bits(max(2, n_examples)) + _bits(n_effects))\n    return model_bits + error_bits, chosen, errors, default\n\n\ndef induce(\n    observations: Sequence[TransitionObservation],\n    *,\n    max_arity: int = 2,\n) -> RuleSet:\n    """Pick the MDL-best rule set over the observed events.\n\n    Exhaustive over selectors of arity 0..``max_arity``; no search heuristic.\n    """\n    by_action: dict[int, list[ObjectEvent]] = defaultdict(list)\n    for obs in observations:\n        for ev in obs.events:\n            by_action[obs.action.id].append(ev)\n\n    rules: list[Rule] = []\n    defaults: dict[int, Effect] = {}\n\n    attr_names = list(ATTRIBUTES)\n    combos: list[tuple[str, ...]] = [()]\n    for k in range(1, max_arity + 1):\n        combos.extend(combinations(attr_names, k))\n    n_candidates = len(combos)\n\n    for action_id, events in by_action.items():\n        if not events:\n            continue\n        effect_vocab = {ev.effect for ev in events}\n        n_effects = max(2, len(effect_vocab))\n        n_examples = len(events)\n\n        best: Optional[tuple[float, tuple[str, ...], dict[Any, Effect], Effect]] = None\n        for combo in combos:\n            groups: dict[Any, list[Effect]] = defaultdict(list)\n            for ev in events:\n                d = ev.as_dict()\n                key = tuple(d[a] for a in combo)\n                groups[key].append(ev.effect)\n            cost, chosen, _, default = _partition_cost(\n                groups, n_effects, n_examples, n_candidates, len(combo)\n            )\n            if best is None or cost < best[0]:\n                best = (cost, combo, chosen, default)\n\n        assert best is not None\n        _, combo, chosen, fallback = best\n        defaults[action_id] = fallback\n        if not combo:\n            continue\n\n        for key, effect in chosen.items():\n            if effect == fallback:\n                continue\n            conditions = tuple(zip(combo, key))\n            rules.append(Rule(action_id=action_id, selector=Selector(conditions), effect=effect))\n\n    rules.sort(key=lambda r: -r.selector.arity)\n    shapes: dict = {}\n    for obs in observations:\n        shapes.update(obs.shapes)\n    return RuleSet(rules=tuple(rules), defaults=defaults, shapes=shapes)\n'), 'arc3.dynamics.predict': (False, '"""Rendering a rule set\'s prediction, and scoring it.\n\nD-019 fixed the primary metric as **exact resting-frame match on transitions\nwhere something changed**, against a measured chance baseline of 0%. That\ndemands a full 64x64 frame out of the model, not a per-object verdict, so the\npredicted object states have to be rendered back into a grid.\n\nRendering is erase-then-paint in two passes. Objects that move or vanish have\ntheir old footprint filled with the background colour first, and only then is\neverything repainted; doing it per object instead lets one object\'s erase clip\nthe previous object\'s paint. The object\'s true per-pixel colours are read from\nthe *before* frame, because a sprite is not one colour.\n\n**Where this is optimistic and where it is pessimistic.**\n\n- *Pessimistic:* what is revealed behind a departing object has to be guessed.\n  :func:`~arc3.dynamics.rules.reveal_colour` guesses the modal colour of the\n  surrounding ring, which is right on a locally uniform surface and wrong where\n  an object conceals structure. The language has no notion of depth, so this is\n  a real and uncounted cost, not a scoring artefact. Erasing to the *global\n  background* instead — the obvious first implementation — was measured and\n  reproduced 0 of 100 frames on ls20 with perfect effects supplied.\n- *Pessimistic:* any transition in which something *appeared* is unpredictable\n  in principle (see :mod:`arc3.dynamics.rules`), and is scored as a miss rather\n  than excluded. `n_appeared` is reported so the ceiling is visible.\n- *Optimistic:* nothing. The comparison is `np.array_equal` on the whole grid.\n\nThe secondary object-level diagnostic is kept, per D-019, as a diagnostic only:\nit is scored in terms of a segmentation, and D-018 makes the segmentation the\naxis being swept, so a metric that varies with that axis cannot be the headline.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Optional\n\nimport numpy as np\n\nfrom ..perception.geometry import Geometry\nfrom ..perception.segment import background_colour, segment\nfrom ..types import Action, Transition\nfrom .rules import STATIC, VANISH, RuleSet, apply_effect, describe_object, reveal_colour\n\n\n#: A bookkeeping band may not be thicker than this many rows (of 64). Above\n#: it, the band is assumed to be playfield and is refused outright.\nMAX_BAND = 4\n\n\ndef bookkeeping_mask(transitions, min_fraction: float = 0.5) -> np.ndarray:\n    """Pixels that change on almost every action, whatever the action was.\n\n    `benchmark.md` correction #2 established that the 64x64 frame is a\n    **composite**: a camera view, letterboxed, with a UI layer drawn last at\n    full resolution. This function recovers that second layer from dynamics\n    alone, with no game knowledge: a move counter, timer or score bar ticks on\n    every action irrespective of which action it was, so its pixels change at a\n    rate no playfield object comes close to.\n\n    It exists because of a measured surprise. On ls20 the induced playfield\n    dynamics were *correct* and exact-frame prediction still scored zero: 71 of\n    100 transitions were wrong by exactly two pixels, at y=61-62, marching one\n    column per action as a colour-11 bar advanced over colour 3. A move counter\n    made a completely correct model of the game score 0%.\n\n    **Deliberately conservative, and horizontal only.** Two restrictions, both\n    chosen so that the failure mode is *missing* UI rather than *excusing* the\n    model from predicting the game:\n\n    - Only row bands are considered. A vertical side-panel UI is missed. That is\n      a stated limitation, and it makes this figure a lower bound.\n    - A band thicker than :data:`MAX_BAND` rows is refused. Without the cap the\n      heuristic eats the playfield: an object confined to a few rows or columns\n      makes them change in 100% of transitions, and labelling those bookkeeping\n      would *inflate* the score by excusing the model from the real dynamics.\n      A test asserts the refusal, because the first version of this function\n      accepted an 8-column playfield band.\n\n    Detection is at **band** level (whole rows), because a counter\n    advances across the frame: any individual pixel of it changes once, while\n    its row changes every time. Measured on ls20, the per-pixel version returns\n    an empty mask while rows 61-62 change in 99% of transitions against 10-70%\n    for the 25 playfield rows - a clean separation rather than a tuned one.\n\n    Must be computed on **training transitions only** — it is part of the model,\n    and fitting it on the test set would be leakage. The threshold is the one\n    tuned constant in the dynamics code and its sensitivity is reported by\n    `scripts/eval_dynamics.py --bookkeeping-sweep`.\n    """\n    changed = [t for t in transitions if not t.is_noop]\n    if not changed:\n        return np.zeros((64, 64), dtype=bool)\n    h, w = changed[0].before.shape\n    rows = np.zeros(h, dtype=np.int32)\n    for t in changed:\n        ys, _ = np.nonzero(t.before != t.after)\n        rows[np.unique(ys)] += 1\n    thresh = max(1, int(min_fraction * len(changed)))\n    mask = np.zeros((h, w), dtype=bool)\n    qualifying = np.nonzero(rows >= thresh)[0]\n    if len(qualifying) and len(qualifying) <= MAX_BAND:\n        mask[qualifying, :] = True\n    return mask\n\n\ndef predict_frame(\n    before: np.ndarray,\n    action: Action,\n    ruleset: RuleSet,\n    name: str,\n    geometry: Optional[Geometry] = None,\n) -> np.ndarray:\n    """The frame the rule set says follows ``action`` from ``before``."""\n    seg = segment(before, geometry, include=(name,))[name]\n    bg = seg.background_colour\n    if bg is None:\n        bg = background_colour(before)\n\n    canvas = before.copy()\n    effects = []\n    for comp in seg.components:\n        effect = ruleset.effect_for(action, describe_object(comp, action))\n        effects.append((comp, effect))\n\n    # Pass 1: erase the footprint of everything that is not staying put, to the\n    # colour that is actually revealed rather than to the global background.\n    for comp, effect in effects:\n        if effect.kind == STATIC:\n            continue\n        canvas[comp.mask] = reveal_colour(before, comp.mask, bg)\n\n    # Pass 2: paint.\n    for comp, effect in effects:\n        if effect.kind == STATIC:\n            continue\n        if effect.kind == VANISH:\n            continue\n        apply_effect(canvas, before, comp, effect, bg, ruleset.shapes)\n\n    return canvas\n\n\n@dataclass\nclass Score:\n    """Outcome of scoring one rule set over a set of transitions."""\n\n    n_total: int = 0\n    n_changed: int = 0\n    n_exact_changed: int = 0\n    n_exact_all: int = 0\n    n_appeared_blocked: int = 0\n    n_objects: int = 0\n    n_objects_correct: int = 0\n    n_exact_ex_bookkeeping: int = 0\n    n_changed_outside: int = 0\n    n_moves: int = 0\n    n_moves_deformed: int = 0\n    n_blind: int = 0\n    n_changed_objects: int = 0\n    n_changed_objects_correct: int = 0\n    residuals: list = None  # type: ignore[assignment]\n\n    def __post_init__(self) -> None:\n        if self.residuals is None:\n            self.residuals = []\n\n    @property\n    def exact_on_changed(self) -> float:\n        return self.n_exact_changed / self.n_changed if self.n_changed else 0.0\n\n    @property\n    def exact_overall(self) -> float:\n        return self.n_exact_all / self.n_total if self.n_total else 0.0\n\n    @property\n    def exact_ex_bookkeeping(self) -> float:\n        """Exact outside the bookkeeping layer, over transitions where that means something.\n\n        The denominator is deliberately ``n_changed_outside`` rather than\n        ``n_changed``. If every change a game makes falls *inside* the induced\n        mask, then "exact outside the mask" is satisfied by predicting nothing\n        and the figure is vacuous. That is not hypothetical: on vc33 **0%** of\n        changed transitions alter a pixel outside the mask, and an earlier\n        version of this property duly reported 100% for a model whose object\n        accuracy was 0%. Read `vacuous` before reading this.\n        """\n        return (\n            self.n_exact_ex_bookkeeping / self.n_changed_outside\n            if self.n_changed_outside\n            else 0.0\n        )\n\n    @property\n    def vacuous(self) -> bool:\n        """True when almost nothing changes outside the bookkeeping mask."""\n        return self.n_changed > 0 and self.n_changed_outside < 0.1 * self.n_changed\n\n    @property\n    def blind_rate(self) -> float:\n        """Objects whose pixels changed while the vocabulary saw nothing.\n\n        The correction factor on :attr:`changed_object_accuracy`. A carving that\n        cannot see a change cannot be scored wrong for it, so a coarse carving\n        buys object accuracy by being blind. Read the two together or neither.\n        """\n        return self.n_blind / self.n_objects if self.n_objects else 0.0\n\n    @property\n    def deformed_move_rate(self) -> float:\n        """Fraction of observed moves the effect vocabulary cannot express.\n\n        An object that moves *and* changes silhouette is recorded as ``static``,\n        because the language has no way to say it. Measured at 27-48% on most\n        games under ``colour_8`` and 100% on vc33, so this is the dominant known\n        loss in the whole pipeline, not a corner case.\n        """\n        return self.n_moves_deformed / self.n_moves if self.n_moves else 0.0\n\n    @property\n    def object_accuracy(self) -> float:\n        return self.n_objects_correct / self.n_objects if self.n_objects else 0.0\n\n    @property\n    def changed_object_accuracy(self) -> float:\n        """Accuracy on objects that actually did something.\n\n        The headline object accuracy is near-worthless on its own: ~91% of\n        observed objects are static, so "predict static for everything" scores\n        ~91% and looks like competence. This is the version that cannot be\n        gamed that way, and it is the same trap D-019 rejected per-pixel\n        accuracy for.\n        """\n        return (\n            self.n_changed_objects_correct / self.n_changed_objects\n            if self.n_changed_objects\n            else 0.0\n        )\n\n    @property\n    def median_residual(self) -> float:\n        return float(np.median(self.residuals)) if self.residuals else 0.0\n\n    @property\n    def noop_fraction(self) -> float:\n        return (self.n_total - self.n_changed) / self.n_total if self.n_total else 0.0\n\n\ndef score_transitions(\n    transitions,\n    ruleset: RuleSet,\n    name: str,\n    geometry: Optional[Geometry] = None,\n    ignore: Optional[np.ndarray] = None,\n) -> Score:\n    """Exact-frame match, plus the object-level diagnostics.\n\n    ``ignore`` is the induced bookkeeping mask (see :func:`bookkeeping_mask`),\n    used only for the secondary ``exact_ex_bookkeeping`` figure. The primary\n    ``exact_on_changed`` is always the whole frame, as D-019 specified.\n    """\n    from .induce import observe_transition\n\n    s = Score()\n    for t in transitions:\n        s.n_total += 1\n        changed = not t.is_noop\n        if changed:\n            s.n_changed += 1\n\n        pred = predict_frame(t.before, t.action, ruleset, name, geometry)\n        wrong = pred != t.after\n        hit = not bool(wrong.any())\n        if hit:\n            s.n_exact_all += 1\n            if changed:\n                s.n_exact_changed += 1\n        if changed:\n            s.residuals.append(int(wrong.sum()))\n            truly_changed = t.before != t.after\n            if ignore is not None:\n                outside = bool((truly_changed & ~ignore).any())\n                if outside:\n                    s.n_changed_outside += 1\n                    if not bool((wrong & ~ignore).any()):\n                        s.n_exact_ex_bookkeeping += 1\n            else:\n                s.n_changed_outside += 1\n                if hit:\n                    s.n_exact_ex_bookkeeping += 1\n\n        obs = observe_transition(t, name, geometry)\n        if obs.n_appeared and changed:\n            s.n_appeared_blocked += 1\n        for ev in obs.events:\n            s.n_objects += 1\n            predicted = ruleset.effect_for(t.action, ev.as_dict())\n            correct = predicted == ev.effect\n            if correct:\n                s.n_objects_correct += 1\n            if not ev.effect.is_identity:\n                s.n_changed_objects += 1\n                if correct:\n                    s.n_changed_objects_correct += 1\n        s.n_moves += obs.n_moves\n        s.n_moves_deformed += obs.n_moves_deformed\n        s.n_blind += obs.n_blind\n    return s\n\n\n# -- baselines ---------------------------------------------------------------\n\n\ndef score_static_baseline(transitions) -> Score:\n    """B0 from D-019: predict that nothing changes. Measured at 0% on changed."""\n    s = Score()\n    for t in transitions:\n        s.n_total += 1\n        if not t.is_noop:\n            s.n_changed += 1\n        if np.array_equal(t.before, t.after):\n            s.n_exact_all += 1\n    return s\n\n\ndef score_replay_baseline(transitions) -> Score:\n    """B1 from D-019: replay the exact pixel delta last seen for this action.\n\n    Rebuilt here rather than cited so the ladder is measured on the same corpus\n    as everything else. D-019 measured it at 0% on changed transitions, and the\n    reason it fails is the argument for representing dynamics over objects\n    rather than pixels.\n    """\n    s = Score()\n    last: dict[int, tuple[np.ndarray, np.ndarray]] = {}\n    for t in transitions:\n        s.n_total += 1\n        changed = not t.is_noop\n        if changed:\n            s.n_changed += 1\n        delta = last.get(t.action.id)\n        pred = t.before.copy()\n        if delta is not None:\n            idx, vals = delta\n            flat = pred.reshape(-1)\n            inb = idx < flat.size\n            flat[idx[inb]] = vals[inb]\n        if np.array_equal(pred, t.after):\n            s.n_exact_all += 1\n            if changed:\n                s.n_exact_changed += 1\n        diff = np.nonzero((t.after != t.before).reshape(-1))[0]\n        last[t.action.id] = (diff, t.after.reshape(-1)[diff])\n    return s\n'), 'arc3.dynamics.rules': (False, '"""The rule language: what a dynamics hypothesis is allowed to say.\n\nThis is the file D-019 named as the last thing blocking the dynamics experiment,\nand the file Guardrail 2 makes delicate. A rule language *is* a hypothesis\nspace, so writing one is exactly the act the guardrail warns about — "do not\nhand-craft the hypothesis space", on the evidence that Tufa Labs found\nhand-crafted tools actively hurt.\n\nThree constraints were adopted to keep the language honest rather than\ngame-flavoured:\n\n1. **A rule may only condition on attributes the perception layer already\n   emits.** No new features were invented for this file. ``colour``, ``size``,\n   ``shape`` and ``rect`` are fields of :class:`~arc3.perception.segment.Component`\n   that existed before dynamics did. The one addition, ``clicked``, is\n   *relational* rather than descriptive — whether the action\'s click landed\n   inside this object — and it is general: every game with a pointer has the\n   notion, and no game-specific content is encoded by it. D-021 is the argument\n   for it: a click\'s effect is about *what* you clicked, not about where the\n   pixel was in absolute terms.\n\n2. **The effect vocabulary is closed and tiny.** An object can hold still,\n   translate, change colour, or cease to exist. These are not a taxonomy of\n   games; they are the four things the *tracker* can report, so the language is\n   exactly as expressive as the perception beneath it and no more.\n\n3. **Which rules to keep is decided by description length, never by a tuned\n   threshold.** See :mod:`arc3.dynamics.induce`.\n\n**What this language cannot say**, stated plainly because the absences are what\na negative result will turn on:\n\n- **Nothing about objects that appear.** Every rule is keyed to an object\n  present in the *before* frame, so a frame containing something new is\n  unpredictable in principle. Appearances are counted and reported separately.\n- **No object-to-object interaction.** "The block stops when it hits a wall" is\n  unsayable. Rules are per-object and context-free apart from the click\n  relation, so collision, containment and support are all outside the language.\n- **No state beyond the current frame.** No velocity, no counters, no memory. A\n  game whose response depends on how you arrived is unmodellable.\n- **No numeric relations.** A selector tests attribute *equality*, so\n  "everything bigger than 4 pixels" cannot be said; it can only be approximated\n  by enumerating sizes, which MDL will correctly refuse to do.\n\nEach of those is a deliberate floor, not an oversight. The experiment asks\nwhether the *simplest* object-level language beats chance; starting richer would\nmean a positive result told us nothing about where the power came from.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Any, Callable, Optional\n\nimport numpy as np\n\nfrom ..perception.segment import Component\nfrom ..types import Action\n\n#: The four things the tracker can report about an object, and therefore the\n#: complete effect vocabulary. ``STATIC`` is listed first because it is the\n#: overwhelming majority of observations and the default prediction.\nSTATIC = "static"\nTRANSLATE = "translate"\nRECOLOUR = "recolour"\nVANISH = "vanish"\n\n#: The object is replaced by a different shape, optionally at an offset.\n#:\n#: Added after measuring what the four original effects could not express.\n#: Profiling the five games that produce abundant evidence and score exactly\n#: zero (D-032), **63% of their non-static object events were inexpressible**,\n#: and 86% of that gap is one thing: an object whose mask changes.\n#:\n#:   deformed-move  330  (37.5% of the gap)   moved AND changed silhouette\n#:   shrank         290  (33.0%)              same place, fewer pixels\n#:   grew           119  (13.5%)              same place, more pixels\n#:   reshaped        20  ( 2.3%)              same size, different mask\n#:\n#: All four are the same statement — *the object is now this other shape* — and\n#: `BECOME` is that statement. It is not a taxonomy of game mechanics; it is the\n#: remaining thing the tracker can report and the language could not repeat.\n#:\n#: The shape is carried as a `shape_key`, so two occurrences of the same form\n#: are the same effect and MDL can generalise over them. The pixel template it\n#: refers to lives in :attr:`RuleSet.shapes`, which keeps the effect small,\n#: hashable and comparable — storing a mask inline would make every effect\n#: unique and MDL would never generalise at all.\nBECOME = "become"\n\n\n@dataclass(frozen=True, slots=True)\nclass Effect:\n    """What happens to one object across one transition."""\n\n    kind: str\n    dx: int = 0\n    dy: int = 0\n    colour: int = -1\n    shape: bytes = b""\n\n    def __str__(self) -> str:\n        if self.kind == TRANSLATE:\n            return f"translate({self.dx:+d},{self.dy:+d})"\n        if self.kind == RECOLOUR:\n            return f"recolour(->{self.colour})"\n        if self.kind == BECOME:\n            return f"become(shape#{hash(self.shape) & 0xFFFF:04x},{self.dx:+d},{self.dy:+d})"\n        return self.kind\n\n    @property\n    def is_identity(self) -> bool:\n        return self.kind == STATIC\n\n\nSTATIC_EFFECT = Effect(STATIC)\n\n\ndef _attr_colour(c: Component, a: Action) -> Any:\n    return c.dominant_colour\n\n\ndef _attr_size(c: Component, a: Action) -> Any:\n    return c.size\n\n\ndef _attr_shape(c: Component, a: Action) -> Any:\n    return c.shape_key\n\n\ndef _attr_rect(c: Component, a: Action) -> Any:\n    return c.is_rectangular\n\n\ndef _attr_clicked(c: Component, a: Action) -> Any:\n    """Did this action\'s click land inside this object?\n\n    False for every non-click action, which makes the attribute useless on\n    keyboard-only games — correctly, since it has nothing to say there, and MDL\n    will decline to split on a constant.\n    """\n    if not a.is_click or a.x is None or a.y is None:\n        return False\n    m = c.mask\n    if not (0 <= a.y < m.shape[0] and 0 <= a.x < m.shape[1]):\n        return False\n    return bool(m[a.y, a.x])\n\n\n#: The attributes a selector may test. Every one is a pre-existing perception\n#: output; the list is deliberately short and deliberately not extensible by\n#: game.\nATTRIBUTES: dict[str, Callable[[Component, Action], Any]] = {\n    "colour": _attr_colour,\n    "size": _attr_size,\n    "shape": _attr_shape,\n    "rect": _attr_rect,\n    "clicked": _attr_clicked,\n}\n\n\ndef describe_object(component: Component, action: Action) -> dict[str, Any]:\n    """The attribute vector a rule is matched against."""\n    return {name: fn(component, action) for name, fn in ATTRIBUTES.items()}\n\n\n@dataclass(frozen=True, slots=True)\nclass Selector:\n    """Which objects a rule applies to: a conjunction of attribute equalities.\n\n    The empty conjunction is the universal selector, which is the cheapest thing\n    the language can say and the hypothesis every induction starts from. Only\n    *equality* is testable — there is no ``size > 4`` — so the language cannot\n    express a numeric relation and MDL will refuse to fake one by enumeration.\n    """\n\n    conditions: tuple[tuple[str, Any], ...] = ()\n\n    @property\n    def arity(self) -> int:\n        return len(self.conditions)\n\n    def matches(self, attrs: dict[str, Any]) -> bool:\n        return all(attrs.get(k) == v for k, v in self.conditions)\n\n    def __str__(self) -> str:\n        if not self.conditions:\n            return "any"\n        parts = []\n        for k, v in self.conditions:\n            if isinstance(v, bytes):\n                v = f"shape#{hash(v) & 0xFFFF:04x}"\n            parts.append(f"{k}={v}")\n        return " & ".join(parts)\n\n\n@dataclass(frozen=True, slots=True)\nclass Rule:\n    """``IF action AND selector THEN effect``."""\n\n    action_id: int\n    selector: Selector\n    effect: Effect\n\n    def __str__(self) -> str:\n        return f"action{self.action_id} & {self.selector} -> {self.effect}"\n\n\n@dataclass\nclass RuleSet:\n    """One induced model of a game\'s dynamics.\n\n    Rules are looked up by action and then matched in order of decreasing\n    selector arity, so a two-condition rule wins over a one-condition rule,\n    with a per-action default beneath both. Nothing here is probabilistic: this is the\n    MDL-selected point hypothesis. Maintaining a weighted *population* of these\n    is what Guardrail 1 demands and is **not** done yet — see the module\n    docstring of :mod:`arc3.dynamics.induce`.\n    """\n\n    rules: tuple[Rule, ...] = ()\n    defaults: dict[int, Effect] = None  # type: ignore[assignment]\n    #: shape_key -> bbox-relative boolean template, for rendering BECOME.\n    shapes: dict = None  # type: ignore[assignment]\n\n    def __post_init__(self) -> None:\n        if self.defaults is None:\n            self.defaults = {}\n        if self.shapes is None:\n            self.shapes = {}\n\n    def effect_for(self, action: Action, attrs: dict[str, Any]) -> Effect:\n        # `rules` is stored most-specific-first by the inducer.\n        for r in self.rules:\n            if r.action_id == action.id and r.selector.matches(attrs):\n                return r.effect\n        return self.defaults.get(action.id, STATIC_EFFECT)\n\n    @property\n    def size(self) -> int:\n        return len(self.rules) + len(self.defaults)\n\n    def describe(self, limit: int = 12) -> str:\n        lines = [f"{len(self.rules)} rules, {len(self.defaults)} defaults"]\n        for r in self.rules[:limit]:\n            lines.append(f"  {r}")\n        if len(self.rules) > limit:\n            lines.append(f"  ... and {len(self.rules) - limit} more")\n        for a, e in sorted(self.defaults.items()):\n            lines.append(f"  action{a} & any -> {e}   [default]")\n        return "\\n".join(lines)\n\n\ndef reveal_colour(source: np.ndarray, mask: np.ndarray, background: int) -> int:\n    """What colour is under an object when it leaves.\n\n    The naive answer — the frame\'s background colour — is wrong whenever objects\n    sit on a surface rather than on the void, which turns out to be most games.\n    On ls20 the sprite moves across floor colour 3 while the frame\'s background\n    is 4, so erasing to the background corrupted **every** transition and made\n    exact-frame prediction impossible even when every object\'s effect was known\n    exactly. That was measured, not guessed: with ground-truth effects supplied,\n    erase-to-background reproduced 0 of 100 frames with a median of 27 pixels\n    wrong.\n\n    The fix used here is the modal colour of the one-pixel ring immediately\n    outside the footprint. It is general — no game knowledge, no tuning — and it\n    is right whenever an object rests on a locally uniform surface, which is the\n    common case. It is *wrong* when an object straddles a boundary or conceals\n    something structured, and that residual is real and reported rather than\n    hidden: this function cannot recover information the frame does not contain.\n    A dynamics model that needs to know what is behind an object needs hidden\n    state, which the language deliberately does not have.\n    """\n    h, w = source.shape\n    ys, xs = np.nonzero(mask)\n    if len(ys) == 0:\n        return background\n    y0, y1 = max(0, ys.min() - 1), min(h, ys.max() + 2)\n    x0, x1 = max(0, xs.min() - 1), min(w, xs.max() + 2)\n    window = np.zeros((y1 - y0, x1 - x0), dtype=bool)\n    window[ys - y0, xs - x0] = True\n    dil = window.copy()\n    dil[1:, :] |= window[:-1, :]\n    dil[:-1, :] |= window[1:, :]\n    dil[:, 1:] |= window[:, :-1]\n    dil[:, :-1] |= window[:, 1:]\n    ring = dil & ~window\n    vals = source[y0:y1, x0:x1][ring]\n    if vals.size == 0:\n        return background\n    counts = np.bincount(vals.astype(np.int64))\n    return int(np.argmax(counts))\n\n\ndef apply_effect(\n    canvas: np.ndarray,\n    source: np.ndarray,\n    component: Component,\n    effect: Effect,\n    background: int,\n    shapes: Optional[dict] = None,\n) -> None:\n    """Draw ``component`` under ``effect`` onto ``canvas``, in place.\n\n    ``source`` is the *before* frame, read for the object\'s true per-pixel\n    colours: an object is not one colour, and repainting it with its dominant\n    colour would corrupt every multi-coloured sprite.\n\n    Assumes the caller has already erased the object\'s old footprint, so that a\n    translate does not leave a copy behind. Erase-then-paint in two passes over\n    all objects, never per object, or one object\'s paint is clipped by the next\n    one\'s erase. See :func:`reveal_colour` for what "erased" has to mean.\n    """\n    if effect.kind == VANISH:\n        return\n    mask = component.mask\n    ys, xs = np.nonzero(mask)\n    if effect.kind == STATIC:\n        canvas[ys, xs] = source[ys, xs]\n        return\n    if effect.kind == RECOLOUR:\n        canvas[ys, xs] = effect.colour\n        return\n    if effect.kind == TRANSLATE:\n        ny, nx = ys + effect.dy, xs + effect.dx\n        h, w = canvas.shape\n        keep = (ny >= 0) & (ny < h) & (nx >= 0) & (nx < w)\n        canvas[ny[keep], nx[keep]] = source[ys[keep], xs[keep]]\n        return\n    if effect.kind == BECOME:\n        tmpl = shapes.get(effect.shape) if shapes else None\n        if tmpl is None:\n            # Shape never seen at induction time: fall back to leaving the\n            # object where it was rather than erasing it, which is the less\n            # destructive of the two wrong answers.\n            canvas[ys, xs] = source[ys, xs]\n            return\n        mask_t, colour_t = tmpl\n        th, tw = mask_t.shape\n        y0 = component.bbox.y0 + effect.dy\n        x0 = component.bbox.x0 + effect.dx\n        h, w = canvas.shape\n        tys, txs = np.nonzero(mask_t)\n        ny, nx = tys + y0, txs + x0\n        keep = (ny >= 0) & (ny < h) & (nx >= 0) & (nx < w)\n        canvas[ny[keep], nx[keep]] = colour_t[tys[keep], txs[keep]]\n'), 'arc3.explore': (False, '"""Exploration that tries to make the game move, rather than sampling blindly.\n\nThe first component in this repository that is agent-like rather than a research\ninstrument: it learns from what comes back and changes what it does next.\n\n**Why this exists.** D-028 decomposed the sixteen zero-scoring games and found\nthat *half* of them are not a modelling failure at all — the agent simply never\nmakes the game do anything. D-030 then measured that the moves are there to be\nfound: vc33 moves the playfield on **2 of 256 probes at the level start and 254\nof 256 five actions later**, and merely aiming clicks at objects rather than\nrandom pixels multiplies playfield activity **2x to 20x** across all eight\nstarved games. Nothing here needs a better learner. It needs a policy that\nnotices what works.\n\nThree things follow from the measurements, and each is a design constraint:\n\n1. **Count playfield changes, not frame changes.** A move counter ticking is not\n   the game responding. vc33 changed its frame on 99 of 99 actions and its\n   playfield on **zero** of them (D-028); a policy rewarded for frame change\n   would have been perfectly satisfied. The bookkeeping band has to be estimated\n   online, which :class:`OnlineBand` does.\n\n2. **Arms are object *signatures*, not pixels.** Objects move, so a pixel that\n   paid off last turn means nothing this turn. An arm here is the attribute\n   vector the rule language already selects on, so experience transfers as\n   objects move around, and the explorer and the learner are looking at the same\n   features. This is the same signature D-026 used to measure whether a corpus\n   could teach a click rule.\n\n   **How coarse the key is matters more than anything else here, and the first\n   two guesses were both wrong.** The failure to explain was that on object-rich\n   games this policy was *less active than uniform sampling* — cd82 at 20%\n   against `random_objects`\' 79%, which a policy maximising activity should not\n   be able to manage.\n\n   The first guess was coverage: draw targets from all six carvings rather than\n   one, since D-021 showed the union is what reaches every distinct click\n   outcome. Measured, it made things **worse** — 45% mean activity against 58%,\n   and 25.2 distinct events against 35.8 — because it roughly doubled the arm\n   count (tn36 136 arms to 288) and Thompson sampling then spends its whole\n   budget establishing posteriors it never gets to exploit. `union_targets` is\n   kept as a measured-and-rejected option.\n\n   The actual constraint is **arms against action budget**. Keying on\n   ``(colour, size)`` gives 83-136 arms on object-rich games and there are only\n   100 actions. Pooling to ``colour`` alone gives 6-14, and at equal budget:\n\n   | | activity | distinct events |\n   |---|---|---|\n   | `random_objects` | 38% | 27.4 |\n   | `explore`, `(colour, size)` | 58% | 35.8 |\n   | **`explore`, `colour`** | **58%** | **44.9** |\n\n   Equal on activity, far better on the diversity a learning corpus actually\n   needs, and it repairs the regression (sk48 31%->52%, ar25 72%->85%, lp85\n   54%->77%). It is a genuine trade — sb26 falls 73%->37% and tn36 53%->36%,\n   games where size distinguishes the productive object. The principled fix for\n   that is a **hierarchical prior**, pooling a new (colour, size) arm towards its\n   colour\'s aggregate rather than starting it at Beta(1,1); that is the known\n   next refinement and is not built.\n\n3. **Evidence decays.** Responsiveness is *state-dependent*, measured on eight of\n   nineteen games for click liveness (D-021) and confirmed for playfield effect\n   on vc33 (D-030). A stationary estimate would learn "clicking does nothing"\n   at the level start and never revise it five actions later when everything\n   works. Counts are discounted every step so the policy tracks the game rather\n   than averaging over it.\n\n**Guardrail 1 is respected by construction.** Thompson sampling from a Beta\nposterior never assigns zero probability to an arm, so nothing is ever hard\nrejected — an arm that has failed twenty times is heavily down-weighted and\nstill occasionally retried, which is exactly what a state-gated game requires.\n\n**What this does not do.**\n\n- **No goal.** It maximises "something happened in the playfield", which is a\n  proxy for interestingness and *not* a proxy for winning. It is the exploration\n  half of D-027\'s target architecture; the half that acts on hypotheses towards\n  a goal is not built.\n- **No model.** It does not know *what* an action does, only whether it tends to\n  do something. It is deliberately dumber than the dynamics learner and exists to\n  feed it.\n- **No RESET policy.** Some games get *less* responsive with depth (cd82 and\n  ka59 fall from 100% to ~6% by twenty actions, D-021), so a policy that noticed\n  a dry spell and restarted the level would sometimes be right. `RESET` costs an\n  action and does not rewind the action count, so this needs the RHAE cost model\n  from D-022 and is left out of the first version deliberately.\n- **No credit assignment across steps.** An action that unlocks responsiveness\n  five moves later gets no credit for it, which is precisely what vc33 rewards.\n"""\n\nfrom __future__ import annotations\n\nimport random\nfrom dataclasses import dataclass, field\nfrom typing import Any, Optional\n\nimport numpy as np\n\nfrom .drive import foreground_object_targets, object_targets\nfrom .dynamics.rules import describe_object\nfrom .perception.segment import segment\nfrom .types import CLICK, GRID_H, GRID_W, Action, Observation\n\n#: Rows changing in at least this fraction of observed changes are bookkeeping.\nBAND_FRACTION = 0.5\n\n#: A band may not be thicker than this, or it is playfield (see\n#: `dynamics.predict.bookkeeping_mask`, which uses the same cap for the same\n#: reason: without it the estimator swallows the game).\nMAX_BAND_ROWS = 4\n\n#: Observations needed before the band estimate is used at all. Below this the\n#: policy counts every change as playfield, which over-credits early and\n#: corrects itself rather than starting with a wrong band.\nBAND_WARMUP = 8\n\n#: Per-step multiplier on accumulated evidence. Responsiveness is state\n#: dependent, so old observations must fade or the policy cannot follow a game\n#: that changes underneath it.\nDECAY = 0.97\n\n\nclass OnlineBand:\n    """Incremental estimate of which rows are bookkeeping rather than game.\n\n    The online counterpart of `dynamics.predict.bookkeeping_mask`, which needs a\n    whole corpus. Same rule, same thresholds, updated one transition at a time.\n    """\n\n    def __init__(self, height: int = GRID_H, width: int = GRID_W) -> None:\n        self._rows = np.zeros(height, dtype=np.float64)\n        self._n = 0.0\n        self._w = width\n        self._h = height\n\n    def update(self, before: np.ndarray, after: np.ndarray) -> None:\n        diff = before != after\n        if not diff.any():\n            return\n        self._n += 1.0\n        self._rows[np.unique(np.nonzero(diff)[0])] += 1.0\n\n    @property\n    def ready(self) -> bool:\n        return self._n >= BAND_WARMUP\n\n    def mask(self) -> np.ndarray:\n        m = np.zeros((self._h, self._w), dtype=bool)\n        if not self.ready:\n            return m\n        rows = np.nonzero(self._rows >= BAND_FRACTION * self._n)[0]\n        if len(rows) and len(rows) <= MAX_BAND_ROWS:\n            m[rows, :] = True\n        return m\n\n    def moved_playfield(self, before: np.ndarray, after: np.ndarray) -> bool:\n        """Did this transition change anything the band does not explain?"""\n        return bool(((before != after) & ~self.mask()).any())\n\n\n@dataclass\nclass _Arm:\n    """Beta posterior over "this choice moves the playfield"."""\n\n    a: float = 1.0\n    b: float = 1.0\n\n    def sample(self, rng: random.Random) -> float:\n        return rng.betavariate(self.a, self.b)\n\n    def update(self, success: bool) -> None:\n        if success:\n            self.a += 1.0\n        else:\n            self.b += 1.0\n\n    def decay(self, gamma: float) -> None:\n        # Decay towards the prior, never below it, so an arm always retains some\n        # optimism and Guardrail 1\'s "never hard reject" survives arbitrarily\n        # long runs of failure.\n        self.a = 1.0 + (self.a - 1.0) * gamma\n        self.b = 1.0 + (self.b - 1.0) * gamma\n\n    @property\n    def mean(self) -> float:\n        return self.a / (self.a + self.b)\n\n\n@dataclass\nclass ExploringDriver:\n    """Chooses actions by Thompson sampling on "does this move the playfield".\n\n    Learns from the observation stream alone: it remembers the frame and action\n    it last emitted and scores them when the next observation arrives, so it\n    satisfies the plain `Driver` protocol and needs no feedback channel.\n    """\n\n    seed: int = 0\n    carving: str = "colour_8"\n    decay: float = DECAY\n    #: Draw click candidates from all six carvings rather than `carving` alone.\n    #: Measured and rejected by default: see the class docstring.\n    union_targets: bool = False\n    #: How coarsely to key click arms. "colour_size" gives one arm per distinct\n    #: (colour, size); "colour" pools every object of a colour into one arm.\n    #: Coarser means fewer arms and faster learning at the cost of precision.\n    key_mode: str = "colour"\n    _rng: random.Random = field(init=False)\n    _arms: dict[Any, _Arm] = field(default_factory=dict, init=False)\n    band: OnlineBand = field(default_factory=OnlineBand, init=False)\n    _last_frame: Optional[np.ndarray] = field(default=None, init=False)\n    _last_key: Optional[Any] = field(default=None, init=False)\n    n_playfield: int = field(default=0, init=False)\n    n_steps: int = field(default=0, init=False)\n\n    def __post_init__(self) -> None:\n        self._rng = random.Random(self.seed)\n\n    @property\n    def name(self) -> str:\n        return "explore"\n\n    # -- learning ----------------------------------------------------------\n\n    def _arm(self, key: Any) -> _Arm:\n        return self._arms.setdefault(key, _Arm())\n\n    def _learn(self, frame: Optional[np.ndarray]) -> None:\n        """Score the previous action against what came back."""\n        if self._last_frame is None or self._last_key is None or frame is None:\n            return\n        self.band.update(self._last_frame, frame)\n        moved = self.band.moved_playfield(self._last_frame, frame)\n        self._arm(self._last_key).update(moved)\n        self.n_playfield += int(moved)\n        self.n_steps += 1\n        for arm in self._arms.values():\n            arm.decay(self.decay)\n\n    # -- acting ------------------------------------------------------------\n\n    def _click_key(self, attrs: dict) -> tuple:\n        # Colour and size, not the full signature: shape is too specific to\n        # generalise across frames, and the pair was enough to isolate the mover\n        # on ls20 (D-024\'s worked example). `key_mode` trades that precision for\n        # a smaller arm space when the budget is short.\n        if self.key_mode == "colour":\n            return ("click", attrs["colour"])\n        return ("click", attrs["colour"], attrs["size"])\n\n    def next_action(self, obs: Observation) -> Optional[Action]:\n        frame = obs.last\n        self._learn(frame)\n\n        acts = sorted(a for a in obs.available_actions if a != 0)\n        if not acts or obs.state.is_terminal:\n            self._last_frame, self._last_key = None, None\n            return None\n\n        candidates: list[tuple[float, Any, Action]] = []\n        for a in acts:\n            if a == CLICK:\n                continue\n            key = ("act", a)\n            candidates.append((self._arm(key).sample(self._rng), key, Action.simple(a)))\n\n        if CLICK in acts and frame is not None:\n            seen: set[tuple[int, int]] = set()\n            segs = segment(frame)\n            pool = (\n                segs.segmentations\n                if self.union_targets\n                else (segs[self.carving],)\n            )\n            for seg in pool:\n                bg = seg.background_colour\n                for comp in seg.components:\n                    if bg is not None and comp.dominant_colour == bg:\n                        continue\n                    x, y = comp.click_target\n                    if (x, y) in seen:\n                        continue\n                    seen.add((x, y))\n                    key = self._click_key(describe_object(comp, Action.click(x, y)))\n                    candidates.append(\n                        (self._arm(key).sample(self._rng), key, Action.click(x, y))\n                    )\n        elif CLICK in acts:\n            x, y = self._rng.randrange(GRID_W), self._rng.randrange(GRID_H)\n            candidates.append((self._arm(("click", -1, -1)).sample(self._rng),\n                               ("click", -1, -1), Action.click(x, y)))\n\n        if not candidates:\n            a = self._rng.choice(acts)\n            action = (\n                Action.click(self._rng.randrange(GRID_W), self._rng.randrange(GRID_H))\n                if a == CLICK\n                else Action.simple(a)\n            )\n            self._last_frame, self._last_key = (\n                None if frame is None else frame.copy(),\n                ("act", a),\n            )\n            return action\n\n        _, key, action = max(candidates, key=lambda c: c[0])\n        self._last_frame = None if frame is None else frame.copy()\n        self._last_key = key\n        return action\n\n    # -- inspection --------------------------------------------------------\n\n    @property\n    def playfield_rate(self) -> float:\n        return self.n_playfield / self.n_steps if self.n_steps else 0.0\n\n    def describe(self, limit: int = 8) -> str:\n        rows = sorted(self._arms.items(), key=lambda kv: -kv[1].mean)\n        out = [f"{len(self._arms)} arms, playfield rate {100 * self.playfield_rate:.0f}%"]\n        for k, v in rows[:limit]:\n            out.append(f"  {str(k):28s} mean={v.mean:.2f}  a={v.a:.1f} b={v.b:.1f}")\n        return "\\n".join(out)\n'), 'arc3.hypothesis': (False, '"""Goal hypotheses as code, held as a weighted population (D-035).\n\nA **goal hypothesis** is a Python function the LLM writes::\n\n    # GOAL: move the blue block onto the red square\n    def progress(scene):\n        ...\n        return 0.4   # in [0, 1]; 1.0 means "the goal is reached"\n\nIt is a claim about what winning looks like *and* a heuristic for how close a\nstate is to it, which is what the planner needs: the frontier is ranked by the\npopulation\'s weighted progress, so exploration heads towards states the live\nhypotheses think are nearer a win.\n\n**The population, and how evidence moves it** (Guardrail 1: soft, never\ndeleting, never trusting one observation):\n\n- *Win.* When a level is won, every hypothesis is scored on the state the\n  winning action was taken from. High progress there is confirmation\n  (weight x2 at progress 1, scaled down linearly); low progress is\n  disconfirmation (x0.5 at progress 0).\n- *False goal.* A visited state a hypothesis scores as reached (>= 0.99) from\n  which the level did *not* complete is disconfirmation, x0.7, once per\n  distinct state.\n- *Flat.* A hypothesis whose progress has been identical on every one of at\n  least 20 distinct states says nothing about where to go. It is marked\n  uninformative and excluded from ranking (not from the population) until its\n  output varies.\n- *Broken.* An exception or a non-numeric return scores 0 and costs x0.8, once\n  per distinct state, capped so a broken function sinks but is kept for the\n  record.\n\nWeights are floored at ``MIN_WEIGHT`` so nothing ever reaches zero.\n\n**The sandbox.** The code runs with a whitelist of builtins, no ``import``,\n``numpy`` and ``math`` pre-bound, and a line-count budget enforced with\n``sys.settrace``, so a runaway loop costs a bounded amount and raises. This is\nprotection against a small model\'s mistakes, not against an adversary: the\ncode comes from our own model, and the process has no network in any case.\n\n**What this does not do.**\n\n- **Goal hypotheses only.** Dynamics hypotheses from the LLM ("ACTION1 moves the\n  player left") are not proposed or tested yet; dynamics still comes from the\n  exact state graph (D-034).\n- **No hypothesis revision.** A hypothesis is never edited. The LLM may propose\n  new ones, which enter at the prior weight; old ones sink.\n- **No credit for partial progress.** Only wins and false goals move weights;\n  a hypothesis whose progress rises along the path that eventually wins gets\n  nothing extra for that.\n"""\n\nfrom __future__ import annotations\n\nimport math\nimport re\nimport sys\nfrom dataclasses import dataclass, field\nfrom typing import Any, Callable, Optional\n\nimport numpy as np\n\nfrom .scene import Scene\n\nPRIOR_WEIGHT = 1.0\nMIN_WEIGHT = 0.01\n\n#: Line events a single progress() call may execute before it is stopped.\nSTEP_BUDGET = 20_000\n\n#: Distinct states needed before a constant-output hypothesis counts as flat.\nFLAT_AFTER = 20\n\nREACHED = 0.99\n\n_SAFE_BUILTINS = {\n    name: getattr(__builtins__, name) if not isinstance(__builtins__, dict) else __builtins__[name]\n    for name in (\n        "abs", "all", "any", "bool", "dict", "enumerate", "filter", "float", "int",\n        "len", "list", "map", "max", "min", "range", "reversed", "round", "set",\n        "sorted", "str", "sum", "tuple", "zip", "isinstance", "ValueError",\n        "Exception", "ZeroDivisionError", "IndexError", "KeyError", "True", "False", "None",\n    )\n    if (name in __builtins__ if isinstance(__builtins__, dict) else hasattr(__builtins__, name))\n}\n\n_BLOCK = re.compile(r"```(?:python|py)?\\s*\\n(.*?)```", re.S)\n_GOAL = re.compile(r"#\\s*GOAL:\\s*(.+)")\n\n\nclass BudgetExceeded(RuntimeError):\n    pass\n\n\ndef _budgeted(fn: Callable[[Scene], Any], scene: Scene, budget: int) -> Any:\n    count = 0\n\n    # Only the hypothesis\'s own lines are counted. Tracing into numpy or the\n    # scene helpers charged the budget for trusted library code: one call to\n    # scene.colours() exceeded 20,000 steps inside np.unique.\n    def local(frame, event, arg):\n        nonlocal count\n        if event == "line":\n            count += 1\n            if count > budget:\n                raise BudgetExceeded(f"more than {budget} steps")\n        return local\n\n    def tracer(frame, event, arg):\n        return local if frame.f_code.co_filename == "<hypothesis>" else None\n\n    old = sys.gettrace()\n    sys.settrace(tracer)\n    try:\n        return fn(scene)\n    finally:\n        sys.settrace(old)\n\n\ndef compile_progress(source: str) -> Callable[[Scene], Any]:\n    """Compile LLM code and return its ``progress`` function, or raise."""\n    if re.search(r"^\\s*(import|from)\\s", source, re.M) or "__" in source:\n        raise ValueError("imports and dunder access are not allowed")\n    ns: dict[str, Any] = {"__builtins__": _SAFE_BUILTINS, "np": np, "math": math}\n    exec(compile(source, "<hypothesis>", "exec"), ns)\n    fn = ns.get("progress")\n    if not callable(fn):\n        # Small models rename it (measured: Qwen2.5-Coder-3B wrote\n        # `progress_same_colour`). Accept one progress-prefixed function.\n        named = [v for k, v in ns.items() if k.startswith("progress") and callable(v)]\n        if len(named) != 1:\n            raise ValueError("no progress(scene) function defined")\n        fn = named[0]\n    return fn\n\n\ndef parse_hypotheses(text: str) -> list[tuple[str, str]]:\n    """``(goal statement, source)`` per hypothesis in the fenced code blocks.\n\n    One block may hold several hypotheses (small models do this); it is split\n    at each ``# GOAL:`` line that is followed by a ``def progress...``.\n    """\n    out = []\n    for block in _BLOCK.findall(text) or [text]:\n        starts = [m.start() for m in _GOAL.finditer(block)]\n        chunks = ([block[a:b] for a, b in zip(starts, [*starts[1:], len(block)])]\n                  if starts else [block])\n        for src in chunks:\n            if not re.search(r"def\\s+progress\\w*\\s*\\(", src):\n                continue\n            m = _GOAL.search(src)\n            out.append(((m.group(1).strip() if m else "(no GOAL line)"), src.strip()))\n    return out\n\n\n@dataclass\nclass Hypothesis:\n    id: int\n    goal: str\n    source: str\n    fn: Optional[Callable[[Scene], Any]]\n    origin: str = ""                 # which call proposed it\n    weight: float = PRIOR_WEIGHT\n    compile_error: Optional[str] = None\n    n_errors: int = 0\n    stalls: int = 0\n    wins_confirmed: int = 0\n    wins_missed: int = 0\n    false_goals: int = 0\n    _seen: dict = field(default_factory=dict, repr=False)     # state key -> value\n    _penalised: set = field(default_factory=set, repr=False)\n\n    def value(self, key: bytes, scene_fn: Callable[[], Scene]) -> float:\n        """Progress on the state ``key`` (cached; scenes are built lazily)."""\n        if key in self._seen:\n            return self._seen[key]\n        v = 0.0\n        if self.fn is not None:\n            try:\n                r = _budgeted(self.fn, scene_fn(), STEP_BUDGET)\n                v = float(r)\n                if not math.isfinite(v):\n                    raise ValueError("non-finite progress")\n                v = min(1.0, max(0.0, v))\n            except Exception:\n                self.n_errors += 1\n                if key not in self._penalised and self.n_errors <= 5:\n                    self._penalised.add(key)\n                    self._scale(0.8)\n                v = 0.0\n        self._seen[key] = v\n        return v\n\n    @property\n    def informative(self) -> bool:\n        if self.fn is None:\n            return False\n        vals = set(self._seen.values())\n        return not (len(self._seen) >= FLAT_AFTER and len(vals) <= 1)\n\n    def _scale(self, f: float) -> None:\n        self.weight = max(MIN_WEIGHT, self.weight * f)\n\n    def describe(self) -> str:\n        flag = "" if self.informative else " [flat]"\n        return (f"H{self.id} w={self.weight:.2f}{flag} wins+{self.wins_confirmed}"\n                f"/-{self.wins_missed} false={self.false_goals} err={self.n_errors}: {self.goal}")\n\n\n@dataclass\nclass Population:\n    """All goal hypotheses for the current level, with their weights."""\n\n    hypotheses: list[Hypothesis] = field(default_factory=list)\n    _next: int = 0\n\n    def add_from_text(self, text: str, origin: str = "") -> list[Hypothesis]:\n        added = []\n        for goal, src in parse_hypotheses(text):\n            try:\n                fn, err = compile_progress(src), None\n            except Exception as exc:  # kept for the record, never ranks\n                fn, err = None, f"{type(exc).__name__}: {exc}"\n            h = Hypothesis(id=self._next, goal=goal, source=src, fn=fn,\n                           origin=origin, compile_error=err,\n                           weight=PRIOR_WEIGHT if fn else MIN_WEIGHT)\n            self._next += 1\n            self.hypotheses.append(h)\n            added.append(h)\n        return added\n\n    def live(self) -> list[Hypothesis]:\n        return [h for h in self.hypotheses if h.informative]\n\n    def value(self, key: bytes, scene_fn: Callable[[], Scene]) -> Optional[float]:\n        """Weighted mean progress over informative hypotheses; None if there are none."""\n        live = self.live()\n        if not live:\n            return None\n        tw = sum(h.weight for h in live)\n        return sum(h.weight * h.value(key, scene_fn) for h in live) / tw\n\n    def observe_state(self, key: bytes, scene_fn: Callable[[], Scene]) -> None:\n        """A state was visited and the level did not complete there."""\n        for h in self.hypotheses:\n            if h.fn is None:\n                continue\n            if h.value(key, scene_fn) >= REACHED and key not in h._penalised:\n                h._penalised.add(key)\n                h.false_goals += 1\n                h._scale(0.7)\n\n    def observe_win(self, key: bytes, scene_fn: Callable[[], Scene]) -> None:\n        """The level was won by an action taken from state ``key``."""\n        for h in self.hypotheses:\n            if h.fn is None:\n                continue\n            v = h.value(key, scene_fn)\n            h._scale(0.5 + 1.5 * v)   # x0.5 at v=0, x2 at v=1\n            if v >= 0.5:\n                h.wins_confirmed += 1\n            else:\n                h.wins_missed += 1\n\n    def describe(self, limit: int = 8) -> str:\n        hs = sorted(self.hypotheses, key=lambda h: -h.weight)[:limit]\n        return "\\n".join(h.describe() for h in hs) or "(no hypotheses)"\n'), 'arc3.llm': (True, '"""The LLM seam (D-035): one narrow interface, several interchangeable backends.\n\nThe LLM\'s only job in this system is to *propose* hypotheses as code. It never\nchooses an action and never sees the engine. Everything it says is tested by\nthe rest of the machinery before it can influence anything.\n\nBackends, all behind :class:`LLM`:\n\n- :class:`ScriptedLLM` — canned responses, for tests. No model, no network.\n- :class:`OpenAIChatLLM` — any OpenAI-compatible chat endpoint. In development\n  that is llama.cpp\'s ``llama-server`` on localhost; on Kaggle it is the same\n  server (or vLLM) running offline on the notebook\'s GPU. Standard library only.\n- :class:`RecordingLLM` / :class:`ReplayLLM` — write every call to JSONL, and\n  serve a recorded run back offline. One GPU run\'s thinking can then be reused\n  to tune everything downstream at zero GPU cost.\n\n**Guardrail 5.** Nothing here may reach the internet at evaluation. The HTTP\nclient refuses any host that is not local unless explicitly allowed.\n"""\n\nfrom .client import (\n    LLM,\n    Message,\n    OpenAIChatLLM,\n    RecordingLLM,\n    ReplayLLM,\n    ReplayMiss,\n    ScriptedLLM,\n    call_key,\n)\n\n__all__ = [\n    "LLM",\n    "Message",\n    "OpenAIChatLLM",\n    "RecordingLLM",\n    "ReplayLLM",\n    "ReplayMiss",\n    "ScriptedLLM",\n    "call_key",\n]\n'), 'arc3.llm.client': (False, '"""LLM backends. See the package docstring for what each is for."""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport time\nimport urllib.request\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Callable, Optional, Protocol, Sequence, Union\nfrom urllib.parse import urlparse\n\n#: A chat message: {"role": "system" | "user" | "assistant", "content": str}.\nMessage = dict\n\n#: Hosts an HTTP backend may talk to without ``allow_remote``. Kaggle\'s\n#: evaluation has no internet (Guardrail 5); a remote host here would work in\n#: development and fail silently in the only run that counts.\nLOCAL_HOSTS = ("localhost", "127.0.0.1", "::1")\n\n\nclass LLM(Protocol):\n    """Chat completion. Deterministic given ``seed`` where the backend allows."""\n\n    @property\n    def name(self) -> str: ...\n\n    def complete(\n        self,\n        messages: Sequence[Message],\n        *,\n        max_tokens: int = 512,\n        temperature: float = 0.0,\n        seed: int = 0,\n        schema: Optional[dict] = None,\n    ) -> str: ...\n\n\ndef call_key(messages: Sequence[Message], max_tokens: int, temperature: float, seed: int) -> str:\n    """Stable identity of one call, for recording and replay."""\n    blob = json.dumps(\n        {"m": list(messages), "t": max_tokens, "temp": temperature, "seed": seed},\n        sort_keys=True,\n        ensure_ascii=False,\n    )\n    return hashlib.sha256(blob.encode("utf-8")).hexdigest()[:24]\n\n\n# -- test backend -------------------------------------------------------------\n\n\n@dataclass\nclass ScriptedLLM:\n    """Returns canned responses: a list consumed in order, or a function.\n\n    A function receives the messages and returns the response, so a test can\n    answer according to what it was asked.\n    """\n\n    responses: Union[Sequence[str], Callable[[Sequence[Message]], str]] = ()\n    calls: list[list[Message]] = field(default_factory=list)\n    _i: int = 0\n\n    @property\n    def name(self) -> str:\n        return "scripted"\n\n    def complete(self, messages, *, max_tokens=512, temperature=0.0, seed=0, schema=None) -> str:\n        self.calls.append(list(messages))\n        if callable(self.responses):\n            return self.responses(messages)\n        if self._i >= len(self.responses):\n            return ""\n        r = self.responses[self._i]\n        self._i += 1\n        return r\n\n\n# -- real backend ---------------------------------------------------------------\n\n\n@dataclass\nclass OpenAIChatLLM:\n    """An OpenAI-compatible ``/v1/chat/completions`` endpoint.\n\n    Written against llama.cpp\'s ``llama-server``; vLLM exposes the same route.\n    """\n\n    base_url: str = "http://127.0.0.1:8080"\n    model: str = "local"\n    timeout: float = 600.0\n    allow_remote: bool = False\n    #: Extra fields merged into every request body, for server-specific options.\n    #: Qwen3.6 thinks by default; ``{"chat_template_kwargs": {"enable_thinking":\n    #: false}}`` turns that off in vLLM, and the model card\'s non-thinking\n    #: sampling (top_p, top_k, presence_penalty) goes here too.\n    extra: dict = field(default_factory=dict)\n\n    def __post_init__(self) -> None:\n        host = urlparse(self.base_url).hostname or ""\n        if not self.allow_remote and host not in LOCAL_HOSTS:\n            raise ValueError(\n                f"{self.base_url} is not local; evaluation has no internet "\n                "(Guardrail 5). Pass allow_remote=True only for prototyping."\n            )\n\n    @property\n    def name(self) -> str:\n        return f"openai-chat:{self.model}@{self.base_url}"\n\n    def complete(self, messages, *, max_tokens=512, temperature=0.0, seed=0,\n                 schema: Optional[dict] = None) -> str:\n        req_body = {\n            "model": self.model,\n            "messages": list(messages),\n            "max_tokens": max_tokens,\n            "temperature": temperature,\n            "seed": seed,\n        }\n        req_body.update(self.extra)\n        if schema is not None:\n            # Constrained decoding: the server only emits JSON matching the\n            # schema (vLLM via xgrammar; llama.cpp via its grammar support).\n            req_body["response_format"] = {\n                "type": "json_schema",\n                "json_schema": {"name": schema.get("title", "output"), "schema": schema},\n            }\n        body = json.dumps(req_body).encode("utf-8")\n        req = urllib.request.Request(\n            self.base_url.rstrip("/") + "/v1/chat/completions",\n            data=body,\n            headers={"Content-Type": "application/json"},\n        )\n        with urllib.request.urlopen(req, timeout=self.timeout) as resp:\n            out = json.loads(resp.read().decode("utf-8"))\n        return out["choices"][0]["message"]["content"] or ""\n\n    def ready(self) -> bool:\n        """Is the server up? llama-server answers ``/health`` once loaded."""\n        try:\n            with urllib.request.urlopen(self.base_url.rstrip("/") + "/health", timeout=5) as r:\n                return r.status == 200\n        except Exception:\n            return False\n\n\n# -- recording and replay ---------------------------------------------------------\n\n\n@dataclass\nclass RecordingLLM:\n    """Wraps a backend and appends every call to a JSONL file.\n\n    Each line: key, backend name, the full request, the response, wall seconds.\n    Written as it goes, so a run killed by a Kaggle timeout keeps what it did.\n    """\n\n    inner: LLM\n    path: Path\n    n_calls: int = 0\n    seconds: float = 0.0\n\n    def __post_init__(self) -> None:\n        self.path = Path(self.path)\n        self.path.parent.mkdir(parents=True, exist_ok=True)\n\n    @property\n    def name(self) -> str:\n        return self.inner.name\n\n    def complete(self, messages, *, max_tokens=512, temperature=0.0, seed=0, schema=None) -> str:\n        t0 = time.time()\n        out = self.inner.complete(messages, max_tokens=max_tokens,\n                                  temperature=temperature, seed=seed, schema=schema)\n        dt = time.time() - t0\n        self.n_calls += 1\n        self.seconds += dt\n        rec = {\n            "key": call_key(messages, max_tokens, temperature, seed),\n            "backend": self.inner.name,\n            "messages": list(messages),\n            "max_tokens": max_tokens,\n            "temperature": temperature,\n            "seed": seed,\n            "response": out,\n            "seconds": round(dt, 3),\n        }\n        with self.path.open("a", encoding="utf-8") as f:\n            f.write(json.dumps(rec, ensure_ascii=False) + "\\n")\n        return out\n\n\nclass ReplayMiss(KeyError):\n    """A call that is not in the recording, with no fallback backend."""\n\n\n@dataclass\nclass ReplayLLM:\n    """Serves recorded responses by call key; optionally falls back to a backend.\n\n    Replay is exact only while the prompts are byte-identical, so any change to\n    scene rendering or prompt text turns hits into misses. ``hits``/``misses``\n    make that visible rather than silent.\n    """\n\n    path: Path\n    fallback: Optional[LLM] = None\n    hits: int = 0\n    misses: int = 0\n    _table: dict[str, str] = field(default_factory=dict)\n\n    def __post_init__(self) -> None:\n        self.path = Path(self.path)\n        if self.path.exists():\n            for line in self.path.read_text(encoding="utf-8").splitlines():\n                if line.strip():\n                    rec = json.loads(line)\n                    self._table[rec["key"]] = rec["response"]\n\n    @property\n    def name(self) -> str:\n        return f"replay:{self.path.name}"\n\n    def complete(self, messages, *, max_tokens=512, temperature=0.0, seed=0, schema=None) -> str:\n        key = call_key(messages, max_tokens, temperature, seed)\n        if key in self._table:\n            self.hits += 1\n            return self._table[key]\n        self.misses += 1\n        if self.fallback is None:\n            raise ReplayMiss(key)\n        return self.fallback.complete(messages, max_tokens=max_tokens,\n                                      temperature=temperature, seed=seed, schema=schema)\n'), 'arc3.perception': (True, '"""Exact symbolic perception (D-006).\n\nPipeline order matters: geometry first, because segmentation and tracking are\nboth wrong at the wrong granularity (D-011).\n\n    geometry  -> what cell size does this game\'s content move on?\n    segment   -> several candidate carvings into objects\n    track     -> which object in frame t is which in frame t+1\n    diff      -> what changed, structurally, between two frames\n\nNothing here imports the engine; everything runs from recorded episodes.\n"""\n\nfrom .components import CONN4, CONN8, label\nfrom .diff import FrameDiff, SequenceDiff, diff, diff_sequence\nfrom .geometry import BlockGrid, Geometry, Rect, induce_geometry\nfrom .segment import Component, Segmentation, SegmentationSet, segment\nfrom .track import FrameMatching, Match, Track, Tracker, match_frames\n\n__all__ = [\n    "CONN4",\n    "CONN8",\n    "BlockGrid",\n    "Component",\n    "FrameDiff",\n    "FrameMatching",\n    "Geometry",\n    "Match",\n    "Rect",\n    "Segmentation",\n    "SegmentationSet",\n    "SequenceDiff",\n    "Track",\n    "Tracker",\n    "diff",\n    "diff_sequence",\n    "induce_geometry",\n    "label",\n    "match_frames",\n    "segment",\n]\n'), 'arc3.perception.components': (False, '"""Connected-component labelling over 64x64 grids.\n\nKept separate from :mod:`arc3.perception.segment` because\n:mod:`arc3.perception.geometry` needs it too, and geometry must not depend on\nsegmentation (segmentation depends on geometry).\n\nPure numpy, no scipy: the grids are 4096 cells, a vectorised neighbour-edge list\nresolved by union-find is exact and runs in about 4 ms, and it keeps the offline\nKaggle dependency set to what ``arc-agi`` already pulls in.\n"""\n\nfrom __future__ import annotations\n\nfrom typing import Optional\n\nimport numpy as np\n\n#: 4-connectivity: orthogonal neighbours only.\nCONN4 = 4\n#: 8-connectivity: orthogonal plus diagonal.\nCONN8 = 8\n\n\nclass _UnionFind:\n    __slots__ = ("parent",)\n\n    def __init__(self, n: int) -> None:\n        self.parent = list(range(n))\n\n    def find(self, a: int) -> int:\n        p = self.parent\n        root = a\n        while p[root] != root:\n            root = p[root]\n        while p[a] != root:  # path compression\n            p[a], a = root, p[a]\n        return root\n\n    def union(self, a: int, b: int) -> None:\n        ra, rb = self.find(a), self.find(b)\n        if ra != rb:\n            self.parent[max(ra, rb)] = min(ra, rb)\n\n    def new(self) -> int:\n        self.parent.append(len(self.parent))\n        return len(self.parent) - 1\n\n\ndef label(\n    values: np.ndarray,\n    *,\n    connectivity: int = CONN4,\n    mask: Optional[np.ndarray] = None,\n    equal_values: bool = True,\n) -> tuple[np.ndarray, int]:\n    """Label connected regions of ``values``.\n\n    Args:\n        values: 2D array. With ``equal_values`` (the default), neighbours join\n            only when they hold the same value - components of one colour. With\n            ``equal_values=False``, ``values`` is treated as a boolean mask and\n            every truthy neighbour joins regardless of value, which gives\n            multi-colour components.\n        connectivity: :data:`CONN4` or :data:`CONN8`.\n        mask: optional boolean array; ``False`` cells never belong to any\n            component.\n\n    Returns:\n        ``(labels, n)`` where ``labels`` is ``int32`` with ``-1`` for excluded\n        cells and ``0..n-1`` for components, numbered in raster order of their\n        first pixel so the labelling is deterministic (D-006).\n    """\n    if connectivity not in (CONN4, CONN8):\n        raise ValueError(f"connectivity must be 4 or 8, got {connectivity}")\n    arr = np.asarray(values)\n    h, w = arr.shape\n    active = np.ones((h, w), dtype=bool) if mask is None else np.asarray(mask, dtype=bool)\n    if not equal_values:\n        active = active & (arr != 0)\n\n    # Build the edge list vectorised, then union-find over it. Much faster than\n    # a per-pixel scan and exactly equivalent: two active neighbours are joined\n    # iff they would have been merged by a raster scan with label resolution.\n    def _edges(da: int, dx: int) -> tuple[np.ndarray, np.ndarray]:\n        """Flat index pairs for neighbours offset by ``(da, dx)`` rows/cols."""\n        ys0, ys1 = max(0, -da), h - max(0, da)\n        xs0, xs1 = max(0, -dx), w - max(0, dx)\n        if ys1 <= ys0 or xs1 <= xs0:\n            empty = np.zeros(0, dtype=np.int64)\n            return empty, empty\n        a_sl = (slice(ys0, ys1), slice(xs0, xs1))\n        b_sl = (slice(ys0 + da, ys1 + da), slice(xs0 + dx, xs1 + dx))\n        ok = active[a_sl] & active[b_sl]\n        if equal_values:\n            ok &= arr[a_sl] == arr[b_sl]\n        if not ok.any():\n            empty = np.zeros(0, dtype=np.int64)\n            return empty, empty\n        flat = np.arange(h * w, dtype=np.int64).reshape(h, w)\n        return flat[a_sl][ok], flat[b_sl][ok]\n\n    offsets = [(0, 1), (1, 0)]\n    if connectivity == CONN8:\n        offsets += [(1, 1), (1, -1)]\n\n    uf = _UnionFind(h * w)\n    for da, dx in offsets:\n        ia, ib = _edges(da, dx)\n        for a_i, b_i in zip(ia.tolist(), ib.tolist(), strict=True):\n            uf.union(a_i, b_i)\n\n    out = np.full((h, w), -1, dtype=np.int32)\n    remap: dict[int, int] = {}\n    n = 0\n    flat_active = active.ravel()\n    flat_out = out.ravel()\n    for idx in np.flatnonzero(flat_active).tolist():\n        root = uf.find(idx)\n        label_id = remap.get(root)\n        if label_id is None:\n            label_id = n\n            remap[root] = label_id\n            n += 1\n        flat_out[idx] = label_id\n    return out, n\n\n\ndef component_masks(labels: np.ndarray, n: int) -> list[np.ndarray]:\n    """One boolean mask per component index."""\n    return [labels == i for i in range(n)]\n\n\ndef largest_component_mask(mask: np.ndarray, *, connectivity: int = CONN8) -> np.ndarray:\n    """The biggest connected region of a boolean mask, as a boolean mask."""\n    m = np.asarray(mask, dtype=bool)\n    if not m.any():\n        return m\n    labels, n = label(m.astype(np.int8), connectivity=connectivity, equal_values=False)\n    if n == 0:\n        return m\n    sizes = np.bincount(labels[labels >= 0].ravel(), minlength=n)\n    return labels == int(np.argmax(sizes))\n\n\ndef component_sizes(labels: np.ndarray, n: int) -> np.ndarray:\n    if n == 0:\n        return np.zeros(0, dtype=np.int64)\n    return np.bincount(labels[labels >= 0].ravel(), minlength=n)\n'), 'arc3.perception.diff': (False, '"""Exact structural diff between frames.\n\nNot "the grid changed" but which cells, which regions, and how (D-007). This\nmodule deliberately stops short of object-level attribution - saying *which\nobject* moved is :mod:`arc3.perception.track`\'s job, and needs a segmentation to\neven be well posed. What lives here is everything derivable from a pair of grids\nalone, exactly and cheaply.\n\nThe most useful thing here is :func:`diff_sequence`. A single action can return\nup to 42 frames, and the intermediate frames of an animation show the trajectory\nof motion rather than just its endpoints - by some distance the richest free\nevidence the environment gives us, and something design-thesis section 3\nundercounted (D-009). :class:`SequenceDiff` summarises a whole animation,\nincluding whether it is a steady drift in one direction, which is the shape a\nsprite translation takes.\n"""\n\nfrom __future__ import annotations\n\nfrom collections import Counter\nfrom dataclasses import dataclass, field\nfrom typing import Iterable, Optional, Sequence\n\nimport numpy as np\n\nfrom .components import CONN8, label\nfrom .geometry import Rect, bounding_rect\n\n\n@dataclass(frozen=True, slots=True)\nclass ChangeRegion:\n    """One connected region of changed cells."""\n\n    mask: np.ndarray\n    bbox: Rect\n    size: int\n    colours_before: tuple[int, ...]\n    colours_after: tuple[int, ...]\n\n    @property\n    def is_recolour(self) -> bool:\n        """The region changed colour without changing shape context.\n\n        True when exactly one colour left and one arrived - the signature of a\n        toggle or a state flip rather than movement.\n        """\n        return len(self.colours_before) == 1 and len(self.colours_after) == 1\n\n\n@dataclass(frozen=True, slots=True)\nclass FrameDiff:\n    """What changed between two grids."""\n\n    before: np.ndarray\n    after: np.ndarray\n    changed: np.ndarray\n    transitions: Counter[tuple[int, int]]\n    regions: tuple[ChangeRegion, ...]\n\n    @property\n    def n_changed(self) -> int:\n        return int(self.changed.sum())\n\n    @property\n    def is_empty(self) -> bool:\n        return self.n_changed == 0\n\n    @property\n    def bbox(self) -> Rect:\n        return bounding_rect(self.changed)\n\n    @property\n    def colour_delta(self) -> dict[int, int]:\n        """Net change in the number of cells of each colour."""\n        delta: Counter[int] = Counter()\n        for (old, new), n in self.transitions.items():\n            delta[old] -= n\n            delta[new] += n\n        return {c: d for c, d in delta.items() if d != 0}\n\n    def translation(self) -> Optional[tuple[int, int]]:\n        """``(dx, dy)`` if the whole frame is a rigid shift of ``before``.\n\n        Checks only shifts small enough to leave overlap worth testing. Returns\n        ``None`` for anything that is not an exact whole-frame translation - a\n        scrolling camera is the case this catches, and it matters because it\n        makes every object appear to move at once.\n        """\n        if self.is_empty:\n            return (0, 0)\n        h, w = self.before.shape\n        for dy in range(-h + 1, h):\n            for dx in range(-w + 1, w):\n                if dx == 0 and dy == 0:\n                    continue\n                ys0, ys1 = max(0, dy), min(h, h + dy)\n                xs0, xs1 = max(0, dx), min(w, w + dx)\n                if ys1 - ys0 < h // 2 or xs1 - xs0 < w // 2:\n                    continue\n                src = self.before[ys0 - dy : ys1 - dy, xs0 - dx : xs1 - dx]\n                dst = self.after[ys0:ys1, xs0:xs1]\n                if np.array_equal(src, dst):\n                    return (dx, dy)\n        return None\n\n\ndef diff(before: np.ndarray, after: np.ndarray, *, connectivity: int = CONN8) -> FrameDiff:\n    """Exact structural diff of two grids."""\n    b = np.asarray(before)\n    a = np.asarray(after)\n    if b.shape != a.shape:\n        raise ValueError(f"shape mismatch: {b.shape} vs {a.shape}")\n    changed = b != a\n\n    transitions: Counter[tuple[int, int]] = Counter()\n    if changed.any():\n        pairs = np.stack([b[changed], a[changed]], axis=1)\n        for old, new in pairs:\n            transitions[(int(old), int(new))] += 1\n\n    regions: list[ChangeRegion] = []\n    if changed.any():\n        labels, n = label(\n            changed.astype(np.int8), connectivity=connectivity, equal_values=False\n        )\n        for i in range(n):\n            m = labels == i\n            regions.append(\n                ChangeRegion(\n                    mask=m,\n                    bbox=bounding_rect(m),\n                    size=int(m.sum()),\n                    colours_before=tuple(sorted(int(v) for v in np.unique(b[m]))),\n                    colours_after=tuple(sorted(int(v) for v in np.unique(a[m]))),\n                )\n            )\n        regions.sort(key=lambda r: (-r.size, r.bbox.y0, r.bbox.x0))\n\n    return FrameDiff(\n        before=b, after=a, changed=changed, transitions=transitions, regions=tuple(regions)\n    )\n\n\n@dataclass(frozen=True, slots=True)\nclass SequenceDiff:\n    """The whole animation one action produced.\n\n    ``steps`` are the consecutive diffs. ``net`` is the endpoint-to-endpoint\n    diff, which is what an agent that only looked at the final frame would see -\n    keeping both makes it obvious how much information the intermediate frames\n    are carrying.\n    """\n\n    frames: tuple[np.ndarray, ...]\n    steps: tuple[FrameDiff, ...]\n    net: Optional[FrameDiff]\n    ever_changed: np.ndarray\n    per_step_translation: tuple[Optional[tuple[int, int]], ...] = field(default=())\n\n    @property\n    def n_frames(self) -> int:\n        return len(self.frames)\n\n    @property\n    def is_static(self) -> bool:\n        return not bool(self.ever_changed.any())\n\n    @property\n    def is_animated(self) -> bool:\n        """More than one frame, and something moved between them."""\n        return len(self.steps) > 0 and any(not s.is_empty for s in self.steps)\n\n    @property\n    def transient_mask(self) -> np.ndarray:\n        """Cells that changed during the animation but ended where they began.\n\n        These are invisible to an endpoint-only view and are exactly what a\n        planner would miss - a sprite that passed through a cell, a flash, a\n        counter that ticked and reset.\n        """\n        if self.net is None:\n            return np.zeros_like(self.ever_changed)\n        return self.ever_changed & ~self.net.changed\n\n    def steady_drift(self) -> Optional[tuple[int, int]]:\n        """``(dx, dy)`` if every step is the same non-zero whole-frame shift."""\n        shifts = [t for t in self.per_step_translation]\n        if not shifts or any(s is None or s == (0, 0) for s in shifts):\n            return None\n        first = shifts[0]\n        return first if all(s == first for s in shifts) else None\n\n\ndef diff_sequence(\n    frames: Sequence[np.ndarray] | Iterable[np.ndarray],\n    *,\n    previous: Optional[np.ndarray] = None,\n    connectivity: int = CONN8,\n    detect_translation: bool = False,\n) -> SequenceDiff:\n    """Diff every consecutive pair in an action\'s frame sequence.\n\n    Args:\n        frames: the frames the action returned. May be empty, in which case the\n            action was a no-op and the result is static.\n        previous: the resting grid before the action. Supply it - without it the\n            first frame of the sequence has nothing to be compared against, and\n            the first and often largest change is lost.\n        detect_translation: also test each step for a whole-frame shift. Off by\n            default because it is O(h*w) shifts per step and only pays off on\n            games with a scrolling camera.\n    """\n    seq = [np.asarray(f) for f in frames]\n    chain = ([np.asarray(previous)] if previous is not None else []) + seq\n    if not chain:\n        raise ValueError("diff_sequence needs at least one frame or a previous grid")\n\n    steps = tuple(diff(chain[i], chain[i + 1], connectivity=connectivity) for i in range(len(chain) - 1))\n    net = diff(chain[0], chain[-1], connectivity=connectivity) if len(chain) > 1 else None\n\n    stack = np.stack(chain)\n    ever = (stack != stack[0]).any(axis=0)\n\n    shifts: tuple[Optional[tuple[int, int]], ...] = ()\n    if detect_translation:\n        shifts = tuple(s.translation() for s in steps)\n\n    return SequenceDiff(\n        frames=tuple(seq),\n        steps=steps,\n        net=net,\n        ever_changed=ever,\n        per_step_translation=shifts,\n    )\n'), 'arc3.perception.geometry': (False, '"""Frame geometry: recovering the cell structure underneath the pixels.\n\nWhy this exists (D-011): the 64x64 frame is frequently an integer upscale of a\nmuch smaller camera. On bp35 and lf52 the camera is 8x8, so a naive connected\ncomponent is 64x its true area and most "cells" are rendering artefacts.\n\nWhat first contact with the frames taught us, and which D-011 did not\nanticipate: **there is no single scale**. ARCEngine composites a scaled camera\nview with a UI layer drawn last at full 64x64 resolution, and on bp35, lf52 and\nm0r0 the UI *is* most of what you see - the camera underneath is nearly empty.\nAsking for "the upscale factor" produces a confident wrong answer exactly where\nit would do most damage.\n\nSo this module answers a different and more useful question: **at what\ngranularity does this game\'s content actually move?** That is what segmentation\nand click enumeration need, and unlike the camera\'s internal scale it is always\nwell defined.\n\nThe method is propose-then-verify, and neither half has a tuned threshold:\n\n1. **Propose** the cell size as the GCD of every interior colour run inside the\n   region that changes across frames. A genuine ``s``-times upscale makes every\n   run length in that region a multiple of ``s``.\n2. **Verify** that every cell the proposal implies holds exactly one colour in\n   *every* observed frame. If not, fall back through the proposal\'s divisors\n   until it does. A cell size of 1 always verifies, so the pipeline is total and\n   its failure mode is "no structure claimed", never "wrong structure claimed".\n\nAn MDL-over-largest-rectangle formulation was tried first and rejected; see\nD-013 for the measurement and why it fails.\n\nEverything here is exact and deterministic (D-006). Nothing is fitted.\n"""\n\nfrom __future__ import annotations\n\nfrom collections import Counter\nfrom dataclasses import dataclass\nfrom itertools import pairwise\nfrom math import gcd\nfrom typing import Iterable, Optional, Sequence\n\nimport numpy as np\n\nfrom ..types import GRID_H, GRID_W\n\n\n@dataclass(frozen=True, slots=True)\nclass Rect:\n    """Half-open pixel rectangle ``[x0, x1) x [y0, y1)``."""\n\n    x0: int\n    y0: int\n    x1: int\n    y1: int\n\n    @property\n    def width(self) -> int:\n        return max(0, self.x1 - self.x0)\n\n    @property\n    def height(self) -> int:\n        return max(0, self.y1 - self.y0)\n\n    @property\n    def area(self) -> int:\n        return self.width * self.height\n\n    @property\n    def is_empty(self) -> bool:\n        return self.area == 0\n\n    def mask(self) -> np.ndarray:\n        m = np.zeros((GRID_H, GRID_W), dtype=bool)\n        m[self.y0 : self.y1, self.x0 : self.x1] = True\n        return m\n\n    def contains(self, x: int, y: int) -> bool:\n        return self.x0 <= x < self.x1 and self.y0 <= y < self.y1\n\n    def snapped(self, cell: int, origin_x: int, origin_y: int) -> "Rect":\n        """Grown outward to the nearest cell boundaries of the given grid."""\n        if cell <= 1:\n            return self\n\n        def down(v: int, o: int) -> int:\n            return o + ((v - o) // cell) * cell\n\n        def up(v: int, o: int) -> int:\n            return o + -((o - v) // cell) * cell\n\n        return Rect(\n            max(0, down(self.x0, origin_x)),\n            max(0, down(self.y0, origin_y)),\n            min(GRID_W, up(self.x1, origin_x)),\n            min(GRID_H, up(self.y1, origin_y)),\n        )\n\n    def __str__(self) -> str:\n        return f"({self.x0},{self.y0})-({self.x1},{self.y1})"\n\n\nFULL_FRAME = Rect(0, 0, GRID_W, GRID_H)\n\n\n@dataclass(frozen=True, slots=True)\nclass BlockGrid:\n    """A verified cell size and origin covering ``rect``.\n\n    ``cell == 1`` is the honest null result: no block structure was found, and\n    callers must work in pixel space. Check :attr:`is_structured` before using\n    cell coordinates for anything.\n    """\n\n    cell: int\n    origin_x: int\n    origin_y: int\n    rect: Rect\n    proposed_cell: int = 1\n\n    @property\n    def is_structured(self) -> bool:\n        return self.cell > 1\n\n    @property\n    def was_reduced(self) -> bool:\n        """The GCD proposal failed verification and had to be coarsened down."""\n        return self.proposed_cell != self.cell\n\n    @property\n    def cols(self) -> int:\n        return self.rect.width // self.cell\n\n    @property\n    def rows(self) -> int:\n        return self.rect.height // self.cell\n\n    def cell_of(self, x: int, y: int) -> Optional[tuple[int, int]]:\n        """Display pixel -> cell coordinate, or None if outside ``rect``."""\n        if not self.rect.contains(x, y):\n            return None\n        return ((x - self.rect.x0) // self.cell, (y - self.rect.y0) // self.cell)\n\n    def centre_of(self, col: int, row: int) -> tuple[int, int]:\n        """Cell coordinate -> the display pixel to click for that cell."""\n        return (\n            self.rect.x0 + col * self.cell + self.cell // 2,\n            self.rect.y0 + row * self.cell + self.cell // 2,\n        )\n\n    def downsample(self, grid: np.ndarray) -> np.ndarray:\n        """``rect`` as a ``(rows, cols)`` grid of cell colours."""\n        sub = grid[\n            self.rect.y0 : self.rect.y0 + self.rows * self.cell,\n            self.rect.x0 : self.rect.x0 + self.cols * self.cell,\n        ]\n        return np.ascontiguousarray(sub[:: self.cell, :: self.cell])\n\n    def tile_frame(self, grid: np.ndarray) -> tuple[np.ndarray, int, int]:\n        """The **whole frame** as cells on this grid\'s lattice.\n\n        Returns ``(cells, ox, oy)`` where ``ox, oy`` is the display-space origin\n        of cell ``(0, 0)``.\n\n        Unlike :meth:`downsample` this is not restricted to :attr:`rect`, because\n        static scenery is made of objects too and segmenting only the changing\n        region would lose every wall in the game. The cost is that cells outside\n        the verified region are represented by their top-left pixel and may not\n        be uniform - so downstream code gets a defined answer everywhere and an\n        exact one only inside :attr:`rect`.\n        """\n        ox, oy = self.origin_x, self.origin_y\n        rows = (GRID_H - oy) // self.cell\n        cols = (GRID_W - ox) // self.cell\n        sub = grid[oy : oy + rows * self.cell, ox : ox + cols * self.cell]\n        return np.ascontiguousarray(sub[:: self.cell, :: self.cell]), ox, oy\n\n    def __str__(self) -> str:\n        if not self.is_structured:\n            return "BlockGrid(unstructured)"\n        note = f", proposed {self.proposed_cell}" if self.was_reduced else ""\n        return (\n            f"BlockGrid(cell={self.cell}, origin=({self.origin_x},{self.origin_y}), "\n            f"rect={self.rect}, {self.rows}x{self.cols} cells{note})"\n        )\n\n\nUNSTRUCTURED = BlockGrid(cell=1, origin_x=0, origin_y=0, rect=FULL_FRAME, proposed_cell=1)\n\n\ndef _interior_runs(row: np.ndarray) -> list[tuple[int, int]]:\n    """``(start, length)`` for each run of equal values, edge runs excluded.\n\n    Runs touching either end are dropped: the frame truncates them, so their\n    length is a lower bound rather than a measurement, and feeding a truncated\n    length into a GCD would wrongly collapse it toward 1.\n    """\n    if row.size < 3:\n        return []\n    changes = np.flatnonzero(row[1:] != row[:-1]) + 1\n    if changes.size == 0:\n        return []\n    edges = np.concatenate(([0], changes, [row.size]))\n    return [\n        (int(a), int(b - a))\n        for a, b in pairwise(edges.tolist())\n        if a != 0 and b != row.size\n    ]\n\n\ndef propose_cell(\n    grids: Sequence[np.ndarray], mask: Optional[np.ndarray] = None\n) -> tuple[int, int, int]:\n    """Propose ``(cell, origin_x, origin_y)`` from run structure inside ``mask``.\n\n    The cell size is the GCD of every interior run length; the origins are the\n    most common residue of the run boundaries. This is a *proposal* - it has not\n    been verified that the implied cells are single-coloured. Use\n    :func:`verify_cell` or, better, :func:`induce_geometry`.\n    """\n    g = 0\n    x_starts: Counter[int] = Counter()\n    y_starts: Counter[int] = Counter()\n\n    for grid in grids:\n        for transposed in (False, True):\n            arr = grid.T if transposed else grid\n            m = None if mask is None else (mask.T if transposed else mask)\n            for i, row in enumerate(arr):\n                if m is not None:\n                    valid = m[i]\n                    if not valid.any():\n                        continue\n                for start, length in _interior_runs(row):\n                    if m is not None and not valid[start : start + length].all():\n                        continue\n                    g = gcd(g, length)\n                    (y_starts if transposed else x_starts)[start] = (\n                        y_starts if transposed else x_starts\n                    )[start] + 1\n\n    cell = g or 1\n    if cell <= 1:\n        return 1, 0, 0\n    ox = _dominant_residue(x_starts, cell)\n    oy = _dominant_residue(y_starts, cell)\n    return cell, ox, oy\n\n\ndef _dominant_residue(starts: Counter[int], cell: int) -> int:\n    if not starts:\n        return 0\n    residues: Counter[int] = Counter()\n    for pos, n in starts.items():\n        residues[pos % cell] += n\n    return residues.most_common(1)[0][0]\n\n\ndef _block_view(arr: np.ndarray, cell: int, ox: int, oy: int) -> Optional[np.ndarray]:\n    """``arr`` reshaped to ``(rows, cell, cols, cell)`` anchored at ``(ox, oy)``."""\n    rows = (GRID_H - oy) // cell\n    cols = (GRID_W - ox) // cell\n    if rows < 1 or cols < 1:\n        return None\n    sub = arr[oy : oy + rows * cell, ox : ox + cols * cell]\n    return sub.reshape(rows, cell, cols, cell)\n\n\ndef verify_cell(\n    grids: Sequence[np.ndarray],\n    cell: int,\n    ox: int,\n    oy: int,\n    mask: Optional[np.ndarray] = None,\n) -> bool:\n    """Is every cell of the implied grid a single colour, in every frame?\n\n    This is the property downstream code actually depends on: a cell that spans\n    two colours makes a downsampled grid a lie, and every component and track\n    built on it inherits the lie.\n\n    Only cells lying **entirely** inside ``mask`` are judged. A cell straddling\n    the mask boundary mixes content we are modelling with content we are not, so\n    it is not evidence either way - counting it would let a single-pixel UI\n    element veto a real 8x upscale, which is exactly the failure that made the\n    first version of this function reject every game.\n    """\n    if cell <= 1:\n        return True\n    for grid in grids:\n        blocks = _block_view(np.asarray(grid), cell, ox, oy)\n        if blocks is None:\n            return False\n        uniform = (blocks == blocks[:, :1, :, :1]).all(axis=(1, 3))\n        if mask is not None:\n            mblocks = _block_view(mask, cell, ox, oy)\n            assert mblocks is not None\n            judged = mblocks.all(axis=(1, 3))\n        else:\n            judged = np.ones_like(uniform, dtype=bool)\n        if not judged.any():\n            return False\n        if not bool(uniform[judged].all()):\n            return False\n    return True\n\n\ndef _divisors_descending(n: int) -> list[int]:\n    return [d for d in range(n, 0, -1) if n % d == 0]\n\n\n@dataclass(frozen=True, slots=True)\nclass Geometry:\n    """What can be said about a frame\'s structure from one or more observations.\n\n    ``static_mask`` and ``active_rect`` are ``None`` when only one frame was\n    available. They are by far the strongest signal here and they cost nothing\n    beyond frames we already have, so prefer passing a whole rollout.\n    """\n\n    block: BlockGrid\n    border_colour: Optional[int]\n    border_rect: Rect\n    static_mask: Optional[np.ndarray] = None\n    active_rect: Optional[Rect] = None\n    n_frames: int = 1\n\n    @property\n    def is_structured(self) -> bool:\n        return self.block.is_structured\n\n    @property\n    def playfield(self) -> Rect:\n        """Best estimate of where the game happens.\n\n        Change evidence beats block structure beats the trimmed border, in that\n        order, because that is the order of how much each can be trusted.\n        """\n        if self.active_rect is not None and not self.active_rect.is_empty:\n            return self.active_rect\n        if self.block.is_structured:\n            return self.block.rect\n        return self.border_rect\n\n    def click_targets(self) -> list[tuple[int, int]]:\n        """One display pixel per playfield cell - D-005\'s pruning, made concrete.\n\n        Restated in playfield cells as D-011 requires. Returns an empty list when\n        geometry found no structure, so every caller must keep a pixel-space\n        path; on the public demo games that is roughly half of them.\n        """\n        if not self.block.is_structured:\n            return []\n        pf = self.playfield\n        out: list[tuple[int, int]] = []\n        for row in range(self.block.rows):\n            for col in range(self.block.cols):\n                x, y = self.block.centre_of(col, row)\n                if pf.contains(x, y):\n                    out.append((x, y))\n        return out\n\n    def describe(self) -> str:\n        parts = [str(self.block)]\n        if self.border_colour is not None:\n            parts.append(f"border={self.border_colour}@{self.border_rect}")\n        if self.active_rect is not None:\n            parts.append(f"active={self.active_rect}")\n        parts.append(f"n_frames={self.n_frames}")\n        return "  ".join(parts)\n\n\ndef _trim_uniform_border(grid: np.ndarray) -> tuple[Optional[int], Rect]:\n    """Strip a solid single-colour frame border, if there is one."""\n    x0, y0, x1, y1 = 0, 0, GRID_W, GRID_H\n    colour: Optional[int] = None\n    while x1 - x0 > 2 and y1 - y0 > 2:\n        edges = np.concatenate(\n            [\n                grid[y0, x0:x1].ravel(),\n                grid[y1 - 1, x0:x1].ravel(),\n                grid[y0:y1, x0].ravel(),\n                grid[y0:y1, x1 - 1].ravel(),\n            ]\n        )\n        vals = np.unique(edges)\n        if vals.size != 1 or (colour is not None and int(vals[0]) != colour):\n            break\n        colour = int(vals[0])\n        x0, y0, x1, y1 = x0 + 1, y0 + 1, x1 - 1, y1 - 1\n    return colour, Rect(x0, y0, x1, y1)\n\n\ndef bounding_rect(mask: np.ndarray) -> Rect:\n    ys, xs = np.nonzero(mask)\n    if ys.size == 0:\n        return Rect(0, 0, 0, 0)\n    return Rect(int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1)\n\n\ndef induce_geometry(\n    frames: Sequence[np.ndarray] | Iterable[np.ndarray],\n) -> Geometry:\n    """Induce geometry from one or more frames of the same game.\n\n    Pass every frame available, including the intermediate frames of animated\n    transitions - they are free, and they move exactly the things that matter.\n    """\n    grids = [np.asarray(f) for f in frames]\n    if not grids:\n        raise ValueError("induce_geometry needs at least one frame")\n\n    reference = grids[0]\n    border_colour, border_rect = _trim_uniform_border(reference)\n\n    static_mask: Optional[np.ndarray] = None\n    active_rect: Optional[Rect] = None\n    changed: Optional[np.ndarray] = None\n    if len(grids) > 1:\n        stack = np.stack(grids)\n        changed = (stack != stack[0]).any(axis=0)\n        static_mask = ~changed\n        if changed.any():\n            active_rect = bounding_rect(changed)\n        else:\n            changed = None\n\n    # Propose from the *mask* of changing pixels, not its bounding box: on games\n    # whose UI layer paints at full resolution the box contains 1-pixel detail\n    # that drags the GCD to 1, while the mask itself does not.\n    #\n    # Deliberately the whole changing mask, not its largest connected piece.\n    # Restricting to the biggest region was tried: it rescues cn04, where a\n    # small animating UI element vetoes the real 3x upscale, but it breaks ft09\n    # (2 -> 6) and sc25 (1 -> 2) because a smaller region contains fewer runs to\n    # constrain the GCD, and the estimate drifts upward. Overestimating is the\n    # dangerous direction - a too-coarse cell silently merges genuinely distinct\n    # game cells - so the variant with fewer overestimates wins. See D-013.\n    cell, ox, oy = propose_cell(grids, changed)\n\n    chosen, proposed = 1, cell\n    for candidate in _divisors_descending(cell):\n        if candidate == 1 or verify_cell(grids, candidate, ox, oy, changed):\n            chosen = candidate\n            break\n\n    if chosen > 1:\n        rect = active_rect if active_rect is not None else border_rect\n        if rect.is_empty:\n            rect = FULL_FRAME\n        block = BlockGrid(\n            cell=chosen,\n            origin_x=ox % chosen,\n            origin_y=oy % chosen,\n            rect=rect.snapped(chosen, ox % chosen, oy % chosen),\n            proposed_cell=proposed,\n        )\n    else:\n        block = UNSTRUCTURED\n\n    return Geometry(\n        block=block,\n        border_colour=border_colour,\n        border_rect=border_rect,\n        static_mask=static_mask,\n        active_rect=active_rect,\n        n_frames=len(grids),\n    )\n'), 'arc3.perception.segment': (False, '"""Segmentation into candidate object sets.\n\nThe right notion of "object" is environment-dependent, so this module refuses to\npick one. It produces a :class:`SegmentationSet` of several segmentations of the\nsame frame, computed by different rules, and leaves the choice to whatever can\nactually evaluate it - which at this stage of the project is the evaluation\nscripts, and later the ontology subsystem.\n\nThat is deliberate and is the one place the living survey attributes a standalone\n**20-30 point** advantage (``docs/field-review.md`` section 4). It also stays on\nthe right side of Guardrail 2: these are general representational rules, not\nguesses about what kinds of games exist.\n\nSegmentations are computed in **cell space** when\n:mod:`arc3.perception.geometry` found verified block structure, and in pixel\nspace otherwise. On the public demo games that is roughly a half-and-half split,\nso both paths are load-bearing and both are exercised.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Iterator, Optional\n\nimport numpy as np\n\nfrom .components import CONN4, CONN8, label\nfrom .geometry import Geometry, Rect, bounding_rect\n\n\n@dataclass(frozen=True, slots=True)\nclass Component:\n    """One candidate object in a frame.\n\n    ``mask`` is in **display pixel** coordinates whatever space the segmentation\n    was computed in, so that a click target derived from it is always directly\n    usable. ``shape_key`` identifies the component\'s form independently of where\n    it sits, which is what cross-frame matching keys on.\n    """\n\n    index: int\n    mask: np.ndarray\n    bbox: Rect\n    size: int\n    colours: tuple[int, ...]\n    dominant_colour: int\n    shape_key: bytes\n    centroid: tuple[float, float]\n\n    @property\n    def is_rectangular(self) -> bool:\n        return self.size == self.bbox.area\n\n    @property\n    def click_target(self) -> tuple[int, int]:\n        """A pixel inside this component, for ACTION6.\n\n        The centroid when it lands inside the mask, otherwise the first mask\n        pixel in raster order - never a pixel outside the object, which would\n        make the click land on something else.\n        """\n        cx, cy = int(round(self.centroid[0])), int(round(self.centroid[1]))\n        if 0 <= cy < self.mask.shape[0] and 0 <= cx < self.mask.shape[1] and self.mask[cy, cx]:\n            return cx, cy\n        ys, xs = np.nonzero(self.mask)\n        return int(xs[0]), int(ys[0])\n\n\ndef _shape_key(mask: np.ndarray, bbox: Rect, colours: np.ndarray) -> bytes:\n    """Translation-invariant identity: the cropped mask plus its colouring."""\n    crop = mask[bbox.y0 : bbox.y1, bbox.x0 : bbox.x1]\n    cols = np.where(crop, colours[bbox.y0 : bbox.y1, bbox.x0 : bbox.x1], -1).astype(np.int8)\n    return bytes([bbox.height, bbox.width]) + cols.tobytes()\n\n\n@dataclass(frozen=True, slots=True)\nclass Segmentation:\n    """One way of carving a frame into objects."""\n\n    name: str\n    components: tuple[Component, ...]\n    cell: int\n    background_colour: Optional[int]\n\n    def __len__(self) -> int:\n        return len(self.components)\n\n    def __iter__(self) -> Iterator[Component]:\n        return iter(self.components)\n\n    @property\n    def covered(self) -> int:\n        return sum(c.size for c in self.components)\n\n    def click_targets(self) -> list[tuple[int, int]]:\n        """One click candidate per component - D-005\'s pruning, concretely."""\n        return [c.click_target for c in self.components]\n\n    def describe(self) -> str:\n        return f"{self.name}: {len(self.components)} components, cell={self.cell}"\n\n\n@dataclass(frozen=True, slots=True)\nclass SegmentationSet:\n    """Several segmentations of the same frame, computed by different rules."""\n\n    frame: np.ndarray\n    segmentations: tuple[Segmentation, ...]\n    geometry: Optional[Geometry] = None\n\n    def __getitem__(self, name: str) -> Segmentation:\n        for s in self.segmentations:\n            if s.name == name:\n                return s\n        raise KeyError(f"no segmentation named {name!r}; have {[s.name for s in self]}")\n\n    def __iter__(self) -> Iterator[Segmentation]:\n        return iter(self.segmentations)\n\n    def __len__(self) -> int:\n        return len(self.segmentations)\n\n    @property\n    def names(self) -> tuple[str, ...]:\n        return tuple(s.name for s in self.segmentations)\n\n    def union_click_targets(self) -> list[tuple[int, int]]:\n        """Click candidates proposed by *any* segmentation, de-duplicated.\n\n        The union rather than one segmentation\'s list, because D-005 warns that\n        pruning away the winning action is the failure mode to avoid, and we do\n        not yet know which notion of object is right for a given game.\n        """\n        seen: dict[tuple[int, int], None] = {}\n        for seg in self.segmentations:\n            for t in seg.click_targets():\n                seen.setdefault(t, None)\n        return list(seen)\n\n    def describe(self) -> str:\n        return "\\n".join(s.describe() for s in self.segmentations)\n\n\ndef background_colour(grid: np.ndarray, mask: Optional[np.ndarray] = None) -> int:\n    """The most common colour, which is the best available background guess.\n\n    A guess, not a fact: a game whose playfield is mostly one object colour will\n    fool it. Every segmentation that uses it is offered alongside one that does\n    not, so a wrong guess costs a candidate rather than the whole set.\n    """\n    values = grid[mask] if mask is not None else grid\n    counts = np.bincount(np.asarray(values).ravel().astype(np.int64), minlength=16)\n    return int(np.argmax(counts))\n\n\ndef _components_from_labels(\n    labels: np.ndarray, n: int, colours: np.ndarray, scale: int, ox: int, oy: int\n) -> tuple[Component, ...]:\n    """Build display-space components from a labelling in cell space."""\n    out: list[Component] = []\n    for i in range(n):\n        cell_mask = labels == i\n        if not cell_mask.any():\n            continue\n        if scale > 1:\n            mask = np.zeros(\n                (cell_mask.shape[0] * scale + oy, cell_mask.shape[1] * scale + ox), dtype=bool\n            )\n            up = np.repeat(np.repeat(cell_mask, scale, axis=0), scale, axis=1)\n            mask[oy : oy + up.shape[0], ox : ox + up.shape[1]] = up\n            mask = _pad_to_frame(mask)\n            pix_colours = _pad_to_frame(\n                np.repeat(np.repeat(colours, scale, axis=0), scale, axis=1), offset=(ox, oy)\n            )\n        else:\n            mask = _pad_to_frame(cell_mask)\n            pix_colours = _pad_to_frame(colours)\n        bbox = bounding_rect(mask)\n        vals = pix_colours[mask]\n        uniq, counts = np.unique(vals, return_counts=True)\n        ys, xs = np.nonzero(mask)\n        out.append(\n            Component(\n                index=len(out),\n                mask=mask,\n                bbox=bbox,\n                size=int(mask.sum()),\n                colours=tuple(int(v) for v in uniq),\n                dominant_colour=int(uniq[int(np.argmax(counts))]),\n                shape_key=_shape_key(mask, bbox, pix_colours),\n                centroid=(float(xs.mean()), float(ys.mean())),\n            )\n        )\n    out.sort(key=lambda c: (c.bbox.y0, c.bbox.x0))\n    return tuple(\n        Component(\n            index=i,\n            mask=c.mask,\n            bbox=c.bbox,\n            size=c.size,\n            colours=c.colours,\n            dominant_colour=c.dominant_colour,\n            shape_key=c.shape_key,\n            centroid=c.centroid,\n        )\n        for i, c in enumerate(out)\n    )\n\n\ndef _pad_to_frame(arr: np.ndarray, offset: tuple[int, int] = (0, 0)) -> np.ndarray:\n    """Place ``arr`` into a 64x64 frame at ``offset``, cropping any overflow."""\n    from ..types import GRID_H, GRID_W\n\n    ox, oy = offset\n    out = np.zeros((GRID_H, GRID_W), dtype=arr.dtype)\n    h = min(GRID_H - oy, arr.shape[0])\n    w = min(GRID_W - ox, arr.shape[1])\n    if h > 0 and w > 0:\n        out[oy : oy + h, ox : ox + w] = arr[:h, :w]\n    return out\n\n\ndef segment(\n    frame: np.ndarray,\n    geometry: Optional[Geometry] = None,\n    *,\n    include: Optional[tuple[str, ...]] = None,\n) -> SegmentationSet:\n    """Produce several candidate segmentations of ``frame``.\n\n    The rules, all general and none game-specific:\n\n    - ``colour_4`` / ``colour_8`` - maximal same-colour regions. The classic ARC\n      notion of an object.\n    - ``foreground_4`` / ``foreground_8`` - regions of anything that is not the\n      background colour, so a multi-coloured sprite stays one object rather than\n      fragmenting into its colours.\n    - ``colour_mass`` - all cells of one colour as a single object regardless of\n      connectivity, which is what a scattered collectible set looks like.\n    - ``bbox`` - the bounding box of each same-colour region filled in, so a\n      hollow outline reads as the region it encloses.\n\n    Pass ``geometry`` whenever you have it: with verified block structure the\n    segmentation runs in cell space, which is both faster and correct on the\n    upscaled games where pixel-space components are ``cell**2`` times too large.\n    """\n    frame = np.asarray(frame)\n    cell, ox, oy = 1, 0, 0\n    work = frame\n    if geometry is not None and geometry.block.is_structured:\n        block = geometry.block\n        work, ox, oy = block.tile_frame(frame)\n        cell = block.cell\n\n    bg = background_colour(work)\n    segs: list[Segmentation] = []\n\n    def add(name: str, labels: np.ndarray, n: int) -> None:\n        if include is not None and name not in include:\n            return\n        segs.append(\n            Segmentation(\n                name=name,\n                components=_components_from_labels(labels, n, work, cell, ox, oy),\n                cell=cell,\n                background_colour=bg,\n            )\n        )\n\n    for conn, suffix in ((CONN4, "4"), (CONN8, "8")):\n        labels, n = label(work, connectivity=conn)\n        add(f"colour_{suffix}", labels, n)\n\n        # Always emitted, even when the frame is entirely background and the\n        # result is empty. A missing key is a worse failure mode than an empty\n        # segmentation: callers select a segmentation by name once and use it\n        # for a whole run, and a name that vanishes on a blank frame turns a\n        # legitimate "nothing is visible" into a KeyError mid-episode.\n        fg_mask = work != bg\n        flabels, fn = label(fg_mask.astype(np.int8), connectivity=conn, equal_values=False)\n        add(f"foreground_{suffix}", flabels, fn)\n\n    # One object per colour, ignoring connectivity.\n    if include is None or "colour_mass" in include:\n        mass = np.full(work.shape, -1, dtype=np.int32)\n        n = 0\n        for colour in sorted(int(c) for c in np.unique(work)):\n            m = work == colour\n            mass[m] = n\n            n += 1\n        add("colour_mass", mass, n)\n\n    # Filled bounding boxes of same-colour regions.\n    if include is None or "bbox" in include:\n        labels, n = label(work, connectivity=CONN8)\n        boxed = np.full(work.shape, -1, dtype=np.int32)\n        for i in range(n):\n            m = labels == i\n            r = bounding_rect(m)\n            boxed[r.y0 : r.y1, r.x0 : r.x1] = i\n        add("bbox", boxed, n)\n\n    return SegmentationSet(frame=frame, segmentations=tuple(segs), geometry=geometry)\n'), 'arc3.perception.track': (False, '"""Object identity across frames.\n\nThe hard component and the acknowledged tarpit risk (D-007). The job: given a\nsegmentation of frame *t* and of frame *t+1*, say which component in one *is*\nwhich component in the other, through movement, colour change, occlusion,\nappearance and disappearance.\n\nTwo commitments shape the design:\n\n- **No hard rejection** (Guardrail 1, D-003). A match is scored, never asserted.\n  A track that loses its object keeps a decaying confidence and a memory of what\n  it looked like, so it can be reacquired after an occlusion rather than being\n  destroyed by one bad frame. Hidden state means a correct-but-conditional\n  hypothesis will sometimes look wrong, and deleting on that is unsound.\n- **Matching is global, not greedy per component.** Greedy nearest-match breaks\n  exactly where it matters - two identical sprites passing each other - so\n  assignment is done by cost minimisation over the whole frame pair. With at\n  most a few dozen components the Hungarian algorithm is affordable, and\n  internal compute is free.\n\nCost between two components is built from evidence that is exact, in decreasing\norder of how much it should count: identical shape and colouring, identical\nshape after recolouring, mask overlap, size similarity, and distance moved.\nNothing here is learned or fitted.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom typing import Optional, Sequence\n\nimport numpy as np\n\nfrom .segment import Component, Segmentation\n\n#: Cost assigned to leaving a component unmatched. A pairing only happens when\n#: it is cheaper than both sides going unexplained, so this sets the bar for\n#: "these are the same object" without being a tuned similarity threshold:\n#: it is the price of the alternative explanation, appearance-plus-disappearance.\nUNMATCHED_COST = 1.0\n\n\n@dataclass(frozen=True, slots=True)\nclass Match:\n    """One component in frame *t* identified with one in frame *t+1*."""\n\n    prev_index: int\n    next_index: int\n    cost: float\n    displacement: tuple[int, int]\n    same_shape: bool\n    same_colour: bool\n\n    @property\n    def moved(self) -> bool:\n        return self.displacement != (0, 0)\n\n    @property\n    def recoloured(self) -> bool:\n        return self.same_shape and not self.same_colour\n\n    @property\n    def confidence(self) -> float:\n        """1.0 for a perfect identification, falling to 0 at the unmatched bar."""\n        return max(0.0, 1.0 - self.cost / UNMATCHED_COST)\n\n\n@dataclass(frozen=True, slots=True)\nclass FrameMatching:\n    """The full correspondence between two consecutive segmentations."""\n\n    matches: tuple[Match, ...]\n    appeared: tuple[int, ...]\n    disappeared: tuple[int, ...]\n\n    @property\n    def n_matched(self) -> int:\n        return len(self.matches)\n\n    def by_prev(self) -> dict[int, Match]:\n        return {m.prev_index: m for m in self.matches}\n\n    def by_next(self) -> dict[int, Match]:\n        return {m.next_index: m for m in self.matches}\n\n    def rigid_motions(self) -> dict[tuple[int, int], int]:\n        """How many objects moved by each displacement.\n\n        A single dominant non-zero displacement is the signature of "the action\n        moved everything" (a scrolling camera); a single object with a non-zero\n        displacement while everything else is still is the signature of "this is\n        me". Both are ontology\'s business, but this is where the evidence is.\n        """\n        counts: dict[tuple[int, int], int] = {}\n        for m in self.matches:\n            counts[m.displacement] = counts.get(m.displacement, 0) + 1\n        return counts\n\n\ndef _pair_cost(a: Component, b: Component) -> float:\n    """Cost of identifying ``a`` in frame *t* with ``b`` in frame *t+1*.\n\n    Zero means indistinguishable and in the same place; costs at or above\n    :data:`UNMATCHED_COST` mean the pairing is not worth making.\n    """\n    if a.shape_key == b.shape_key:\n        base = 0.0\n    else:\n        # Same silhouette, different colouring - a toggle or state flip.\n        same_silhouette = (\n            a.bbox.width == b.bbox.width\n            and a.bbox.height == b.bbox.height\n            and a.size == b.size\n            and np.array_equal(\n                a.mask[a.bbox.y0 : a.bbox.y1, a.bbox.x0 : a.bbox.x1],\n                b.mask[b.bbox.y0 : b.bbox.y1, b.bbox.x0 : b.bbox.x1],\n            )\n        )\n        if same_silhouette:\n            base = 0.25\n        else:\n            overlap = float(np.logical_and(a.mask, b.mask).sum())\n            union = float(np.logical_or(a.mask, b.mask).sum())\n            iou = overlap / union if union else 0.0\n            size_ratio = min(a.size, b.size) / max(a.size, b.size)\n            # Deformation is real (growing, shrinking, merging), so it is priced\n            # rather than forbidden - but it must cost more than a clean move.\n            base = 0.5 + 0.5 * (1.0 - iou) * (1.0 - size_ratio)\n\n    dx = b.centroid[0] - a.centroid[0]\n    dy = b.centroid[1] - a.centroid[1]\n    distance = (dx * dx + dy * dy) ** 0.5\n    # Movement is expected and cheap, but not free: between two otherwise\n    # identical candidates the nearer one wins. Saturates so that a long jump is\n    # still preferred over inventing an appearance and a disappearance.\n    move_penalty = 0.4 * (1.0 - 1.0 / (1.0 + distance / 8.0))\n    return base + move_penalty\n\n\ndef _hungarian(cost: np.ndarray) -> list[tuple[int, int]]:\n    """Minimum-cost assignment (Jonker-Volgenant style shortest augmenting path).\n\n    Rectangular input; returns ``(row, col)`` pairs for the ``min(n, m)`` rows\n    that get assigned. Written out rather than pulled from scipy to keep the\n    offline dependency set to what ``arc-agi`` already installs.\n    """\n    n, m = cost.shape\n    if n == 0 or m == 0:\n        return []\n    transposed = n > m\n    c = cost.T if transposed else cost\n    n, m = c.shape\n\n    INF = float("inf")\n    u = np.zeros(n + 1)\n    v = np.zeros(m + 1)\n    p = np.zeros(m + 1, dtype=int)  # p[j] = row assigned to column j\n    way = np.zeros(m + 1, dtype=int)\n\n    for i in range(1, n + 1):\n        p[0] = i\n        j0 = 0\n        minv = np.full(m + 1, INF)\n        used = np.zeros(m + 1, dtype=bool)\n        while True:\n            used[j0] = True\n            i0 = p[j0]\n            delta = INF\n            j1 = -1\n            for j in range(1, m + 1):\n                if used[j]:\n                    continue\n                cur = c[i0 - 1, j - 1] - u[i0] - v[j]\n                if cur < minv[j]:\n                    minv[j] = cur\n                    way[j] = j0\n                if minv[j] < delta:\n                    delta = minv[j]\n                    j1 = j\n            if j1 < 0:\n                break\n            for j in range(m + 1):\n                if used[j]:\n                    u[p[j]] += delta\n                    v[j] -= delta\n                else:\n                    minv[j] -= delta\n            j0 = j1\n            if p[j0] == 0:\n                break\n        while j0:\n            j1 = way[j0]\n            p[j0] = p[j1]\n            j0 = j1\n\n    pairs = []\n    for j in range(1, m + 1):\n        if p[j] > 0:\n            row, col = p[j] - 1, j - 1\n            pairs.append((col, row) if transposed else (row, col))\n    return pairs\n\n\ndef match_frames(prev: Segmentation, nxt: Segmentation) -> FrameMatching:\n    """Identify the components of ``prev`` with those of ``nxt``.\n\n    Pairings costing at least :data:`UNMATCHED_COST` are discarded, which is what\n    lets objects genuinely appear and disappear rather than being forced into a\n    correspondence that does not exist.\n    """\n    a, b = prev.components, nxt.components\n    if not a or not b:\n        return FrameMatching(\n            matches=(),\n            appeared=tuple(range(len(b))),\n            disappeared=tuple(range(len(a))),\n        )\n\n    cost = np.array([[_pair_cost(x, y) for y in b] for x in a], dtype=float)\n    matches: list[Match] = []\n    used_prev: set[int] = set()\n    used_next: set[int] = set()\n    for i, j in _hungarian(cost):\n        if cost[i, j] >= UNMATCHED_COST:\n            continue\n        ca, cb = a[i], b[j]\n        matches.append(\n            Match(\n                prev_index=i,\n                next_index=j,\n                cost=float(cost[i, j]),\n                displacement=(\n                    int(round(cb.centroid[0] - ca.centroid[0])),\n                    int(round(cb.centroid[1] - ca.centroid[1])),\n                ),\n                same_shape=ca.shape_key == cb.shape_key\n                or (ca.size == cb.size and ca.bbox.width == cb.bbox.width),\n                same_colour=ca.dominant_colour == cb.dominant_colour,\n            )\n        )\n        used_prev.add(i)\n        used_next.add(j)\n\n    matches.sort(key=lambda m: m.prev_index)\n    return FrameMatching(\n        matches=tuple(matches),\n        appeared=tuple(j for j in range(len(b)) if j not in used_next),\n        disappeared=tuple(i for i in range(len(a)) if i not in used_prev),\n    )\n\n\n@dataclass\nclass Track:\n    """One object\'s identity carried through time.\n\n    ``confidence`` decays while unseen instead of the track being deleted\n    (Guardrail 1). A track below :data:`REACQUIRE_FLOOR` is dormant, not gone,\n    and will be revived if something matching its remembered appearance shows up\n    - which is exactly what happens when an object emerges from behind another.\n    """\n\n    track_id: int\n    first_frame: int\n    last_seen_frame: int\n    component: Component\n    history: list[tuple[int, Component]] = field(default_factory=list)\n    confidence: float = 1.0\n    frames_unseen: int = 0\n\n    @property\n    def is_active(self) -> bool:\n        return self.frames_unseen == 0\n\n    @property\n    def positions(self) -> list[tuple[float, float]]:\n        return [c.centroid for _, c in self.history]\n\n    def displacement_since(self, frames: int = 1) -> Optional[tuple[float, float]]:\n        if len(self.history) <= frames:\n            return None\n        a = self.history[-1 - frames][1].centroid\n        b = self.history[-1][1].centroid\n        return (b[0] - a[0], b[1] - a[1])\n\n    @property\n    def total_displacement(self) -> tuple[float, float]:\n        if len(self.history) < 2:\n            return (0.0, 0.0)\n        a, b = self.history[0][1].centroid, self.history[-1][1].centroid\n        return (b[0] - a[0], b[1] - a[1])\n\n    @property\n    def is_stationary(self) -> bool:\n        return all(c.centroid == self.history[0][1].centroid for _, c in self.history)\n\n\n#: Confidence below which a track is dormant. It is never deleted, so this only\n#: controls whether a track is offered as a current object, not whether it is\n#: remembered.\nREACQUIRE_FLOOR = 0.1\n\n#: Multiplied into confidence for each frame an object goes unseen.\nDECAY = 0.6\n\n\nclass Tracker:\n    """Maintains :class:`Track` identities across a sequence of segmentations.\n\n    Feed it one segmentation per frame, in order. The segmentation must be\n    produced by the same rule each time - mixing ``colour_4`` and\n    ``foreground_8`` frame to frame would make identity meaningless. That\n    requirement used to be a docstring only: ``segmentation_name`` was stored\n    and never read, so the error it warns about was undetectable. It is now\n    checked in :meth:`update` (D-015).\n\n    The default is ``colour_8``, which is what D-014\'s 0.8% id-switch rate was\n    measured on. It was ``foreground_8`` until D-015 pointed out that the\n    measured carving and the default carving were not the same one.\n    """\n\n    def __init__(self, segmentation_name: str = "colour_8") -> None:\n        self.segmentation_name = segmentation_name\n        self.tracks: dict[int, Track] = {}\n        self._next_id = 0\n        self._frame = -1\n        self._prev: Optional[Segmentation] = None\n        self._prev_track_of: dict[int, int] = {}\n\n    @property\n    def active_tracks(self) -> list[Track]:\n        return [t for t in self.tracks.values() if t.is_active]\n\n    def current_ids(self) -> dict[int, int]:\n        """Component index in the last segmentation fed -> its track id."""\n        return dict(self._prev_track_of)\n\n    @property\n    def live_tracks(self) -> list[Track]:\n        """Active tracks plus dormant ones still above the reacquisition floor."""\n        return [t for t in self.tracks.values() if t.confidence >= REACQUIRE_FLOOR]\n\n    def update(self, seg: Segmentation) -> FrameMatching:\n        """Advance one frame. Returns the frame-to-frame matching."""\n        if seg.name != self.segmentation_name:\n            raise ValueError(\n                f"tracker is running on {self.segmentation_name!r} but was fed "\n                f"{seg.name!r} at frame {self._frame + 1}; identity across a "\n                "mixed carving is meaningless"\n            )\n        self._frame += 1\n        if self._prev is None:\n            self._prev = seg\n            self._prev_track_of = {}\n            for c in seg.components:\n                t = self._new_track(c)\n                self._prev_track_of[c.index] = t.track_id\n            return FrameMatching(\n                matches=(), appeared=tuple(range(len(seg.components))), disappeared=()\n            )\n\n        matching = match_frames(self._prev, seg)\n        next_track_of: dict[int, int] = {}\n\n        for m in matching.matches:\n            tid = self._prev_track_of.get(m.prev_index)\n            if tid is None:\n                continue\n            track = self.tracks[tid]\n            comp = seg.components[m.next_index]\n            track.component = comp\n            track.history.append((self._frame, comp))\n            track.last_seen_frame = self._frame\n            track.frames_unseen = 0\n            # Confidence follows the quality of the identification, and recovers\n            # rather than ratcheting down permanently.\n            track.confidence = min(1.0, 0.5 * track.confidence + 0.5 * m.confidence + 0.25)\n            next_track_of[m.next_index] = tid\n\n        for i in matching.disappeared:\n            tid = self._prev_track_of.get(i)\n            if tid is None:\n                continue\n            track = self.tracks[tid]\n            track.frames_unseen += 1\n            track.confidence *= DECAY  # down-weight, never delete\n\n        for j in matching.appeared:\n            comp = seg.components[j]\n            revived = self._try_reacquire(comp)\n            if revived is not None:\n                revived.component = comp\n                revived.history.append((self._frame, comp))\n                revived.last_seen_frame = self._frame\n                revived.frames_unseen = 0\n                revived.confidence = min(1.0, revived.confidence + 0.4)\n                next_track_of[j] = revived.track_id\n            else:\n                next_track_of[j] = self._new_track(comp).track_id\n\n        self._prev = seg\n        self._prev_track_of = next_track_of\n        return matching\n\n    def _try_reacquire(self, comp: Component) -> Optional[Track]:\n        """Match a newly appeared component against dormant tracks.\n\n        This is what makes occlusion survivable: an object hidden behind another\n        for three frames comes back as the same identity instead of a new one.\n        """\n        best: Optional[Track] = None\n        best_cost = UNMATCHED_COST\n        for track in self.tracks.values():\n            if track.is_active or track.confidence < REACQUIRE_FLOOR:\n                continue\n            cost = _pair_cost(track.component, comp)\n            if cost < best_cost:\n                best, best_cost = track, cost\n        return best\n\n    def _new_track(self, comp: Component) -> Track:\n        track = Track(\n            track_id=self._next_id,\n            first_frame=self._frame,\n            last_seen_frame=self._frame,\n            component=comp,\n            history=[(self._frame, comp)],\n        )\n        self.tracks[self._next_id] = track\n        self._next_id += 1\n        return track\n\n\ndef track_sequence(\n    segmentations: Sequence[Segmentation], *, name: str = "colour_8"\n) -> Tracker:\n    """Run a tracker over a whole sequence of segmentations."""\n    tracker = Tracker(segmentation_name=name)\n    for seg in segmentations:\n        tracker.update(seg)\n    return tracker\n'), 'arc3.propose': (False, '"""Prompts that ask the LLM for goal hypotheses (D-035).\n\nThe model is told only what the agent actually knows: the current scene, what\neach action has been observed to do, what won the previous level (if one has\nbeen won), and how the existing hypotheses have fared. It is asked for several\n*different* hypotheses as code, never for an action.\n\nPrompt text is part of the experiment: change it and recorded calls stop\nreplaying (`llm.ReplayLLM` keys on the exact messages). Bump ``PROMPT_VERSION``\nwhenever the wording changes so recordings say which prompt produced them.\n\n**What this does not do.** It gives no hint about what kinds of games exist.\nGuardrail 2 forbids hand-crafting the hypothesis space, so the prompt describes\nthe *interface* (what a scene is, what an action is) and nothing about genres,\ntypical goals or known games.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Optional, Sequence\n\nfrom .hypothesis import Hypothesis\nfrom .scene import Scene\n\nPROMPT_VERSION = "goal-v1"\n\nSYSTEM = """\\\nYou help an agent play an unknown turn-based puzzle game on a 64x64 grid of \\\n16 colours (0-15). Nobody knows the rules or the goal; the agent learns them by \\\nacting. Your job is to propose hypotheses about what state of the grid WINS the \\\ncurrent level, written as Python functions the agent can evaluate on any state.\n\nWrite each hypothesis as a separate ```python code block that starts with a \\\nline `# GOAL: <one sentence>` and defines exactly one function:\n\n    def progress(scene):\n        return <float between 0.0 and 1.0>   # 1.0 = goal reached\n\nIt must be higher for states closer to the goal, so the agent can steer by it. \\\nAvailable inside the function: `np` (numpy) and `math`. No imports. The scene \\\nAPI is:\n  scene.objects        list of objects, largest first. Each has: id, colour, size,\n                       x0, y0, x1, y1 (box, x1/y1 exclusive), w, h, cx, cy (centre)\n  scene.of_colour(c)   objects of colour c\n  scene.count(c)       number of cells of colour c\n  scene.colours()      colours present\n  scene.grid           64x64 numpy int array, grid[y, x]; -1 marks hidden UI rows\n  scene.background     the background colour\nObjects are same-colour connected regions; ids are NOT stable between states, \\\nso find objects by colour, size or position, never by id.\nPropose hypotheses that differ from each other. Keep each function short."""\n\n\n@dataclass\nclass Evidence:\n    """What the agent knows, for the prompt."""\n\n    scene: Scene\n    actions: Sequence[str]                      # e.g. ["ACTION1", "CLICK(x,y)"]\n    effects: Sequence[str]                      # observed action -> effect lines\n    level: int\n    actions_used: int\n    previous_win: Optional[str] = None          # what changed on the winning move\n    hypotheses: Sequence[Hypothesis] = ()\n\n\ndef goal_messages(ev: Evidence, n: int = 3) -> list[dict]:\n    parts = [f"Level {ev.level + 1}. Actions used so far on this level: {ev.actions_used}.",\n             "Available actions: " + ", ".join(ev.actions) + "."]\n    if ev.effects:\n        parts.append("What actions have been observed to do (most recent last):")\n        parts += ["  " + e for e in ev.effects[-12:]]\n    if ev.previous_win:\n        parts.append("The previous level was won. On the winning move: " + ev.previous_win)\n    if ev.hypotheses:\n        parts.append("Hypotheses already being tested, with their evidence so far:")\n        for h in ev.hypotheses[:6]:\n            parts.append("  " + h.describe())\n        parts.append("Propose NEW hypotheses; do not repeat these. A high \'false\' count "\n                     "means the agent reached states that hypothesis called the goal, "\n                     "and the level did not end.")\n    parts.append("Current state:")\n    parts.append(ev.scene.describe())\n    parts.append(f"Propose {n} different goal hypotheses as described.")\n    return [{"role": "system", "content": SYSTEM},\n            {"role": "user", "content": "\\n".join(parts)}]\n\n\ndef describe_effect(before: Scene, after: Scene, action: str) -> str:\n    """One line: what an action did, in object terms, for the prompt."""\n    if (before.grid == after.grid).all():\n        return f"{action}: nothing changed"\n    moved, changed = [], 0\n    b_by = {(o.colour, o.size): o for o in before.objects}\n    a_by = {(o.colour, o.size): o for o in after.objects}\n    for k, o in a_by.items():\n        p = b_by.get(k)\n        if p is not None and (p.x0, p.y0) != (o.x0, o.y0):\n            moved.append(f"colour {o.colour} size {o.size} moved "\n                         f"({o.x0 - p.x0:+d},{o.y0 - p.y0:+d})")\n    changed = int(((before.grid != after.grid) & (after.grid >= 0)).sum())\n    appeared = len(set(a_by) - set(b_by))\n    vanished = len(set(b_by) - set(a_by))\n    bits = moved[:3]\n    if appeared or vanished:\n        bits.append(f"{appeared} object(s) appeared, {vanished} vanished")\n    bits.append(f"{changed} cells changed")\n    return f"{action}: " + "; ".join(bits)\n'), 'arc3.scene': (False, '"""The symbolic scene: what the LLM is shown, and what its code runs against (D-035).\n\nOne object, two uses, deliberately the same:\n\n- :meth:`Scene.describe` renders it as text for a prompt.\n- Hypothesis code the LLM writes receives a :class:`Scene` and computes on it.\n\nSo the model writes code against exactly the representation it was shown, and\nno field appears in one that is absent from the other. Perception stays exact\n(D-006): objects are `colour_8` connected components, not a vision model\'s\nreading of a picture.\n\n**What this does not do.**\n\n- **One carving only.** Objects are same-colour 8-connected regions. A\n  multi-coloured sprite arrives as several objects (the 2.55 fragmentation of\n  §3 in architecture.md); the grid text is there so the model can see through\n  that, the object list will not.\n- **No identity across frames.** Object ``id`` is an index into this frame\'s\n  list. Tracking (`perception.track`) exists but is not wired into scenes yet.\n- **The bookkeeping band is removed**, when known, from both the objects and\n  the grid. A goal that is literally "make the counter reach N" is therefore\n  invisible to hypotheses, which is a deliberate trade against the far more\n  common case of the counter polluting every state.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom typing import Optional\n\nimport numpy as np\n\nfrom .perception.segment import segment\n\n#: Hex digit per colour, the grid encoding used in prompts.\nHEX = "0123456789abcdef"\n\n#: Objects listed in a prompt, largest first. Hypothesis code sees all of them.\nPROMPT_OBJECTS = 40\n\n#: Objects smaller than this are omitted from the prompt list (still in code).\nPROMPT_MIN_SIZE = 2\n\n\n@dataclass(frozen=True)\nclass Obj:\n    """One object: a same-colour 8-connected region."""\n\n    id: int\n    colour: int\n    size: int\n    x0: int\n    y0: int\n    x1: int  # exclusive\n    y1: int  # exclusive\n    cx: float\n    cy: float\n\n    @property\n    def w(self) -> int:\n        return self.x1 - self.x0\n\n    @property\n    def h(self) -> int:\n        return self.y1 - self.y0\n\n    def describe(self) -> str:\n        return (f"#{self.id} colour {self.colour} size {self.size} "\n                f"box x{self.x0}-{self.x1 - 1} y{self.y0}-{self.y1 - 1} ({self.w}x{self.h})")\n\n\n@dataclass\nclass Scene:\n    """A frame as objects plus the raw grid. Built by :func:`make_scene`."""\n\n    grid: np.ndarray            # 64x64 int, band rows set to -1\n    background: int\n    objects: list[Obj]\n    band_rows: tuple[int, ...] = ()\n    _by_colour: dict = field(default_factory=dict, repr=False)\n\n    # -- helpers the LLM\'s code may use -----------------------------------------\n\n    def of_colour(self, colour: int) -> list[Obj]:\n        if not self._by_colour:\n            for o in self.objects:\n                self._by_colour.setdefault(o.colour, []).append(o)\n        return list(self._by_colour.get(colour, []))\n\n    def count(self, colour: int) -> int:\n        """Pixels of ``colour`` outside the band."""\n        return int((self.grid == colour).sum())\n\n    def colours(self) -> list[int]:\n        return sorted(int(c) for c in np.unique(self.grid) if c >= 0)\n\n    # -- rendering -------------------------------------------------------------------\n\n    def crop(self) -> tuple[int, int, int, int]:\n        """Bounding box (x0, y0, x1, y1) of everything that is not background."""\n        fg = (self.grid != self.background) & (self.grid >= 0)\n        if not fg.any():\n            return 0, 0, self.grid.shape[1], self.grid.shape[0]\n        ys, xs = np.nonzero(fg)\n        return int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1\n\n    def grid_text(self) -> str:\n        """The non-background crop as hex rows, with coordinates."""\n        x0, y0, x1, y1 = self.crop()\n        lines = [f"(grid crop x{x0}-{x1 - 1}, y{y0}-{y1 - 1}; one hex digit per cell, "\n                 f"\'.\' = background {self.background}, \'_\' = hidden UI row)"]\n        for y in range(y0, y1):\n            row = self.grid[y, x0:x1]\n            if (row < 0).all():\n                lines.append(f"y{y:02d} " + "_" * (x1 - x0))\n                continue\n            lines.append(f"y{y:02d} " + "".join(\n                "." if v == self.background else ("_" if v < 0 else HEX[v]) for v in row))\n        return "\\n".join(lines)\n\n    def describe(self, with_grid: bool = True) -> str:\n        shown = [o for o in self.objects if o.size >= PROMPT_MIN_SIZE][:PROMPT_OBJECTS]\n        parts = [f"background colour {self.background}; {len(self.objects)} objects "\n                 f"(showing {len(shown)} largest)"]\n        parts += ["  " + o.describe() for o in shown]\n        if with_grid:\n            parts.append(self.grid_text())\n        return "\\n".join(parts)\n\n\ndef make_scene(frame: np.ndarray, band: Optional[np.ndarray] = None) -> Scene:\n    """Build a scene from a resting frame. ``band`` is a 64x64 bool mask of UI rows."""\n    grid = np.asarray(frame, dtype=np.int16).copy()\n    rows: tuple[int, ...] = ()\n    if band is not None and band.any():\n        rows = tuple(int(r) for r in np.nonzero(band.any(axis=1))[0])\n        grid[band] = -1\n    seg = segment(np.asarray(frame), include=("colour_8",))["colour_8"]\n    bg = seg.background_colour if seg.background_colour is not None else 0\n    objs: list[Obj] = []\n    comps = [c for c in seg.components if c.dominant_colour != bg]\n    if band is not None and band.any():\n        comps = [c for c in comps if not band[c.mask].all()]\n    comps.sort(key=lambda c: (-c.size, c.bbox.y0, c.bbox.x0))\n    for i, c in enumerate(comps):\n        objs.append(Obj(\n            id=i, colour=int(c.dominant_colour), size=int(c.size),\n            x0=int(c.bbox.x0), y0=int(c.bbox.y0), x1=int(c.bbox.x1), y1=int(c.bbox.y1),\n            cx=round(float(c.centroid[0]), 1), cy=round(float(c.centroid[1]), 1),\n        ))\n    return Scene(grid=grid, background=int(bg), objects=objs, band_rows=rows)\n'), 'arc3.types': (False, '"""Core typed values shared by every subsystem.\n\nDesign notes that matter downstream:\n\n- An :class:`Observation` is *one action\'s worth* of return, which is a\n  **sequence** of 0..N grids, not a single grid. Sequences of up to 42 frames\n  occur (sb26), and the intermediate frames show the trajectory of motion rather\n  than just its endpoints. Anything that reduces an observation to its last\n  frame is discarding the richest free evidence the environment gives us.\n  See ``docs/benchmark.md`` and D-009.\n- Zero frames is a legal, meaningful return: the action was accepted and did\n  nothing. It is distinct from an error.\n- Grids are immutable. Perception is exact (D-006) and every consumer assumes it\n  can hold a reference without defensive copying.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom enum import Enum\nfrom typing import Any, Iterator, Optional, Sequence\n\nimport numpy as np\n\nGRID_H = 64\nGRID_W = 64\nN_COLOURS = 16\n\n# Action ids, mirroring arcengine.GameAction without importing it, so that\n# replayed episodes and pure-perception tests never need the engine present.\nRESET = 0\nCLICK = 6  # ACTION6, the only parameterised action: takes (x, y)\nALL_ACTION_IDS = (0, 1, 2, 3, 4, 5, 6, 7)\n\n\nclass GameState(str, Enum):\n    NOT_PLAYED = "NOT_PLAYED"\n    NOT_FINISHED = "NOT_FINISHED"\n    WIN = "WIN"\n    GAME_OVER = "GAME_OVER"\n\n    @property\n    def is_terminal(self) -> bool:\n        return self in (GameState.WIN, GameState.GAME_OVER)\n\n\ndef freeze_grid(a: Any) -> np.ndarray:\n    """Return ``a`` as a validated, read-only 64x64 int8 grid."""\n    arr = np.asarray(a, dtype=np.int8)\n    if arr.shape != (GRID_H, GRID_W):\n        raise ValueError(f"expected a {GRID_H}x{GRID_W} grid, got {arr.shape}")\n    if arr.size and (int(arr.min()) < 0 or int(arr.max()) >= N_COLOURS):\n        raise ValueError(\n            f"colours out of range [0, {N_COLOURS}): {int(arr.min())}..{int(arr.max())}"\n        )\n    out = arr.copy() if arr.flags.writeable else arr\n    out.flags.writeable = False\n    return out\n\n\n@dataclass(frozen=True, slots=True)\nclass Action:\n    """An environment action. Constructing one is free; *spending* one is not."""\n\n    id: int\n    x: Optional[int] = None\n    y: Optional[int] = None\n\n    def __post_init__(self) -> None:\n        if self.id not in ALL_ACTION_IDS:\n            raise ValueError(f"unknown action id {self.id}")\n        if self.id == CLICK:\n            if self.x is None or self.y is None:\n                raise ValueError("ACTION6 requires x and y")\n            if not (0 <= self.x < GRID_W and 0 <= self.y < GRID_H):\n                raise ValueError(f"click out of bounds: ({self.x}, {self.y})")\n        elif self.x is not None or self.y is not None:\n            raise ValueError(f"action {self.id} takes no coordinates")\n\n    @property\n    def is_click(self) -> bool:\n        return self.id == CLICK\n\n    @property\n    def is_reset(self) -> bool:\n        return self.id == RESET\n\n    @property\n    def data(self) -> Optional[dict[str, int]]:\n        """The ``data`` payload the engine expects, or None."""\n        if self.is_click:\n            assert self.x is not None and self.y is not None\n            return {"x": self.x, "y": self.y}\n        return None\n\n    def __str__(self) -> str:\n        if self.is_click:\n            return f"ACTION6({self.x},{self.y})"\n        return "RESET" if self.is_reset else f"ACTION{self.id}"\n\n    def to_json(self) -> dict[str, Any]:\n        d: dict[str, Any] = {"id": self.id}\n        if self.is_click:\n            d["x"], d["y"] = self.x, self.y\n        return d\n\n    @staticmethod\n    def from_json(d: dict[str, Any]) -> "Action":\n        return Action(id=int(d["id"]), x=d.get("x"), y=d.get("y"))\n\n    @staticmethod\n    def click(x: int, y: int) -> "Action":\n        return Action(CLICK, x, y)\n\n    @staticmethod\n    def simple(action_id: int) -> "Action":\n        return Action(action_id)\n\n\n@dataclass(frozen=True, slots=True)\nclass GameMetadata:\n    """Per-game facts the engine supplies outside the frame stream.\n\n    ``tags`` and ``baseline_actions`` are permitted agent inputs (D-012) but are\n    served through a provider that may withhold them, because we cannot verify\n    that the private environments expose them. **Every consumer must work when\n    they are None.**\n    """\n\n    game_id: str\n    win_levels: int\n    title: Optional[str] = None\n    tags: Optional[tuple[str, ...]] = None\n    baseline_actions: Optional[tuple[int, ...]] = None\n\n    def baseline_for_level(self, level_index: int) -> Optional[int]:\n        """Human baseline actions for a zero-indexed level, or None."""\n        if self.baseline_actions is None:\n            return None\n        if not 0 <= level_index < len(self.baseline_actions):\n            return None\n        return self.baseline_actions[level_index]\n\n    def withheld(self) -> "GameMetadata":\n        """This metadata with the unverifiable fields removed (D-012 test path)."""\n        return GameMetadata(\n            game_id=self.game_id, win_levels=self.win_levels, title=self.title\n        )\n\n    def to_json(self) -> dict[str, Any]:\n        return {\n            "game_id": self.game_id,\n            "win_levels": self.win_levels,\n            "title": self.title,\n            "tags": list(self.tags) if self.tags is not None else None,\n            "baseline_actions": (\n                list(self.baseline_actions) if self.baseline_actions is not None else None\n            ),\n        }\n\n    @staticmethod\n    def from_json(d: dict[str, Any]) -> "GameMetadata":\n        tags = d.get("tags")\n        base = d.get("baseline_actions")\n        return GameMetadata(\n            game_id=d["game_id"],\n            win_levels=int(d["win_levels"]),\n            title=d.get("title"),\n            tags=tuple(tags) if tags is not None else None,\n            baseline_actions=tuple(int(b) for b in base) if base is not None else None,\n        )\n\n\n@dataclass(frozen=True, slots=True)\nclass Observation:\n    """Everything one environment step returned.\n\n    ``frames`` may be empty. That is not an error - it means the action was\n    accepted and changed nothing observable, which is a wasted action and a fact\n    worth learning from.\n    """\n\n    frames: tuple[np.ndarray, ...]\n    state: GameState\n    levels_completed: int\n    win_levels: int\n    available_actions: tuple[int, ...]\n    full_reset: bool = False\n    action: Optional[Action] = None\n    actions_used: int = 0\n    error: Optional[str] = None\n\n    def __len__(self) -> int:\n        return len(self.frames)\n\n    def __iter__(self) -> Iterator[np.ndarray]:\n        return iter(self.frames)\n\n    @property\n    def is_noop(self) -> bool:\n        """The action was accepted but produced no frame."""\n        return not self.frames and self.error is None\n\n    @property\n    def is_error(self) -> bool:\n        return self.error is not None\n\n    @property\n    def last(self) -> Optional[np.ndarray]:\n        """The resting grid after the action, or None if nothing was drawn."""\n        return self.frames[-1] if self.frames else None\n\n    @property\n    def first(self) -> Optional[np.ndarray]:\n        return self.frames[0] if self.frames else None\n\n    def to_json(self, include_frames: bool = True) -> dict[str, Any]:\n        d: dict[str, Any] = {\n            "state": self.state.value,\n            "levels_completed": self.levels_completed,\n            "win_levels": self.win_levels,\n            "available_actions": list(self.available_actions),\n            "full_reset": self.full_reset,\n            "action": self.action.to_json() if self.action else None,\n            "actions_used": self.actions_used,\n            "error": self.error,\n            "n_frames": len(self.frames),\n        }\n        if include_frames:\n            d["frames"] = [f.tolist() for f in self.frames]\n        return d\n\n    @staticmethod\n    def from_json(d: dict[str, Any]) -> "Observation":\n        act = d.get("action")\n        return Observation(\n            frames=tuple(freeze_grid(f) for f in d.get("frames", ())),\n            state=GameState(d["state"]),\n            levels_completed=int(d["levels_completed"]),\n            win_levels=int(d["win_levels"]),\n            available_actions=tuple(int(a) for a in d["available_actions"]),\n            full_reset=bool(d.get("full_reset", False)),\n            action=Action.from_json(act) if act else None,\n            actions_used=int(d.get("actions_used", 0)),\n            error=d.get("error"),\n        )\n\n\n@dataclass(frozen=True, slots=True)\nclass Transition:\n    """A (before, action, frames, after) tuple - one dynamics example.\n\n    ``frames`` is the whole animation, not just the endpoints. A transition with\n    a long ``frames`` sequence carries far more information than its endpoints\n    do, which is the correction to design-thesis section 3\'s "one labelled\n    example per action" (D-009).\n    """\n\n    before: np.ndarray\n    action: Action\n    frames: tuple[np.ndarray, ...]\n    levels_completed_before: int\n    levels_completed_after: int\n    state_after: GameState\n\n    @property\n    def after(self) -> np.ndarray:\n        """The resting grid after the action - ``before`` if nothing was drawn."""\n        return self.frames[-1] if self.frames else self.before\n\n    @property\n    def is_noop(self) -> bool:\n        return not self.frames or bool(np.array_equal(self.before, self.after))\n\n    @property\n    def advanced_level(self) -> bool:\n        return self.levels_completed_after > self.levels_completed_before\n\n    @property\n    def n_intermediate(self) -> int:\n        """Frames strictly between ``before`` and ``after``."""\n        return max(0, len(self.frames) - 1)\n\n\n@dataclass\nclass Episode:\n    """A recorded run: metadata, seed, and every observation in order."""\n\n    game_id: str\n    seed: int\n    metadata: GameMetadata\n    observations: list[Observation] = field(default_factory=list)\n    engine_version: str = ""\n    notes: dict[str, Any] = field(default_factory=dict)\n\n    @property\n    def actions_used(self) -> int:\n        return sum(1 for o in self.observations if o.action is not None)\n\n    @property\n    def final(self) -> Observation:\n        return self.observations[-1]\n\n    def grids(self) -> list[np.ndarray]:\n        """Every grid in the episode, in order, flattened across observations."""\n        return [f for obs in self.observations for f in obs.frames]\n\n    def resting_grids(self) -> list[np.ndarray]:\n        """One grid per observation that drew anything - the settled states."""\n        return [obs.last for obs in self.observations if obs.last is not None]\n\n    def transitions(self) -> list[Transition]:\n        """Consecutive observations as dynamics examples.\n\n        Skips errored steps, and carries the previous resting grid forward\n        through no-ops so that ``before`` is always a real observed grid.\n        """\n        out: list[Transition] = []\n        prev: Optional[np.ndarray] = None\n        prev_levels = 0\n        for obs in self.observations:\n            if obs.is_error:\n                continue\n            if obs.action is not None and prev is not None:\n                out.append(\n                    Transition(\n                        before=prev,\n                        action=obs.action,\n                        frames=obs.frames,\n                        levels_completed_before=prev_levels,\n                        levels_completed_after=obs.levels_completed,\n                        state_after=obs.state,\n                    )\n                )\n            if obs.last is not None:\n                prev = obs.last\n            prev_levels = obs.levels_completed\n        return out\n\n\ndef as_sequence(frames: Sequence[Any]) -> tuple[np.ndarray, ...]:\n    return tuple(freeze_grid(f) for f in frames)\n'), 'arc3.v3': (True, '"""v3: the LLM reasons in the loop; exact code is its senses, tools and referee (D-039).\n\nModules, in the order information flows:\n\n- `facts` — what is exactly true about the frames: tracked objects, groups,\n  static objects, co-motion, transformations, per-action effects, player\n  candidates. Deterministic, recomputed every step.\n- `world` — the state the LLM\'s world model runs on, the sandbox that runs it,\n  the verifier that checks it against every recorded transition, and the\n  planner that searches it for the shortest path to the goal.\n- `prompts` — the narrow, structured calls the LLM is asked to make.\n- `agent` — the controller that ties them into assess → plan → act → verify →\n  revise.\n"""\n'), 'arc3.v3.agent': (False, '"""The v3 controller: assess -> plan or experiment -> act -> verify -> revise (D-039).\n\nPer step, deterministically:\n\n1. The fact layer observes the new frame (objects, changes, relations).\n2. If the last action came with a prediction (it was part of a plan or an\n   experiment run under the current world model), the prediction is checked\n   against what happened. A mismatch is a **surprise**: the queue is dropped\n   and the exact counterexample goes to the model.\n3. If actions are queued, the next one is taken.\n4. Otherwise a decision is made:\n   - **Probe** at the start of a level: each simple action once (and clicks on\n     a few differently coloured objects), because dynamics is cheap to test and\n     goals are not (Guardrail 4).\n   - **Assess** (LLM) when there is no model yet, after a surprise, when a plan\n     ended without a win, after a game over, or every ``ASSESS_EVERY`` actions.\n     The model\'s `predict` is verified against the recorded steps at once; while\n     it mispredicts, **repair** (LLM) is called with the counterexamples, up to\n     ``MAX_REPAIRS`` times — thinking is free, actions are not.\n   - Then the model\'s chosen **next**: *plan* — search its world model for the\n     shortest path to its goal and queue it with a predicted state per step;\n     *experiment* — queue its test actions, predicted where the model can;\n     *explore* — hand a few steps to v1\'s graph explorer.\n5. A plan that completes with the goal predicted reached and every step\n   verified, yet no win, **refutes the goal**; that goal is kept in a refuted\n   list the model sees from then on.\n\nAcross levels, the last world model, goal and hypotheses are carried forward\nas the starting point, and the facts of the winning move are described.\n\n**What this does not do.**\n\n- *No population of rival world models.* One `predict`/`goal` pair is live at a\n  time; alternatives exist only as the model\'s stated hypotheses and the\n  refuted-goal list. Rival models are the next extension if assessments churn.\n- *Verification is object-level* (world.compare): boxes, colours, existence.\n- *Clicks in plans are object centres only* (world.action_space).\n"""\n\nfrom __future__ import annotations\n\nimport time\nfrom dataclasses import dataclass, field\nfrom typing import Any, Optional\n\nimport numpy as np\n\nfrom ..agent import GraphAgent\nfrom ..llm import LLM\nfrom ..scene import make_scene\nfrom ..types import CLICK, RESET, Action, GameState, Observation\nfrom .facts import FactBook\nfrom .prompts import (\n    ASSESS_SCHEMA,\n    PROMPT_VERSION,\n    REPAIR_SCHEMA,\n    Context,\n    assess_messages,\n    parse_json,\n    repair_messages,\n)\nfrom .world import State, WorldModel, compare, parse_action, plan, verify\n\nASSESS_EVERY = 30        # actions without an assessment before one is forced\nMAX_REPAIRS = 2          # repair calls per assessment\nTRUST = 0.9              # verification accuracy at which a model may be planned with\nVERIFY_WINDOW = 80       # recent steps a model is verified on\nEXPLORE_STEPS = 12       # steps handed to the explorer per \'explore\'\nMAX_EXPERIMENT = 8\nPROBE_CLICKS = 4\nMAX_CALLS = 200          # per game\nMAX_LEVEL_CALLS = 40\nASSESS_TOKENS = 3000\nREPAIR_TOKENS = 2000\n\n\n@dataclass\nclass V3Agent:\n    llm: Optional[LLM] = None\n    seed: int = 0\n    baselines: Optional[tuple] = None\n    temperature: float = 0.0\n    graph: GraphAgent = field(default=None)  # type: ignore[assignment]\n    fb: FactBook = field(default_factory=FactBook)\n    level: int = 0\n    level_actions: int = 0\n    queue: list = field(default_factory=list)          # [(Action, expected State | None, source)]\n    wm: Optional[WorldModel] = None\n    predict_code: str = ""\n    goal_code: str = ""\n    hypotheses: list = field(default_factory=list)\n    ignore_ids: list = field(default_factory=list)\n    check_text: str = ""\n    counterexamples: list = field(default_factory=list)\n    refuted_goals: list = field(default_factory=list)\n    last_outcome: str = ""\n    previous_level: str = ""\n    trigger: str = "start"                           # why the next assessment is due\n    since_assess: int = 0\n    explore_left: int = 0\n    probed: set = field(default_factory=set)\n    calls: list = field(default_factory=list)        # diagnosis log, one dict per call\n    stats: dict = field(default_factory=dict)\n    _last_action: Optional[Action] = None\n    _expect: Optional[tuple] = None                  # (State, source) for the last action\n    _level_calls: int = 0\n    _plan_goal_claim: str = ""\n    _plan_last: bool = False\n    _accuracy: float = 0.0\n    _next: Optional[dict] = None\n\n    def __post_init__(self) -> None:\n        if self.graph is None:\n            self.graph = GraphAgent(seed=self.seed, edge_band=1.0)\n\n    # -- bookkeeping -----------------------------------------------------------\n\n    def _bump(self, k: str, n: int = 1) -> None:\n        self.stats[k] = self.stats.get(k, 0) + n\n\n    def _available(self, obs: Observation) -> list[str]:\n        out = [f"ACTION{a}" for a in sorted(obs.available_actions) if a not in (RESET, CLICK)]\n        if CLICK in obs.available_actions:\n            out.append("CLICK(x,y)")\n        return out\n\n    def _simple(self, obs: Observation) -> list[int]:\n        return sorted(a for a in obs.available_actions if a not in (RESET, CLICK))\n\n    def _state(self) -> State:\n        return State.from_objects(self.fb.objects, self.fb.grid, self.fb.masks, self.fb.background)\n\n    def _baseline(self) -> Optional[int]:\n        if not self.baselines or self.level >= len(self.baselines):\n            return None\n        return int(self.baselines[self.level])\n\n    # -- the loop ----------------------------------------------------------------\n\n    def act(self, obs: Observation) -> Action:\n        """The next action. **Returning it is free; the caller spends it.**"""\n        graph_choice = self.graph.act(obs)\n        frame = obs.last if obs.last is not None else self.fb.grid\n\n        if obs.levels_completed > self.level:\n            self._new_level(obs)\n        if frame is not None:\n            self.fb.observe(frame, self._last_action)\n\n        if obs.state == GameState.GAME_OVER:\n            self.queue, self._expect = [], None\n            self.last_outcome = (f"GAME OVER after {self.level_actions} actions on this level "\n                                 "(the level restarts on RESET).")\n            self.trigger = "game over"\n            return self._emit(Action.simple(RESET), None, "reset")\n        if obs.state == GameState.WIN or frame is None:\n            return self._emit(Action.simple(RESET), None, "reset")\n\n        self._check_expectation()\n\n        if not self.queue:\n            self._decide(obs, graph_choice)\n        if self.queue:\n            a, exp, src = self.queue.pop(0)\n            if src == "plan" and not self.queue:\n                self._plan_last = True\n            return self._emit(a, exp, src)\n        self._bump("explore_steps")\n        return self._emit(graph_choice, None, "explore")\n\n    def _emit(self, a: Action, expected: Optional[State], source: str) -> Action:\n        self.graph.took(a)\n        self._last_action = a\n        self._expect = (expected, source) if expected is not None else None\n        self.level_actions += 1\n        self.since_assess += 1\n        if self.explore_left and source == "explore":\n            self.explore_left -= 1\n        self._bump(f"src_{source}")\n        return a\n\n    def _check_expectation(self) -> None:\n        if self._expect is None:\n            return\n        expected, source = self._expect\n        self._expect = None\n        miss = compare(expected, self.fb.objects, self.ignore_ids)\n        if miss:\n            step = self.fb.steps[-1] if self.fb.steps else None\n            ce = f"t{step.t} {step.action}: " + "; ".join(miss[:4]) if step else "; ".join(miss[:4])\n            self.counterexamples = [ce, *self.counterexamples][:6]\n            self.queue = []\n            self.last_outcome = (f"SURPRISE during {source}: your predict() was wrong at {ce}. "\n                                 "The rest of the plan was dropped.")\n            self.trigger = "surprise"\n            self._bump("surprises")\n        elif source == "plan" and getattr(self, "_plan_last", False):\n            # Every step of the plan was predicted correctly and it ended where\n            # the model said the goal is met - yet the level did not end.\n            self._plan_last = False\n            if self.wm is not None and self.wm.goal(self._state()) >= 0.99:\n                claim = self._plan_goal_claim or "the current goal()"\n                self.refuted_goals.append(claim)\n                self.last_outcome = ("Your plan executed exactly as predicted and reached a state "\n                                     "your goal() scores 1.0, but the level did NOT end. That goal "\n                                     "is refuted.")\n                self.trigger = "goal refuted"\n                self._bump("goals_refuted")\n\n    # -- deciding ------------------------------------------------------------------\n\n    def _decide(self, obs: Observation, graph_choice: Action) -> None:\n        if self.trigger == "new level" and self.wm is not None and self.wm.goal_fn is not None:\n            # Levels of a game usually share their rules. Plan with the last\n            # level\'s model before probing (probing cost 4 actions per level on\n            # the synthetic room): every planned step is still checked, so a\n            # changed rule shows up as a surprise, and probing resumes then.\n            steps = self.fb.play_steps()\n            if steps:\n                self._verify()\n            if (not steps or self._accuracy >= 1.0) and self._queue_plan(obs):\n                self.trigger = ""\n                self._bump("carried_plans")\n                return\n        probe = self._next_probe(obs)\n        if probe is not None:\n            self.queue.append((probe, None, "probe"))\n            return\n        if self.explore_left > 0:\n            return  # the explorer\'s choice is used\n        if self.llm is None or len(self.calls) >= MAX_CALLS or self._level_calls >= MAX_LEVEL_CALLS:\n            return\n        if self.trigger or self.since_assess >= ASSESS_EVERY or self.wm is None:\n            self._assess(obs)\n        self._act_on_next(obs)\n\n    def _next_probe(self, obs: Observation) -> Optional[Action]:\n        for a in self._simple(obs):\n            if a not in self.probed:\n                self.probed.add(a)\n                return Action.simple(a)\n        if CLICK in obs.available_actions:\n            clicked_colours = {k for k in self.probed if isinstance(k, tuple)}\n            if len(clicked_colours) < PROBE_CLICKS:\n                for o in sorted(self.fb.objects.values(), key=lambda o: -o.size):\n                    key = ("click", o.colour)\n                    if key not in self.probed:\n                        self.probed.add(key)\n                        return Action.click(min(63, o.x + o.w // 2), min(63, o.y + o.h // 2))\n        return None\n\n    def _context(self, obs: Observation) -> Context:\n        return Context(\n            level=self.level, actions_on_level=self.level_actions,\n            human_baseline=self._baseline(), available=self._available(obs),\n            facts=self.fb.describe(), grid=make_scene(self.fb.grid).grid_text(),\n            hypotheses=self.hypotheses, predict_code=self.predict_code,\n            goal_code=self.goal_code, ignore_ids=self.ignore_ids, check=self.check_text,\n            counterexamples=self.counterexamples, refuted_goals=self.refuted_goals,\n            last_outcome=self.last_outcome, previous_level=self.previous_level)\n\n    def _call(self, kind: str, messages: list, schema: dict, tokens: int) -> Optional[dict]:\n        t0 = time.time()\n        try:\n            text = self.llm.complete(messages, max_tokens=tokens, temperature=self.temperature,\n                                     seed=self.seed + len(self.calls), schema=schema)\n        except Exception as exc:  # a dead server must not end the game\n            text = ""\n            self._bump("llm_errors")\n            self.last_outcome = f"(model call failed: {type(exc).__name__})"\n        data = parse_json(text)\n        self.calls.append({"kind": kind, "level": self.level, "at": self.level_actions,\n                           "ok": data is not None, "seconds": round(time.time() - t0, 2),\n                           "prompt": PROMPT_VERSION})\n        self._level_calls += 1\n        self._bump(f"calls_{kind}")\n        return data\n\n    def _assess(self, obs: Observation) -> None:\n        data = self._call("assess", assess_messages(self._context(obs)), ASSESS_SCHEMA, ASSESS_TOKENS)\n        self.since_assess = 0\n        self.trigger = ""\n        if data is None:\n            self._next = {"kind": "explore", "purpose": "the model gave no usable answer"}\n            return\n        self.hypotheses = [h for h in data.get("hypotheses", []) if isinstance(h, dict)]\n        self.ignore_ids = [int(i) for i in data.get("ignore_ids", []) if isinstance(i, (int, float))]\n        self._install(str(data.get("predict_code", "")), str(data.get("goal_code", "")))\n        goals = [h.get("claim", "") for h in self.hypotheses if h.get("kind") == "goal"]\n        self._plan_goal_claim = goals[0] if goals else ""\n        self._next = data.get("next") if isinstance(data.get("next"), dict) else {"kind": "explore"}\n        repairs = 0\n        while self._needs_repair() and repairs < MAX_REPAIRS and len(self.calls) < MAX_CALLS:\n            repairs += 1\n            fix = self._call("repair", repair_messages(self._context(obs)), REPAIR_SCHEMA, REPAIR_TOKENS)\n            if fix is None:\n                break\n            if isinstance(fix.get("ignore_ids"), list):\n                self.ignore_ids = [int(i) for i in fix["ignore_ids"] if isinstance(i, (int, float))]\n            self._install(str(fix.get("predict_code", "")), self.goal_code)\n\n    def _install(self, predict_code: str, goal_code: str) -> None:\n        wm = WorldModel.compile(predict_code, goal_code)\n        self.predict_code, self.goal_code, self.wm = predict_code, goal_code, wm\n        self._verify()\n\n    def _verify(self) -> None:\n        if self.wm is None:\n            return\n        if self.wm.error and self.wm.predict_fn is None:\n            self.check_text = f"does not compile: {self.wm.error}"\n            self.counterexamples = []\n            self._accuracy = 0.0\n            return\n        steps = self.fb.play_steps()[-VERIFY_WINDOW:]\n        chk = verify(self.wm, steps, self.ignore_ids, self.fb.background)\n        self.check_text = chk.describe() + (f"; runtime error: {self.wm.error}" if self.wm.error else "")\n        self.counterexamples = chk.counterexamples\n        self._accuracy = chk.accuracy if chk.n else 0.0\n        self.calls and self.calls[-1].setdefault("accuracy", round(self._accuracy, 3))\n\n    def _needs_repair(self) -> bool:\n        return self.wm is not None and getattr(self, "_accuracy", 0.0) < 1.0 and bool(self.fb.play_steps())\n\n    def _act_on_next(self, obs: Observation) -> None:\n        nxt = getattr(self, "_next", None) or {"kind": "explore"}\n        kind = nxt.get("kind", "explore")\n        if kind == "plan" and self.wm is not None and getattr(self, "_accuracy", 0.0) >= TRUST:\n            if self._queue_plan(obs):\n                return\n            kind = "experiment" if nxt.get("actions") else "explore"\n        if kind == "experiment" and nxt.get("actions"):\n            if self._queue_actions(nxt["actions"], obs, "experiment"):\n                self.last_outcome = f"Ran your experiment: {nxt.get(\'purpose\', \'\')}"\n                return\n        self.explore_left = EXPLORE_STEPS\n        self.last_outcome = f"Explored {EXPLORE_STEPS} steps ({nxt.get(\'purpose\', \'no plan available\')})."\n        self._bump("explore_handoffs")\n\n    def _queue_plan(self, obs: Observation) -> bool:\n        start = self._state()\n        p = plan(self.wm, start, self._simple(obs), CLICK in obs.available_actions)\n        self._bump("plans")\n        if not p.actions:\n            self.last_outcome = (f"The planner searched {p.nodes} states of your model and found no "\n                                 f"path that raises goal() above {p.start_progress:.2f}.")\n            self.trigger = "no plan"\n            return False\n        if not p.reached:\n            self.last_outcome = (f"No path to goal()=1 in {p.nodes} states; following the best path "\n                                 f"found (goal {p.start_progress:.2f} -> {p.progress:.2f}).")\n        self._plan_last = False\n        return self._queue_actions(p.actions, obs, "plan")\n\n    def _queue_actions(self, actions: list, obs: Observation, source: str) -> bool:\n        s = self._state()\n        q = []\n        for a in actions[: (200 if source == "plan" else MAX_EXPERIMENT)]:\n            parsed = parse_action(str(a))\n            if parsed is None or (parsed[0] not in obs.available_actions and parsed[0] != RESET):\n                continue\n            aid, x, y = parsed\n            act = Action.click(min(63, x), min(63, y)) if aid == CLICK else Action.simple(aid)\n            name = f"CLICK({act.x},{act.y})" if act.is_click else ("RESET" if aid == RESET else f"ACTION{aid}")\n            expected = None\n            if self.wm is not None and self.wm.predict_fn is not None and aid != RESET:\n                s = self.wm.predict(s, name)\n                expected = s\n            q.append((act, expected, source))\n        self.queue = q\n        return bool(q)\n\n    # -- levels ----------------------------------------------------------------------\n\n    def _new_level(self, obs: Observation) -> None:\n        won_with = self._last_action\n        self.previous_level = (\n            f"Level {self.level + 1} was won in {self.level_actions} actions; the winning action "\n            f"was {won_with}. The hypotheses at that point were: "\n            + "; ".join(f"[{h.get(\'kind\')}] {h.get(\'claim\')}" for h in self.hypotheses[:8])\n            + ". Its predict() and goal() are carried over as your starting point.")\n        self._bump("levels")\n        self.level = obs.levels_completed\n        self.level_actions = 0\n        self._level_calls = 0\n        self.fb = FactBook()\n        self._last_action = None\n        self.queue, self._expect = [], None\n        self.refuted_goals, self.counterexamples = [], []\n        self.probed = set()\n        self.explore_left = 0\n        self.trigger = "new level"\n\n    def describe(self) -> str:\n        return (f"level {self.level}, calls {len(self.calls)}, "\n                + ", ".join(f"{k}={v}" for k, v in sorted(self.stats.items())))\n'), 'arc3.v3.facts': (False, '"""Exact facts about the frames, recomputed every step (D-039, part 1).\n\nThe LLM reasons; this module sees. Everything here is computed, never guessed:\nit is the evidence the model\'s hypotheses must explain and the referee that\nscores them.\n\n**What it computes.**\n\n- *Objects with stable ids.* `colour_8` components (background excluded,\n  bookkeeping band excluded) tracked across frames by the global-assignment\n  tracker (D-014), so "object #3" means the same thing from step to step.\n- *Changes per step,* typed: ``moved`` (with dx, dy), ``recoloured`` (from,\n  to), ``grew`` / ``shrank`` (size before, after), ``reshaped``, ``appeared``,\n  ``vanished``. One object can have several at once (moved and reshaped).\n- *Relations:* same-colour groups, same-shape groups (translation-invariant\n  shape identity), objects never changed so far (static), pairs that moved with\n  the same displacement on the same step (co-motion / tandem).\n- *Action effects:* for every action (clicks keyed by the clicked object), how\n  often it was tried, how often it did nothing, and what it did to what.\n- *Player candidates:* objects moved by two or more different actions with a\n  consistent displacement per action — the signature of something controlled.\n\n**What this does not do.**\n\n- *No interpretation.* It will say "#3 moved (0,-5) on all 7 ACTION1 steps and\n  never moved otherwise"; it will not say "#3 is the player". That is a\n  hypothesis, and hypotheses are the LLM\'s (Guardrail 2).\n- *One carving.* Multi-coloured sprites arrive as several objects; co-motion\n  facts are how the model can see they belong together.\n- *No animation frames.* Each step is judged on its resting frame.\n- *Identity is the tracker\'s best assignment,* not ground truth: 0.8% switch\n  rate measured on resting frames (D-014).\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nfrom collections import Counter, defaultdict\nfrom dataclasses import dataclass, field, replace\nfrom typing import Optional\n\nimport numpy as np\n\nfrom ..perception.segment import Segmentation, segment\nfrom ..perception.track import Tracker\nfrom ..types import Action\n\nCARVING = "colour_8"\n\n\n@dataclass(frozen=True)\nclass Obj:\n    """One object in one frame. Also the element of a world-model state."""\n\n    id: int\n    colour: int\n    x: int          # bounding box, x1/y1 exclusive\n    y: int\n    w: int\n    h: int\n    size: int\n    shape: str      # short translation-invariant shape identity\n\n    @property\n    def cx(self) -> float:\n        return self.x + self.w / 2\n\n    @property\n    def cy(self) -> float:\n        return self.y + self.h / 2\n\n    def describe(self) -> str:\n        return (f"#{self.id} colour {self.colour} at x{self.x} y{self.y} "\n                f"size {self.w}x{self.h} ({self.size} cells) shape {self.shape}")\n\n\n@dataclass(frozen=True)\nclass Change:\n    id: int\n    kinds: tuple[str, ...]      # subset of KINDS\n    dx: int = 0\n    dy: int = 0\n    colour_from: Optional[int] = None\n    colour_to: Optional[int] = None\n    size_from: Optional[int] = None\n    size_to: Optional[int] = None\n\n    def describe(self) -> str:\n        bits = []\n        for k in self.kinds:\n            if k == "moved":\n                bits.append(f"moved ({self.dx:+d},{self.dy:+d})")\n            elif k == "recoloured":\n                bits.append(f"recoloured {self.colour_from}->{self.colour_to}")\n            elif k in ("grew", "shrank"):\n                bits.append(f"{k} {self.size_from}->{self.size_to}")\n            else:\n                bits.append(k)\n        return f"#{self.id} " + ", ".join(bits)\n\n\nKINDS = ("moved", "recoloured", "grew", "shrank", "reshaped", "appeared", "vanished")\n\n\n@dataclass\nclass Step:\n    t: int\n    action: str\n    clicked: Optional[int]          # id of the clicked object, for clicks\n    before: dict[int, Obj]\n    after: dict[int, Obj]\n    changes: list[Change]\n    reset: bool = False\n    #: The frame and exact object masks before the action, for world-model\n    #: collision queries (`world.State`). Masks are cropped to each box.\n    grid_before: Optional[np.ndarray] = None\n    masks_before: Optional[dict] = None\n\n    @property\n    def noop(self) -> bool:\n        return not self.changes\n\n\ndef _shape_id(key: bytes) -> str:\n    return hashlib.blake2b(key, digest_size=3).hexdigest()\n\n\ndef action_name(a: Action) -> str:\n    return f"CLICK({a.x},{a.y})" if a.is_click else ("RESET" if a.is_reset else f"ACTION{a.id}")\n\n\n@dataclass\nclass FactBook:\n    """All facts for one level. Feed it every resting frame with :meth:`observe`."""\n\n    band: Optional[np.ndarray] = None       # bool mask of bookkeeping rows\n    tracker: Tracker = field(default_factory=lambda: Tracker(CARVING))\n    steps: list[Step] = field(default_factory=list)\n    objects: dict[int, Obj] = field(default_factory=dict)\n    background: int = 0\n    grid: Optional[np.ndarray] = None       # the current frame\n    masks: dict = field(default_factory=dict)  # id -> cropped bool mask, current frame\n    _t: int = 0\n\n    # -- observing ---------------------------------------------------------\n\n    def _objects_of(self, frame: np.ndarray) -> dict[int, Obj]:\n        seg = segment(np.asarray(frame), include=(CARVING,))[CARVING]\n        bg = seg.background_colour if seg.background_colour is not None else 0\n        self.background = int(bg)\n        keep = []\n        for c in seg.components:\n            if c.dominant_colour == bg:\n                continue\n            if self.band is not None and self.band.any() and self.band[c.mask].all():\n                continue\n            keep.append(c)\n        comps = [replace(c, index=i) for i, c in enumerate(keep)]\n        self.tracker.update(Segmentation(name=CARVING, components=tuple(comps),\n                                         cell=seg.cell, background_colour=bg))\n        ids = self.tracker.current_ids()\n        out = {}\n        masks = {}\n        for c in comps:\n            tid = ids.get(c.index)\n            if tid is None:\n                continue\n            r = c.bbox\n            masks[tid] = c.mask[r.y0:r.y1, r.x0:r.x1].copy()\n            out[tid] = Obj(id=tid, colour=int(c.dominant_colour), x=int(c.bbox.x0),\n                           y=int(c.bbox.y0), w=int(c.bbox.x1 - c.bbox.x0),\n                           h=int(c.bbox.y1 - c.bbox.y0), size=int(c.size),\n                           shape=_shape_id(c.shape_key))\n        self._last_masks = masks\n        return out\n\n    def observe(self, frame: np.ndarray, action: Optional[Action] = None) -> Optional[Step]:\n        """Record the frame that followed ``action`` (None for the first frame)."""\n        before, grid_before, masks_before = self.objects, self.grid, self.masks\n        after = self._objects_of(frame)\n        self.objects = after\n        self.grid = np.asarray(frame).copy()\n        self.masks = self._last_masks\n        if action is None:\n            return None\n        changes = diff_objects(before, after)\n        clicked = None\n        if action.is_click:\n            clicked = object_at(before, action.x, action.y)\n        step = Step(t=self._t, action=action_name(action), clicked=clicked, before=before,\n                    after=after, changes=changes, reset=action.is_reset,\n                    grid_before=grid_before, masks_before=masks_before)\n        self._t += 1\n        self.steps.append(step)\n        return step\n\n    # -- relations ---------------------------------------------------------\n\n    def play_steps(self) -> list[Step]:\n        return [s for s in self.steps if not s.reset]\n\n    def static_ids(self) -> list[int]:\n        """Objects present since they appeared and never changed on any step."""\n        changed = {c.id for s in self.play_steps() for c in s.changes}\n        return sorted(i for i in self.objects if i not in changed)\n\n    def colour_groups(self) -> dict[int, list[int]]:\n        g: dict[int, list[int]] = defaultdict(list)\n        for o in self.objects.values():\n            g[o.colour].append(o.id)\n        return {c: sorted(ids) for c, ids in g.items() if len(ids) > 1}\n\n    def shape_groups(self) -> dict[str, list[int]]:\n        g: dict[str, list[int]] = defaultdict(list)\n        for o in self.objects.values():\n            g[o.shape].append(o.id)\n        return {s: sorted(ids) for s, ids in g.items() if len(ids) > 1}\n\n    def tandem(self, min_times: int = 2) -> list[tuple[int, int, int]]:\n        """(a, b, n): a and b moved with the same displacement on n steps, and\n        never moved on a step without the other."""\n        together: Counter = Counter()\n        moved_steps: dict[int, set[int]] = defaultdict(set)\n        for s in self.play_steps():\n            moves = {c.id: (c.dx, c.dy) for c in s.changes if "moved" in c.kinds}\n            for i in moves:\n                moved_steps[i].add(s.t)\n            ids = sorted(moves)\n            for i, a in enumerate(ids):\n                for b in ids[i + 1:]:\n                    if moves[a] == moves[b]:\n                        together[(a, b)] += 1\n        out = []\n        for (a, b), n in together.items():\n            if n >= min_times and moved_steps[a] == moved_steps[b]:\n                out.append((a, b, n))\n        return sorted(out, key=lambda t: -t[2])\n\n    def action_effects(self) -> dict[str, dict]:\n        """Per action key: tries, no-ops, and a Counter of change descriptions.\n\n        Clicks are keyed by the clicked object\'s colour ("CLICK colour 9"), so\n        experience with one blue square transfers to the other blue squares.\n        """\n        out: dict[str, dict] = {}\n        for s in self.play_steps():\n            key = s.action\n            if s.action.startswith("CLICK"):\n                col = s.before[s.clicked].colour if s.clicked in s.before else None\n                key = f"CLICK colour {col}" if col is not None else "CLICK background"\n            e = out.setdefault(key, {"tries": 0, "noops": 0, "effects": Counter()})\n            e["tries"] += 1\n            if s.noop:\n                e["noops"] += 1\n            for c in s.changes:\n                who = s.before.get(c.id) or s.after.get(c.id)\n                tag = f"colour-{who.colour}" if who else f"#{c.id}"\n                e["effects"][f"{tag} " + c.describe().split(" ", 1)[1]] += 1\n        return out\n\n    def player_candidates(self) -> list[dict]:\n        """Objects moved by >= 2 distinct non-click actions, each with one dominant\n        displacement: what a controlled object looks like, not a claim that it is."""\n        per: dict[int, dict[str, Counter]] = defaultdict(lambda: defaultdict(Counter))\n        for s in self.play_steps():\n            if s.action.startswith("CLICK"):\n                continue\n            for c in s.changes:\n                if "moved" in c.kinds:\n                    per[c.id][s.action][(c.dx, c.dy)] += 1\n        out = []\n        for i, acts in per.items():\n            if len(acts) < 2:\n                continue\n            summary = {a: cnt.most_common(1)[0][0] for a, cnt in acts.items()}\n            consistency = sum(cnt.most_common(1)[0][1] for cnt in acts.values()) / \\\n                sum(sum(cnt.values()) for cnt in acts.values())\n            out.append({"id": i, "moves": summary, "consistency": round(consistency, 2)})\n        return sorted(out, key=lambda d: (-len(d["moves"]), -d["consistency"]))\n\n    # -- rendering -----------------------------------------------------------\n\n    def describe(self, max_objects: int = 40, max_effects: int = 6) -> str:\n        """The facts as text for a prompt. Every line is exact."""\n        objs = sorted(self.objects.values(), key=lambda o: -o.size)\n        static = set(self.static_ids())\n        moved_n = Counter(c.id for s in self.play_steps() for c in s.changes)\n        lines = [f"OBJECTS now ({len(objs)}; background colour {self.background}):"]\n        for o in objs[:max_objects]:\n            tag = "static" if o.id in static else f"changed on {moved_n[o.id]} steps"\n            lines.append(f"  {o.describe()} [{tag}]")\n        if len(objs) > max_objects:\n            lines.append(f"  ... and {len(objs) - max_objects} smaller objects")\n        cg, sg = self.colour_groups(), self.shape_groups()\n        if cg:\n            lines.append("SAME COLOUR: " + "; ".join(\n                f"colour {c}: " + " ".join(f"#{i}" for i in ids[:12]) for c, ids in cg.items()))\n        if sg:\n            lines.append("SAME SHAPE: " + "; ".join(\n                " ".join(f"#{i}" for i in ids[:12]) for ids in list(sg.values())[:8]))\n        td = self.tandem()\n        if td:\n            lines.append("MOVE TOGETHER: " + "; ".join(\n                f"#{a} and #{b} ({n} times)" for a, b, n in td[:8]))\n        eff = self.action_effects()\n        if eff:\n            lines.append("ACTION EFFECTS so far:")\n            for k, e in sorted(eff.items()):\n                top = ", ".join(f"{d} x{n}" for d, n in e["effects"].most_common(max_effects))\n                lines.append(f"  {k}: tried {e[\'tries\']}, no effect {e[\'noops\']}"\n                             + (f"; {top}" if top else ""))\n        pc = self.player_candidates()\n        if pc:\n            lines.append("CONTROLLED-LOOKING objects (moved by several actions):")\n            for p in pc[:3]:\n                mv = ", ".join(f"{a}->({d[0]:+d},{d[1]:+d})" for a, d in sorted(p["moves"].items()))\n                lines.append(f"  #{p[\'id\']}: {mv} (consistency {p[\'consistency\']})")\n        recent = [s for s in self.steps[-6:]]\n        if recent:\n            lines.append("LAST STEPS:")\n            for s in recent:\n                ch = "; ".join(c.describe() for c in s.changes[:5]) or "nothing changed"\n                if len(s.changes) > 5:\n                    ch += f"; +{len(s.changes) - 5} more"\n                lines.append(f"  t{s.t} {s.action}: {ch}")\n        return "\\n".join(lines)\n\n\ndef diff_objects(before: dict[int, Obj], after: dict[int, Obj]) -> list[Change]:\n    out = []\n    for i, a in after.items():\n        b = before.get(i)\n        if b is None:\n            out.append(Change(id=i, kinds=("appeared",)))\n            continue\n        kinds = []\n        dx, dy = a.x - b.x, a.y - b.y\n        if dx or dy:\n            kinds.append("moved")\n        if a.colour != b.colour:\n            kinds.append("recoloured")\n        if a.size > b.size:\n            kinds.append("grew")\n        elif a.size < b.size:\n            kinds.append("shrank")\n        elif a.shape != b.shape:\n            kinds.append("reshaped")\n        if kinds:\n            out.append(Change(id=i, kinds=tuple(kinds), dx=dx, dy=dy,\n                              colour_from=b.colour, colour_to=a.colour,\n                              size_from=b.size, size_to=a.size))\n    for i in before:\n        if i not in after:\n            out.append(Change(id=i, kinds=("vanished",)))\n    return sorted(out, key=lambda c: c.id)\n\n\ndef object_at(objs: dict[int, Obj], x: Optional[int], y: Optional[int]) -> Optional[int]:\n    """The smallest object whose box contains (x, y) — the likeliest click target."""\n    if x is None or y is None:\n        return None\n    hits = [o for o in objs.values() if o.x <= x < o.x + o.w and o.y <= y < o.y + o.h]\n    return min(hits, key=lambda o: o.size).id if hits else None\n'), 'arc3.v3.prompts': (False, '"""The LLM\'s calls in v3: each has one job, exact inputs and a schema (D-039).\n\nTwo calls, deliberately narrow:\n\n- **assess** — form or revise the whole hypothesis set: what the goal is,\n  which object is the player, what each action does, which objects are static\n  or counters, what the others are for; write them as a world model\n  (`predict`) and a goal test (`goal`); choose what to do next — plan to the\n  goal, run an experiment that tests a hypothesis, or explore.\n- **repair** — the world model mispredicted recorded steps; fix the code given\n  the exact counterexamples. Nothing else is asked, so nothing else can drift.\n\nEvery input is exact: the fact layer\'s text, a compact map of the frame, the\nverifier\'s score and counterexamples, the goals already refuted, what the last\nplan did. The prompt describes the *interface* — what a state, an action and a\nhelper are — and never what kinds of games exist (Guardrail 2).\n\nBump ``PROMPT_VERSION`` whenever wording changes: recorded calls replay only\nwhile prompts are byte-identical.\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport re\nfrom dataclasses import dataclass, field\nfrom typing import Optional\n\nPROMPT_VERSION = "v3-1"\n\nAPI = """\\\nWORLD-MODEL API (your code runs in a sandbox with `np` and `math`; no imports):\n  predict(state, action) -> state   # simulate ONE action; mutate `state` and return it\n  goal(state) -> float in [0, 1]    # 1.0 exactly when the level is won; graded so\n                                    # that states closer to winning score higher\n  `action` is a string: "ACTION1".."ACTION7" or "CLICK(x,y)" (x = column, y = row).\n  state.objects            dict id -> {"id","colour","x","y","w","h","size","shape"}\n                           (x, y = top-left of the bounding box; ids are the object\n                           ids in the facts, stable across steps within a level)\n  state.get(id)            the object dict or None\n  state.of_colour(c)       list of objects of colour c\n  state.at(x, y)           smallest object whose box contains (x, y), or None\n  state.colours_in_box(x, y, w, h, ignore=[ids])   set of frame colours in that\n                           box (-1 if it leaves the frame), not counting the cells\n                           the ignored objects occupied at the start of the step\n  state.free(x, y, w, h, passable={colours}, ignore=[ids])  True if the box only\n                           holds passable colours (default: the background)\n  state.overlaps(a, b)     do two object dicts\' boxes overlap\n  state.move(id, dx, dy) / state.place(id, x, y) / state.recolour(id, c) /\n  state.resize(id, w, h) / state.remove(id) / state.add(colour, x, y, w, h)\n  state.meta               a dict for anything else you need to track (counters,\n                           flags); it persists along a simulated path\nFind objects robustly (by colour, shape or position) rather than hard-coding a\nsingle id when an object might be replaced."""\n\nSYSTEM = f"""\\\nYou are the reasoning core of an agent playing an unknown turn-based puzzle game\non a 64x64 grid of 16 colours (0-15). There are no instructions: the rules and\nthe goal must be discovered by acting and observing. The level is scored on how\nFEW actions it takes to win (compared with a human), so think carefully and act\npurposefully: every action should either test a hypothesis or make progress.\nRESET restarts the current level (it costs one action).\n\nThe agent\'s code tells you exact facts about the frames: objects with stable\nids, which ones never changed, same-colour and same-shape groups, objects that\nmove together, what each action has done so far, and objects that look\ncontrolled. Use them. Your hypotheses must explain these facts.\n\nYou express your understanding as code the agent can execute:\n- `predict(state, action)` is your theory of how the game works. The agent\n  checks it against EVERY recorded step and reports exactly where it was wrong.\n- `goal(state)` is your theory of what wins. The agent searches your predict()\n  for the SHORTEST action sequence that makes goal() reach 1.0, and executes it.\nSo a correct predict() plus a correct goal() wins the level in the fewest\nactions. Objects you believe are irrelevant to the rules (move counters, score\nbars, decorations) can be listed in ignore_ids so they are not checked.\n\n{API}\n\nAnswer with JSON only, matching the schema you are given."""\n\n\nASSESS_SCHEMA = {\n    "title": "assess",\n    "type": "object",\n    "properties": {\n        "thoughts": {"type": "string",\n                     "description": "Brief reasoning: what the facts show and what is still uncertain."},\n        "hypotheses": {\n            "type": "array",\n            "items": {\n                "type": "object",\n                "properties": {\n                    "kind": {"type": "string", "enum": ["goal", "player", "action", "static",\n                                                        "role", "counter", "other"]},\n                    "claim": {"type": "string"},\n                    "objects": {"type": "array", "items": {"type": "integer"}},\n                    "confidence": {"type": "number"},\n                },\n                "required": ["kind", "claim", "confidence"],\n            },\n        },\n        "ignore_ids": {"type": "array", "items": {"type": "integer"}},\n        "predict_code": {"type": "string", "description": "Python defining predict(state, action)."},\n        "goal_code": {"type": "string", "description": "Python defining goal(state)."},\n        "next": {\n            "type": "object",\n            "properties": {\n                "kind": {"type": "string", "enum": ["plan", "experiment", "explore"]},\n                "actions": {"type": "array", "items": {"type": "string"}},\n                "purpose": {"type": "string"},\n            },\n            "required": ["kind", "purpose"],\n        },\n    },\n    "required": ["thoughts", "hypotheses", "predict_code", "goal_code", "next"],\n}\n\nREPAIR_SCHEMA = {\n    "title": "repair",\n    "type": "object",\n    "properties": {\n        "thoughts": {"type": "string", "description": "What the counterexamples show was wrong."},\n        "predict_code": {"type": "string"},\n        "ignore_ids": {"type": "array", "items": {"type": "integer"}},\n    },\n    "required": ["thoughts", "predict_code"],\n}\n\n\n@dataclass\nclass Context:\n    """Everything the agent knows, gathered for a prompt."""\n\n    level: int\n    actions_on_level: int\n    human_baseline: Optional[int]\n    available: list[str]\n    facts: str\n    grid: str\n    hypotheses: list[dict] = field(default_factory=list)\n    predict_code: str = ""\n    goal_code: str = ""\n    ignore_ids: list[int] = field(default_factory=list)\n    check: str = ""                     # verifier summary for predict_code\n    counterexamples: list[str] = field(default_factory=list)\n    refuted_goals: list[str] = field(default_factory=list)\n    last_outcome: str = ""              # what the last plan/experiment did\n    previous_level: str = ""            # what won the previous level\n\n\ndef _common(ctx: Context) -> list[str]:\n    p = [f"LEVEL {ctx.level + 1}. Actions used on this level: {ctx.actions_on_level}"\n         + (f" (a human needed about {ctx.human_baseline})." if ctx.human_baseline else ".")]\n    p.append("AVAILABLE ACTIONS: " + ", ".join(ctx.available))\n    if ctx.previous_level:\n        p.append("PREVIOUS LEVEL: " + ctx.previous_level)\n    p.append("FACTS (exact, computed by the agent):\\n" + ctx.facts)\n    p.append("CURRENT FRAME:\\n" + ctx.grid)\n    return p\n\n\ndef assess_messages(ctx: Context) -> list[dict]:\n    p = _common(ctx)\n    if ctx.hypotheses:\n        p.append("YOUR CURRENT HYPOTHESES:\\n" + "\\n".join(\n            f"  [{h.get(\'kind\')}] {h.get(\'claim\')} (confidence {h.get(\'confidence\')})"\n            for h in ctx.hypotheses[:20]))\n    if ctx.predict_code:\n        p.append("YOUR CURRENT predict():\\n" + ctx.predict_code.strip())\n        p.append("CHECK AGAINST RECORDED STEPS: " + (ctx.check or "not yet checked"))\n        if ctx.counterexamples:\n            p.append("MISPREDICTIONS:\\n" + "\\n".join("  " + c for c in ctx.counterexamples))\n    if ctx.goal_code:\n        p.append("YOUR CURRENT goal():\\n" + ctx.goal_code.strip())\n    if ctx.refuted_goals:\n        p.append("GOALS ALREADY REFUTED (their goal() reached 1.0 and the level did NOT end):\\n"\n                 + "\\n".join("  - " + g for g in ctx.refuted_goals[-6:]))\n    if ctx.last_outcome:\n        p.append("WHAT HAPPENED LAST: " + ctx.last_outcome)\n    p.append(\n        "TASK: Update your hypotheses about this game (goal, player, what each action does, "\n        "static objects, counters, roles of other objects). Write predict() and goal() that "\n        "express them. Then choose next: \'plan\' (the agent searches your model for the "\n        "shortest path to goal() = 1 and executes it), \'experiment\' (give up to 8 actions that "\n        "will test the hypothesis you are least sure of; say what you expect), or \'explore\' "\n        "(let the agent try untested actions for a while). Prefer \'plan\' only when predict() "\n        "explains the recorded steps and you believe goal()."\n    )\n    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": "\\n\\n".join(p)}]\n\n\ndef repair_messages(ctx: Context) -> list[dict]:\n    p = _common(ctx)\n    p.append("YOUR predict():\\n" + ctx.predict_code.strip())\n    p.append("CHECK AGAINST RECORDED STEPS: " + ctx.check)\n    p.append("MISPREDICTIONS (predicted vs what actually happened):\\n"\n             + "\\n".join("  " + c for c in ctx.counterexamples))\n    p.append("TASK: Fix predict() so it explains these steps, keeping what already works. "\n             "If a mismatch is only a counter or decoration you do not need to model, add its "\n             "id to ignore_ids instead.")\n    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": "\\n\\n".join(p)}]\n\n\n_OBJ = re.compile(r"\\{.*\\}", re.S)\n\n\ndef parse_json(text: str) -> Optional[dict]:\n    """The JSON object in a response, tolerating stray prose or code fences."""\n    if not text:\n        return None\n    t = text.strip()\n    if t.startswith("```"):\n        t = re.sub(r"^```(?:json)?\\s*|\\s*```$", "", t)\n    for candidate in (t, *(_OBJ.findall(t)[:1])):\n        try:\n            v = json.loads(candidate)\n            if isinstance(v, dict):\n                return v\n        except (ValueError, TypeError):\n            continue\n    return None\n'), 'arc3.v3.world': (False, '"""The LLM\'s world model: its state, its sandbox, its verifier, its planner (D-039).\n\nThe LLM writes two functions in Python::\n\n    def predict(state, action):     # how the game works\n        ...                         # mutate `state` with its helpers and return it\n        return state\n\n    def goal(state):                # what winning is\n        return 0.0 .. 1.0           # 1.0 = level won (a bool also works)\n\n`predict` runs on a :class:`State` — the objects of one frame (the same ids the\nfact layer tracks) plus the frame itself for collision queries. This is the\ncode-world-model approach: a program that can be *executed*, so it can be\nchecked against every recorded transition (:func:`verify`), repaired from the\nexact mismatches it produces, and *searched* for the shortest path to its own\ngoal (:func:`plan`) — internal planning, costing no actions.\n\n**What this does not do.**\n\n- *Object level only.* Predictions are compared on each object\'s box and colour\n  and on which objects exist; the exact cells inside a box are not checked. An\n  occluded maze whose mask changes as the player passes over it therefore does\n  not count as a mispredicted maze.\n- *Collision is against the frame the step started from* (`colours_in_box`),\n  with the moving objects\' own cells removed. Two objects moved in one\n  simulated step do not see each other unless the model checks boxes itself.\n- *Search is bounded* (`max_nodes`). A goal further than the budget reaches is\n  reported as not found, not as unreachable.\n"""\n\nfrom __future__ import annotations\n\nimport math\nimport re\nfrom collections.abc import Callable\nfrom dataclasses import dataclass, field\nfrom heapq import heappop, heappush\nfrom typing import Any, Optional\n\nimport numpy as np\n\nfrom ..hypothesis import _SAFE_BUILTINS, _budgeted\nfrom .facts import Obj, Step\n\n#: Line events one predict() or goal() call may execute.\nCALL_BUDGET = 20_000\n\n\nclass State:\n    """Objects of one frame, mutable, plus the frame for collision queries.\n\n    Objects are plain dicts with keys id, colour, x, y, w, h, size, shape, so\n    the model\'s code can read and write them directly. The helpers are the\n    operations games are made of: move, place, recolour, remove, add, and\n    queries about what occupies a region.\n    """\n\n    def __init__(self, objects: dict[int, dict], grid: Optional[np.ndarray],\n                 masks: Optional[dict], background: int, meta: Optional[dict] = None):\n        self.objects = objects\n        self.grid = grid\n        self.masks = masks or {}\n        self.background = background\n        self.meta = meta if meta is not None else {}   # free-form: counters, flags\n        self._origin = {i: (o["x"], o["y"]) for i, o in objects.items()}\n        self._next_new = -1\n\n    # -- construction --------------------------------------------------------\n\n    @staticmethod\n    def from_objects(objs: dict[int, Obj], grid=None, masks=None, background: int = 0) -> "State":\n        return State({i: {"id": o.id, "colour": o.colour, "x": o.x, "y": o.y, "w": o.w,\n                          "h": o.h, "size": o.size, "shape": o.shape} for i, o in objs.items()},\n                     grid, masks, background)\n\n    @staticmethod\n    def before(step: Step, background: int = 0) -> "State":\n        return State.from_objects(step.before, step.grid_before, step.masks_before, background)\n\n    def copy(self) -> "State":\n        s = State({i: dict(o) for i, o in self.objects.items()}, self.grid, self.masks,\n                  self.background, dict(self.meta))\n        s._origin = self._origin\n        s._next_new = self._next_new\n        return s\n\n    def key(self) -> tuple:\n        return tuple(sorted((i, o["colour"], o["x"], o["y"], o["w"], o["h"])\n                            for i, o in self.objects.items())) + tuple(sorted(\n                                (k, v) for k, v in self.meta.items()\n                                if isinstance(v, (int, float, str, bool))))\n\n    # -- queries the model may use ---------------------------------------------\n\n    def get(self, i: int) -> Optional[dict]:\n        return self.objects.get(i)\n\n    def of_colour(self, c: int) -> list[dict]:\n        return sorted((o for o in self.objects.values() if o["colour"] == c), key=lambda o: o["id"])\n\n    def at(self, x: int, y: int) -> Optional[dict]:\n        hits = [o for o in self.objects.values()\n                if o["x"] <= x < o["x"] + o["w"] and o["y"] <= y < o["y"] + o["h"]]\n        return min(hits, key=lambda o: o["size"]) if hits else None\n\n    def colours_in_box(self, x: int, y: int, w: int, h: int, ignore=()) -> set:\n        """Colours of the frame inside a box, with the ORIGINAL cells of the\n        ``ignore`` ids removed (so a moving object does not collide with itself).\n        Out-of-frame area reports -1."""\n        out: set = set()\n        if self.grid is None:\n            return out\n        H, W = self.grid.shape\n        if x < 0 or y < 0 or x + w > W or y + h > H:\n            out.add(-1)\n        x0, y0, x1, y1 = max(0, x), max(0, y), min(W, x + w), min(H, y + h)\n        if x0 >= x1 or y0 >= y1:\n            return out\n        region = self.grid[y0:y1, x0:x1]\n        keep = np.ones(region.shape, dtype=bool)\n        for i in ignore:\n            m = self.masks.get(i)\n            ox, oy = self._origin.get(i, (None, None))\n            if m is None or ox is None:\n                continue\n            mh, mw = m.shape\n            ax0, ay0 = max(x0, ox), max(y0, oy)\n            ax1, ay1 = min(x1, ox + mw), min(y1, oy + mh)\n            if ax0 < ax1 and ay0 < ay1:\n                keep[ay0 - y0:ay1 - y0, ax0 - x0:ax1 - x0] &= ~m[ay0 - oy:ay1 - oy, ax0 - ox:ax1 - ox]\n        out.update(int(v) for v in np.unique(region[keep]))\n        return out\n\n    def free(self, x: int, y: int, w: int, h: int, passable=None, ignore=()) -> bool:\n        """True if the box holds only ``passable`` colours (default: background)."""\n        ok = set(passable) if passable is not None else {self.background}\n        return self.colours_in_box(x, y, w, h, ignore) <= ok\n\n    @staticmethod\n    def overlaps(a: dict, b: dict) -> bool:\n        return (a["x"] < b["x"] + b["w"] and b["x"] < a["x"] + a["w"]\n                and a["y"] < b["y"] + b["h"] and b["y"] < a["y"] + a["h"])\n\n    # -- mutations -----------------------------------------------------------------\n\n    def move(self, i: int, dx: int, dy: int) -> "State":\n        o = self.objects.get(i)\n        if o is not None:\n            o["x"] += int(dx)\n            o["y"] += int(dy)\n        return self\n\n    def place(self, i: int, x: int, y: int) -> "State":\n        o = self.objects.get(i)\n        if o is not None:\n            o["x"], o["y"] = int(x), int(y)\n        return self\n\n    def recolour(self, i: int, c: int) -> "State":\n        o = self.objects.get(i)\n        if o is not None:\n            o["colour"] = int(c)\n        return self\n\n    def resize(self, i: int, w: int, h: int) -> "State":\n        o = self.objects.get(i)\n        if o is not None:\n            o["w"], o["h"] = int(w), int(h)\n            o["size"] = int(w) * int(h)\n        return self\n\n    def remove(self, i: int) -> "State":\n        self.objects.pop(i, None)\n        return self\n\n    def add(self, colour: int, x: int, y: int, w: int, h: int) -> "State":\n        """A new object; its id is negative until the real frame names it."""\n        i = self._next_new\n        self._next_new -= 1\n        self.objects[i] = {"id": i, "colour": int(colour), "x": int(x), "y": int(y),\n                           "w": int(w), "h": int(h), "size": int(w) * int(h), "shape": "new"}\n        return self\n\n\n# -- the sandboxed model ---------------------------------------------------------\n\n\n_FENCE = re.compile(r"```(?:python|py)?\\s*\\n(.*?)```", re.S)\n\n\ndef strip_fences(code: str) -> str:\n    m = _FENCE.search(code or "")\n    return m.group(1) if m else (code or "")\n\n\n@dataclass\nclass WorldModel:\n    """`predict` and `goal` compiled from the LLM\'s code, run in the sandbox."""\n\n    source: str\n    predict_fn: Optional[Callable] = None\n    goal_fn: Optional[Callable] = None\n    error: Optional[str] = None\n\n    @staticmethod\n    def compile(predict_src: str, goal_src: str = "") -> "WorldModel":\n        src = strip_fences(predict_src) + "\\n\\n" + strip_fences(goal_src)\n        wm = WorldModel(source=src)\n        if re.search(r"^\\s*(import|from)\\s", src, re.M) or "__" in src:\n            wm.error = "imports and dunder access are not allowed (np and math are provided)"\n            return wm\n        ns: dict[str, Any] = {"__builtins__": _SAFE_BUILTINS, "np": np, "math": math}\n        try:\n            exec(compile(src, "<hypothesis>", "exec"), ns)\n        except Exception as exc:\n            wm.error = f"{type(exc).__name__}: {exc}"\n            return wm\n        wm.predict_fn = ns.get("predict") if callable(ns.get("predict")) else None\n        wm.goal_fn = ns.get("goal") if callable(ns.get("goal")) else None\n        if wm.predict_fn is None:\n            wm.error = "no predict(state, action) function"\n        return wm\n\n    def predict(self, state: State, action: str) -> State:\n        """One simulated step. Never raises: a failing model predicts no change."""\n        if self.predict_fn is None:\n            return state.copy()\n        s = state.copy()\n        try:\n            out = _budgeted(lambda st: self.predict_fn(st, action), s, CALL_BUDGET)\n        except Exception as exc:\n            self.error = f"predict raised {type(exc).__name__}: {exc}"\n            return state.copy()\n        return out if isinstance(out, State) else s\n\n    def goal(self, state: State) -> float:\n        if self.goal_fn is None:\n            return 0.0\n        try:\n            v = _budgeted(self.goal_fn, state, CALL_BUDGET)\n            v = float(v)\n            return min(1.0, max(0.0, v)) if math.isfinite(v) else 0.0\n        except Exception:\n            return 0.0\n\n\n# -- verification -----------------------------------------------------------------\n\n\n@dataclass\nclass Check:\n    """How well a world model explains the recorded steps of a level."""\n\n    n: int = 0\n    exact: int = 0\n    counterexamples: list[str] = field(default_factory=list)\n\n    @property\n    def accuracy(self) -> float:\n        return self.exact / self.n if self.n else 0.0\n\n    def describe(self) -> str:\n        return f"{self.exact}/{self.n} steps predicted exactly ({100 * self.accuracy:.0f}%)"\n\n\ndef compare(pred: State, after: dict[int, Obj], ignore=()) -> list[str]:\n    """Mismatches between a predicted state and the observed objects, as text."""\n    ign = set(ignore)\n    out = []\n    new_pred = [o for i, o in pred.objects.items() if i < 0]\n    for i, a in after.items():\n        if i in ign:\n            continue\n        p = pred.objects.get(i)\n        if p is None:\n            # An object the model did not keep: it was either not predicted to\n            # survive, or it is new. New ones may match a predicted add().\n            hit = next((q for q in new_pred if q["colour"] == a.colour and q["x"] == a.x\n                        and q["y"] == a.y), None)\n            if hit is not None:\n                new_pred.remove(hit)\n                continue\n            out.append(f"#{i} (colour {a.colour}) exists at x{a.x} y{a.y}; predicted absent")\n            continue\n        diffs = []\n        if (p["x"], p["y"]) != (a.x, a.y):\n            diffs.append(f"at x{p[\'x\']} y{p[\'y\']} -> actually x{a.x} y{a.y}")\n        if p["colour"] != a.colour:\n            diffs.append(f"colour {p[\'colour\']} -> actually {a.colour}")\n        if (p["w"], p["h"]) != (a.w, a.h):\n            diffs.append(f"size {p[\'w\']}x{p[\'h\']} -> actually {a.w}x{a.h}")\n        if diffs:\n            out.append(f"#{i} predicted " + "; ".join(diffs))\n    for i, p in pred.objects.items():\n        if i >= 0 and i not in after and i not in ign:\n            out.append(f"#{i} (colour {p[\'colour\']}) predicted present; actually vanished")\n    for q in new_pred:\n        out.append(f"predicted a new colour-{q[\'colour\']} object at x{q[\'x\']} y{q[\'y\']}; none appeared")\n    return out\n\n\ndef verify(model: WorldModel, steps: list[Step], ignore=(), background: int = 0,\n           max_examples: int = 6) -> Check:\n    """Replay the model on every recorded non-reset step of the level."""\n    chk = Check()\n    for s in steps:\n        if s.reset or s.grid_before is None:\n            continue\n        chk.n += 1\n        pred = model.predict(State.before(s, background), s.action)\n        miss = compare(pred, s.after, ignore)\n        if not miss:\n            chk.exact += 1\n        elif len(chk.counterexamples) < max_examples:\n            chk.counterexamples.append(f"t{s.t} {s.action}: " + "; ".join(miss[:4]))\n    return chk\n\n\n# -- planning -----------------------------------------------------------------------\n\n\n@dataclass\nclass Plan:\n    actions: list[str]\n    reached: bool          # does the model say the goal is met at the end?\n    progress: float        # the goal\'s value at the end state\n    start_progress: float\n    nodes: int\n\n\ndef action_space(state: State, simple: list[int], click: bool, max_clicks: int = 40) -> list[str]:\n    acts = [f"ACTION{a}" for a in simple]\n    if click:\n        objs = sorted(state.objects.values(), key=lambda o: -o["size"])[:max_clicks]\n        acts += [f"CLICK({o[\'x\'] + o[\'w\'] // 2},{o[\'y\'] + o[\'h\'] // 2})" for o in objs]\n    return acts\n\n\ndef plan(model: WorldModel, start: State, simple: list[int], click: bool,\n         max_nodes: int = 4000, max_depth: int = 120, bfs_nodes: int = 20_000) -> Plan:\n    """The shortest path to the model\'s goal, searched entirely inside the model.\n\n    Phase 1 is breadth-first up to ``bfs_nodes`` expansions: the first goal\n    state it reaches is at the fewest actions, which is what RHAE\'s quadratic\n    penalty rewards. Greedy best-first was the first version and returned 19\n    moves where 17 were optimal (tests/test_v3_world.py). Phase 2, only when the\n    state space outgrows phase 1, is best-first on the goal\'s progress, which\n    finds *a* path in large spaces without promising the shortest; if that\n    finds none either, the path to the best state seen is returned, flagged\n    not reached.\n    """\n    found = _bfs(model, start, simple, click, bfs_nodes, max_depth)\n    if found is not None:\n        return found\n    p0 = model.goal(start)\n    best = (p0, [])\n    seen = {start.key()}\n    heap: list = [(-p0, 0, 0, [], start)]\n    tie = 0\n    nodes = 0\n    while heap and nodes < max_nodes:\n        negp, depth, _, path, s = heappop(heap)\n        nodes += 1\n        if -negp >= 0.99:\n            return Plan(path, True, -negp, p0, nodes)\n        if depth >= max_depth:\n            continue\n        for a in action_space(s, simple, click):\n            t = model.predict(s, a)\n            k = t.key()\n            if k in seen:\n                continue\n            seen.add(k)\n            p = model.goal(t)\n            tie += 1\n            np_ = path + [a]\n            if p > best[0] or (p == best[0] and len(np_) < len(best[1]) and best[1]):\n                best = (p, np_)\n            heappush(heap, (-p, depth + 1, tie, np_, t))\n    return Plan(best[1], best[0] >= 0.99, best[0], p0, nodes)\n\n\ndef _bfs(model: WorldModel, start: State, simple: list[int], click: bool,\n         max_nodes: int, max_depth: int) -> Optional[Plan]:\n    """Breadth-first to the nearest goal state; None if the budget ran out first.\n\n    Returns a not-reached Plan (empty path) when the reachable space is\n    exhausted without a goal: the model then says the goal is unreachable,\n    which is itself evidence against model or goal.\n    """\n    from collections import deque\n\n    p0 = model.goal(start)\n    if p0 >= 0.99:\n        return Plan([], True, p0, p0, 0)\n    seen = {start.key()}\n    q = deque([(start, [])])\n    nodes = 0\n    while q:\n        if nodes >= max_nodes:\n            return None\n        s, path = q.popleft()\n        nodes += 1\n        if len(path) >= max_depth:\n            continue\n        for a in action_space(s, simple, click):\n            t = model.predict(s, a)\n            k = t.key()\n            if k in seen:\n                continue\n            seen.add(k)\n            p = model.goal(t)\n            if p >= 0.99:\n                return Plan(path + [a], True, p, p0, nodes)\n            q.append((t, path + [a]))\n    return Plan([], False, p0, p0, nodes)\n\n\ndef parse_action(a: str):\n    """\'ACTION3\' -> (3, None, None); \'CLICK(10,20)\' -> (6, 10, 20); else None."""\n    a = a.strip().upper().replace(" ", "")\n    m = re.fullmatch(r"ACTION([1-7])", a)\n    if m:\n        return int(m.group(1)), None, None\n    m = re.fullmatch(r"CLICK\\((\\d+),(\\d+)\\)", a)\n    if m:\n        return 6, int(m.group(1)), int(m.group(2))\n    if a == "RESET":\n        return 0, None, None\n    return None\n')}


class _Arc3Finder(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path, target=None):
        if fullname in _ARC3_SOURCES:
            is_pkg = _ARC3_SOURCES[fullname][0]
            return importlib.util.spec_from_loader(fullname, self, is_package=is_pkg)
        return None

    def create_module(self, spec):
        return None

    def exec_module(self, module):
        is_pkg, src = _ARC3_SOURCES[module.__name__]
        if is_pkg:
            module.__path__ = []
        exec(compile(src, "<arc3:" + module.__name__ + ">", "exec"), module.__dict__)


if not any(isinstance(f, _Arc3Finder) for f in sys.meta_path):
    sys.meta_path.insert(0, _Arc3Finder())



In [ ]:
%%writefile /tmp/eval_agent.py
"""End-to-end agent evaluation: levels cleared and offline RHAE, cold start.

The first script in the repository that reports the competition metric for a
policy rather than a proxy (D-034). Every game is played once from a cold start
with a fixed action cap, and the score is computed by `client/budget.py` against
the human baselines shipped in each game's metadata.

Also runs the `explore` driver under the same cap, with a RESET on GAME_OVER, as
the baseline: before this agent it was the only policy that cleared any level.

Usage:
    python scripts/eval_agent.py [--actions 500] [--agents "graph;graph_nomodel;explore"]
                                 [--games ls20 vc33] [--out docs/experiments/agent.json]
"""

from __future__ import annotations

import argparse
import json
import logging
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

from arc3.agent import GraphAgent
from arc3.client.budget import ActionBudget
from arc3.client.env import EnvClient, available_environments, make_arcade
from arc3.agent_v2 import HypothesisAgent
from arc3.v3.agent import V3Agent
from arc3.explore import ExploringDriver
from arc3.llm import OpenAIChatLLM, RecordingLLM, ReplayLLM
from arc3.types import RESET, Action, GameState

#: Set from the command line in main(); module-level so play() stays picklable.
LLM_URL = ""
CALLS_DIR = "docs/experiments/llm_calls"
REPLAY = False
REPLAY_ONLY = False
LLM_EXTRA: dict = {}


def play(gid: str, agent_name: str, n_actions: int, seed: int, arcade, env_dir: str) -> dict:
    client = EnvClient(gid, seed=seed, record=False, arcade=arcade, environments_dir=env_dir)
    meta = client.metadata
    budget = ActionBudget(gid, meta.win_levels, meta.baseline_actions)
    if agent_name == "explore":
        driver = ExploringDriver(seed=seed)

        def choose(obs):
            if obs.state in (GameState.GAME_OVER, GameState.WIN):
                return Action.simple(RESET)
            a = driver.next_action(obs)
            return a if a is not None else Action.simple(RESET)
        agent = None
    else:
        # "graph_nomodel:local_min=0.2" -> GraphAgent(use_model=False, local_min=0.2)
        base, _, opts = agent_name.partition(":")
        kw = {k: float(v) for k, v in (o.split("=") for o in opts.split(",") if o)}
        if base in ("v2", "v3"):
            # The LLM at LLM_URL; every call recorded under CALLS_DIR, or
            # replayed from there when REPLAY is set (then no model is needed).
            llm = None
            if LLM_URL:
                calls = Path(CALLS_DIR) / f"{agent_name.replace(':', '_')}_s{seed}_{gid}.jsonl"
                live = OpenAIChatLLM(base_url=LLM_URL, extra=LLM_EXTRA)
                llm = (ReplayLLM(calls, fallback=live if not REPLAY_ONLY else None) if REPLAY
                       else RecordingLLM(live, calls))
            if base == "v3":
                agent = V3Agent(seed=seed, llm=llm, baselines=meta.baseline_actions,
                                temperature=float(kw.pop("temperature", 0.0)))
            else:
                agent = HypothesisAgent(seed=seed, llm=llm, **kw)
        else:
            agent = GraphAgent(seed=seed, use_model=(base == "graph"), **kw)
        choose = agent.act

    t0 = time.time()
    level_actions: list[int] = []
    for _ in range(n_actions):
        obs = client.current
        if obs.state == GameState.WIN:
            break
        before = obs.levels_completed
        obs = client.step(choose(obs))
        budget.observe(obs)
        if obs.levels_completed > before:
            level_actions.append(client.actions_used)
    res = budget.result()
    return {
        "game": gid,
        "agent": agent_name,
        "levels": res.levels_completed,
        "total_levels": res.total_levels,
        "score": res.score(),
        "actions": client.actions_used,
        "cleared_at": level_actions,
        "per_level": [
            {"i": lv.index, "actions": lv.actions, "baseline": lv.baseline,
             "score": lv.score()} for lv in res.levels if lv.completed
        ],
        "seconds": round(time.time() - t0, 1),
        "detail": agent.describe() if agent else "",
        # v2 only: (level, actions into level, hypotheses proposed, compiled) per call,
        # and the population as it stood at the end.
        "llm_calls": getattr(agent, "calls", None),
        "hypotheses": agent.pop.describe() if hasattr(agent, "pop") else None,
        "level_report": agent.level_report() if hasattr(agent, "level_report") else None,
        "stats": dict(agent.stats) if agent else None,
    }


def main() -> None:
    global LLM_URL, CALLS_DIR, REPLAY, REPLAY_ONLY, LLM_EXTRA
    ap = argparse.ArgumentParser()
    ap.add_argument("--actions", type=int, default=500)
    ap.add_argument("--agents", default="graph;graph_nomodel;explore")
    ap.add_argument("--games", nargs="*")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--env-dir", default="environment_files")
    ap.add_argument("--out")
    ap.add_argument("--llm-url", default="", help="OpenAI-compatible server for v2, e.g. http://127.0.0.1:8080")
    ap.add_argument("--calls-dir", default=CALLS_DIR)
    ap.add_argument("--replay", action="store_true", help="serve v2's LLM calls from --calls-dir")
    ap.add_argument("--replay-only", action="store_true", help="with --replay: never call the live model")
    ap.add_argument("--llm-extra", default="{}", help="JSON merged into every LLM request body")
    args = ap.parse_args()
    logging.basicConfig(level=logging.WARNING)

    LLM_URL, CALLS_DIR = args.llm_url, args.calls_dir
    REPLAY, REPLAY_ONLY = args.replay, args.replay_only
    LLM_EXTRA = json.loads(args.llm_extra)

    arcade = make_arcade(environments_dir=args.env_dir)
    games = args.games or available_environments(args.env_dir)
    agents = args.agents.split(";")
    rows = []
    for gid in games:
        for name in agents:
            r = play(gid, name, args.actions, args.seed, arcade, args.env_dir)
            rows.append(r)
            print(f"{gid:6s} {name:14s} levels {r['levels']}/{r['total_levels']} "
                  f"score {100 * (r['score'] or 0):5.1f}%  cleared_at {r['cleared_at']}  "
                  f"{r['seconds']}s  {r['detail']}", flush=True)

    print("\n== summary ==")
    for name in agents:
        rs = [r for r in rows if r["agent"] == name]
        lv = sum(r["levels"] for r in rs)
        tot = sum(r["total_levels"] for r in rs)
        sc = sum((r["score"] or 0) for r in rs) / len(rs)
        g = sum(1 for r in rs if r["levels"])
        print(f"{name:14s} levels {lv}/{tot}  games with a level {g}/{len(rs)}  "
              f"mean RHAE {100 * sc:.2f}%")
    if args.out:
        Path(args.out).write_text(json.dumps({"args": vars(args), "rows": rows}, indent=1))


if __name__ == "__main__":
    main()


In [ ]:
import glob, os, subprocess, sys
env_dirs = sorted({os.path.dirname(os.path.dirname(os.path.dirname(p)))
                    for p in glob.glob('/kaggle/input/**/metadata.json', recursive=True)})
env_dir = next((d for d in env_dirs if len(os.listdir(d)) >= 20), None)
print('environment_files:', env_dir)
os.makedirs('/kaggle/working/results', exist_ok=True)
url = os.environ.get('ARC3_LLM_URL', '')
which = os.environ.get('ARC3_BENCH_AGENT', 'v3')   # 'v3' (default) or 'v2'
agents = (which + ':temperature=' + os.environ.get('ARC3_LLM_TEMPERATURE', '0.7')
          + ',max_tokens=' + os.environ.get('ARC3_LLM_MAX_TOKENS', '900')) if url else 'graph_nomodel'
print('LLM:', url or 'none -- scoring v1 only')

# Games run in parallel shards so the LLM server sees many requests at
# once: vLLM's throughput comes from batching, and one game at a time
# would take ~3 h per seed at single-stream decode speed.
# ARC3_BENCH_GAMES="tn36 sp80" restricts the games (a cheap smoke test);
# ARC3_BENCH_SEEDS overrides the seeds; ARC3_BENCH_SHARDS the parallelism.
import json, time
games = os.environ.get('ARC3_BENCH_GAMES', '').split() or sorted(os.listdir(env_dir))
seeds = [int(s) for s in os.environ.get('ARC3_BENCH_SEEDS', '0,1,2').split(',')]
shards = max(1, min(len(games), int(os.environ.get('ARC3_BENCH_SHARDS', '8'))))
R = '/kaggle/working/results'
t0 = time.time()
procs = []
for seed in seeds:
    for k in range(shards):
        part = games[k::shards]
        cmd = [sys.executable, '-c',
               "import sys; sys.path.insert(0, '/tmp'); import arc3_bundle, runpy; "
               "sys.argv = ['eval_agent.py'] + sys.argv[1:]; "
               "runpy.run_path('/tmp/eval_agent.py', run_name='__main__')",
               '--actions', '500', '--seed', str(seed), '--agents', agents,
               '--env-dir', env_dir, '--calls-dir', R + '/llm_calls',
               '--games', *part, '--out', f'{R}/eval_s{seed}_k{k}.json']
        if url:
            cmd += ['--llm-url', url, '--llm-extra', os.environ.get('ARC3_LLM_EXTRA', '{}')]
        log = open(f'{R}/log_s{seed}_k{k}.txt', 'w')
        # arc-agi 0.9.8 lets OPERATION_MODE=competition in the
        # environment override the OFFLINE mode our client requests,
        # and competition mode needs the gateway. This benchmark plays
        # the public games locally, so force offline for it only.
        penv = dict(os.environ, OPERATION_MODE='offline')
        procs.append(subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=penv))
print(f'{len(procs)} processes: {len(games)} games x {len(seeds)} seeds in {shards} shards')
while any(p.poll() is None for p in procs):
    time.sleep(60)
    done = sum(p.poll() is not None for p in procs)
    print(f'{(time.time() - t0) / 60:.0f} min: {done}/{len(procs)} shards finished', flush=True)

# Merge shards into one file per seed, in the local eval_agent format.
for seed in seeds:
    rows = []
    for k in range(shards):
        p = f'{R}/eval_s{seed}_k{k}.json'
        if os.path.exists(p):
            rows += json.load(open(p))['rows']
        else:
            # A shard that wrote nothing crashed; show why, never hide it.
            print(f'--- shard s{seed} k{k} produced no results; end of its log:')
            print(open(f'{R}/log_s{seed}_k{k}.txt', errors='replace').read()[-3000:])
    json.dump({'rows': rows, 'seed': seed, 'agents': agents},
              open(f'{R}/eval_s{seed}.json', 'w'), indent=1)
    sc = sum((r['score'] or 0) for r in rows) / max(1, len(rows))
    lv = sum(r['levels'] for r in rows)
    print(f'seed {seed}: {len(rows)} games, {lv} levels, mean RHAE {100 * sc:.2f}%')
print(f'total {(time.time() - t0) / 60:.0f} min')
